# 04d Extraction, batch d: eleven studies coded from their PDFs and one full-text exclusion

Design log.
- Batch d of notebook 04 (plan v2 B4), coded under the frozen schema (schema/record_schema.yaml, v1) and the coding rules, as batches a to c were. Twelve studies were read; eleven are coded: AgentDyn (2602.03117), ICON (2602.20708), IH-Challenge (2603.10521), PISmith (2603.13026, COLM 2026), The Autonomy Tax (2603.19423), PIArena (2604.08499), LocalAlign (2605.01462, CCS 2026), AI Agents May Always Fall for Prompt Injections (2605.17634), activation consistency training (2605.28467), dynamic separators for Polymorphic Prompt Assembling (2605.30534) and Gravity-Weighted DPO (2606.10860).
- Full-text exclusion found while coding: Prompt Injection as Role Confusion (2603.12277, ICML 2026) was included at title and abstract with a note to check whether a training fix is proposed and measured. Its full text introduces an attack (CoT Forgery) and role probes and tests frontier models as they are; no defence is added, trained or measured, and Section 7 argues for role perception without experiments. It is excluded as attack only, the reason used for 2509.05883 in notebook 02d. Stage 2b checks the anchor in the PDF and carries the decision into schema/fulltext_decisions.csv, schema/screening_decisions.csv, reports/screening.csv, reports/defences.csv, reports/fulltext_check.csv and reports/prisma_counts.json, as notebook 02d did. The number of included studies falls from 137 to 136.
- 2601.04666 is still held back (its PDF is larger than the Drive connector can fetch).
- Each record's extraction_scope names the tables and rows coded. As before, detectors, sanitisers, rewriters, tool filters and system-level designs around the model (PromptArmor, PISanitizer, DataFilter, MELON, CaMeL, Progent, DRIFT, Tool Filter and the guard classifiers) are left out and the scope says so (amendment 4).
- Coding decisions worth knowing when reading these records (reading aids for notebook 05, not findings):
  - PIArena does not name the SecAlign++ checkpoint it ran or say whether it was trained on its default Qwen3-4B-Instruct backend, so the base condition of its six comparisons is unreported and they grade U.
  - The Autonomy Tax compares Meta SecAlign on Llama-3.1-8B with undefended Llama-3-8B in its agent tables, so that comparison grades B. Its Table 3 places Meta SecAlign in the 350-prompt subset while Appendix A.7 says the Meta SecAlign runs used a 1,000-prompt set, so those two comparisons grade U. Its Table 8 (Meta SecAlign with and without the input role) is coded as a component comparison of the role signal. Its StruQ and SecAlign are the authors' own re-implementations (SecAlign trained to prefer refusals), and their agent completion rate is measured with attack-like trigger phrases rather than attacks, so it is coded other.
  - ICON's probe is a small classifier fitted on 255 samples and the model is not trained, so it is coded as a fixed inference rule, as coding rule 11 treats a direction fitted from examples. ICON is trained on TrojanTools, which is also one of its three test attacks; a residual note says so.
  - Activation consistency training matches the frozen base model's hidden states, so axis 2 is distillation. Its Tables 2 and 3 print different values for the same Phi-4-reasoning BCT cells; the record codes Table 3 and a residual note records the mismatch.
  - Dynamic Polymorphic Prompt Assembling is compared only with static PPA under attack (no undefended arm), so those comparisons vary the separator generation only.
  - For Gravity-Weighted DPO only the hierarchy pairs whose lower level is data or tool output (L4) are coded, with the SEP probe-execution rate and the over-refusal rate.
  - IH-Challenge's prompt injection results are robustness scores (1 minus attack success) and are coded as such, with the direction higher is better.
- Outputs: reports/records/ (eleven files), reports/extraction_04d_check.csv, reports/extraction_04d_summary.csv, reports/extraction_batches.csv (rows for batch d added, earlier batches kept), reports/decision_04d.txt, and the full-text decision files listed above.
- CPU; no network calls beyond pip; model calls: 0.

## Stage 0. Session bootstrap

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/SourceMap_Research"
    REPO_DIR = PROJECT_ROOT + "/sourcemap-research"
except ImportError:
    PROJECT_ROOT = os.environ["SOURCEMAP_PROJECT_ROOT"]
    REPO_DIR = os.environ["SOURCEMAP_REPO_ROOT"]
os.environ["SOURCEMAP_PROJECT_ROOT"] = PROJECT_ROOT
os.environ["SOURCEMAP_REPO_ROOT"] = REPO_DIR
os.chdir(REPO_DIR)
if "src" not in sys.path:
    sys.path.insert(0, "src")
for module, package in (("yaml", "pyyaml"), ("pypdf", "pypdf"), ("pypdfium2", "pypdfium2")):
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)
import pypdf
if int(pypdf.__version__.split(".")[0]) < 4:                 # the column-preserving extraction needs pypdf 4 or later
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pypdf>=4"], check=False)
    print("pypdf upgraded; restart the runtime if the next stage reports an old version")
print(subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True).stdout.strip() or "pull: nothing to report")
print("cwd:", os.getcwd(), "| pypdf", pypdf.__version__)

## Stage 1. The 11 records

In [ ]:
Path("reports/records").mkdir(parents=True, exist_ok=True)       # %%writefile does not create folders

`agentdyn_arxiv2026` (2602.03117)

In [ ]:
%%writefile reports/records/agentdyn_arxiv2026.yaml
report:
  id: agentdyn_arxiv2026
  slug: '2602.03117'
  title: 'AgentDyn: Are Your Agent Security Defenses Deployable in Real-World Dynamic Environments?'
  first_author: Hao Li
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2602.03117'
  version_read: arXiv v3
  first_date: '2026-02-03'
  read: F
  role: benchmark
  code_release: https://github.com/leolee99/AgentDyn (linked from the abstract)
  extraction_scope: 'Sections 3 and 4: Table 3 rows None, Prompt Sandwiching and Spotlighting for the four models, and Meta SecAlign 70B coded as the defended model against Llama 3.3 70B; the abstract claim (ProtectAI, PIGuard
    and PromptGuard2, which are detectors, Tool Filter, CaMeL, Progent and DRIFT, which are system-level designs around the model, the eight further models of Appendix G and Tables 8 to 17 not coded)'
defences:
- id: msa70b
  name: Meta SecAlign 70B (released model, re-measured)
  origin: other_report
  base_checkpoint: Llama 3.3 70B
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  layout: AgentDyn agent loop (AgentDojo framework); tool outputs carry the injection
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Meta SecAlign-70B [20], a defensive model trained on Llama-3.3-70B-Instruct
- id: sand_g4o
  name: Prompt Sandwiching
  origin: other_report
  base_checkpoint: GPT-4o
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  layout: AgentDyn agent loop (AgentDojo framework); tool outputs carry the injection
  layout_compat: *id001
  anchor: we evaluate Prompt Sandwiching [11] and Spotlighting [12]
- id: spot_g4o
  name: Spotlighting
  origin: other_report
  base_checkpoint: GPT-4o
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  layout: AgentDyn agent loop (AgentDojo framework); tool outputs carry the injection
  layout_compat: *id001
  anchor: we evaluate Prompt Sandwiching [11] and Spotlighting [12]
- id: sand_gem
  name: Prompt Sandwiching
  origin: other_report
  base_checkpoint: Gemini-2.5 Pro
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  layout: AgentDyn agent loop (AgentDojo framework); tool outputs carry the injection
  layout_compat: *id001
  anchor: we evaluate Prompt Sandwiching [11] and Spotlighting [12]
- id: spot_gem
  name: Spotlighting
  origin: other_report
  base_checkpoint: Gemini-2.5 Pro
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  layout: AgentDyn agent loop (AgentDojo framework); tool outputs carry the injection
  layout_compat: *id001
  anchor: we evaluate Prompt Sandwiching [11] and Spotlighting [12]
- id: sand_cl35
  name: Prompt Sandwiching
  origin: other_report
  base_checkpoint: Claude-Sonnet-3.5
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  layout: AgentDyn agent loop (AgentDojo framework); tool outputs carry the injection
  layout_compat: *id001
  anchor: we evaluate Prompt Sandwiching [11] and Spotlighting [12]
- id: spot_cl35
  name: Spotlighting
  origin: other_report
  base_checkpoint: Claude-Sonnet-3.5
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  layout: AgentDyn agent loop (AgentDojo framework); tool outputs carry the injection
  layout_compat: *id001
  anchor: we evaluate Prompt Sandwiching [11] and Spotlighting [12]
- id: sand_l33
  name: Prompt Sandwiching
  origin: other_report
  base_checkpoint: Llama 3.3 70B
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  layout: AgentDyn agent loop (AgentDojo framework); tool outputs carry the injection
  layout_compat: *id001
  anchor: we evaluate Prompt Sandwiching [11] and Spotlighting [12]
- id: spot_l33
  name: Spotlighting
  origin: other_report
  base_checkpoint: Llama 3.3 70B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  layout: AgentDyn agent loop (AgentDojo framework); tool outputs carry the injection
  layout_compat: *id001
  anchor: we evaluate Prompt Sandwiching [11] and Spotlighting [12]
evaluations:
- id: ev_none_g4o_clean
  defence_id: none:GPT-4o
  benchmark: AgentDyn, 60 user tasks without injection (Shopping, GitHub, DailyLife; mean over the three suites)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn user-task check
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '60'
  pages:
    benchmark: 6
    sample_size: 6
  baseline_kind: off_the_shelf
- id: ev_none_g4o_att
  defence_id: none:GPT-4o
  benchmark: AgentDyn, 560 injection cases with the important_instructions attack (mean over the three suites)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn user-task check; attacker goal executed
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  attacker_budget: AgentDojo important_instructions template
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 6
    attacker_budget: 6
    sample_size: 6
  baseline_kind: off_the_shelf
- id: ev_none_gem_clean
  defence_id: none:Gemini-2.5 Pro
  benchmark: AgentDyn, 60 user tasks without injection (Shopping, GitHub, DailyLife; mean over the three suites)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn user-task check
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '60'
  pages:
    benchmark: 6
    sample_size: 6
  baseline_kind: off_the_shelf
- id: ev_none_gem_att
  defence_id: none:Gemini-2.5 Pro
  benchmark: AgentDyn, 560 injection cases with the important_instructions attack (mean over the three suites)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn user-task check; attacker goal executed
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  attacker_budget: AgentDojo important_instructions template
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 6
    attacker_budget: 6
    sample_size: 6
  baseline_kind: off_the_shelf
- id: ev_none_cl35_clean
  defence_id: none:Claude-Sonnet-3.5
  benchmark: AgentDyn, 60 user tasks without injection (Shopping, GitHub, DailyLife; mean over the three suites)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn user-task check
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '60'
  pages:
    benchmark: 6
    sample_size: 6
  baseline_kind: off_the_shelf
- id: ev_none_cl35_att
  defence_id: none:Claude-Sonnet-3.5
  benchmark: AgentDyn, 560 injection cases with the important_instructions attack (mean over the three suites)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn user-task check; attacker goal executed
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  attacker_budget: AgentDojo important_instructions template
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 6
    attacker_budget: 6
    sample_size: 6
  baseline_kind: off_the_shelf
- id: ev_none_l33_clean
  defence_id: none:Llama 3.3 70B
  benchmark: AgentDyn, 60 user tasks without injection (Shopping, GitHub, DailyLife; mean over the three suites)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn user-task check
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '60'
  pages:
    benchmark: 6
    sample_size: 6
  baseline_kind: off_the_shelf
- id: ev_none_l33_att
  defence_id: none:Llama 3.3 70B
  benchmark: AgentDyn, 560 injection cases with the important_instructions attack (mean over the three suites)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn user-task check; attacker goal executed
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  attacker_budget: AgentDojo important_instructions template
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 6
    attacker_budget: 6
    sample_size: 6
  baseline_kind: off_the_shelf
- id: ev_sand_g4o_clean
  defence_id: sand_g4o
  benchmark: AgentDyn, 60 user tasks without injection (Shopping, GitHub, DailyLife; mean over the three suites)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn user-task check
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '60'
  pages:
    benchmark: 6
    sample_size: 6
- id: ev_sand_g4o_att
  defence_id: sand_g4o
  benchmark: AgentDyn, 560 injection cases with the important_instructions attack (mean over the three suites)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn user-task check; attacker goal executed
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  attacker_budget: AgentDojo important_instructions template
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 6
    attacker_budget: 6
    sample_size: 6
- id: ev_sand_gem_clean
  defence_id: sand_gem
  benchmark: AgentDyn, 60 user tasks without injection (Shopping, GitHub, DailyLife; mean over the three suites)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn user-task check
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '60'
  pages:
    benchmark: 6
    sample_size: 6
- id: ev_sand_gem_att
  defence_id: sand_gem
  benchmark: AgentDyn, 560 injection cases with the important_instructions attack (mean over the three suites)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn user-task check; attacker goal executed
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  attacker_budget: AgentDojo important_instructions template
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 6
    attacker_budget: 6
    sample_size: 6
- id: ev_sand_cl35_clean
  defence_id: sand_cl35
  benchmark: AgentDyn, 60 user tasks without injection (Shopping, GitHub, DailyLife; mean over the three suites)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn user-task check
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '60'
  pages:
    benchmark: 6
    sample_size: 6
- id: ev_sand_cl35_att
  defence_id: sand_cl35
  benchmark: AgentDyn, 560 injection cases with the important_instructions attack (mean over the three suites)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn user-task check; attacker goal executed
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  attacker_budget: AgentDojo important_instructions template
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 6
    attacker_budget: 6
    sample_size: 6
- id: ev_sand_l33_clean
  defence_id: sand_l33
  benchmark: AgentDyn, 60 user tasks without injection (Shopping, GitHub, DailyLife; mean over the three suites)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn user-task check
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '60'
  pages:
    benchmark: 6
    sample_size: 6
- id: ev_sand_l33_att
  defence_id: sand_l33
  benchmark: AgentDyn, 560 injection cases with the important_instructions attack (mean over the three suites)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn user-task check; attacker goal executed
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  attacker_budget: AgentDojo important_instructions template
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 6
    attacker_budget: 6
    sample_size: 6
- id: ev_spot_g4o_clean
  defence_id: spot_g4o
  benchmark: AgentDyn, 60 user tasks without injection (Shopping, GitHub, DailyLife; mean over the three suites)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn user-task check
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '60'
  pages:
    benchmark: 6
    sample_size: 6
- id: ev_spot_g4o_att
  defence_id: spot_g4o
  benchmark: AgentDyn, 560 injection cases with the important_instructions attack (mean over the three suites)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn user-task check; attacker goal executed
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  attacker_budget: AgentDojo important_instructions template
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 6
    attacker_budget: 6
    sample_size: 6
- id: ev_spot_gem_clean
  defence_id: spot_gem
  benchmark: AgentDyn, 60 user tasks without injection (Shopping, GitHub, DailyLife; mean over the three suites)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn user-task check
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '60'
  pages:
    benchmark: 6
    sample_size: 6
- id: ev_spot_gem_att
  defence_id: spot_gem
  benchmark: AgentDyn, 560 injection cases with the important_instructions attack (mean over the three suites)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn user-task check; attacker goal executed
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  attacker_budget: AgentDojo important_instructions template
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 6
    attacker_budget: 6
    sample_size: 6
- id: ev_spot_cl35_clean
  defence_id: spot_cl35
  benchmark: AgentDyn, 60 user tasks without injection (Shopping, GitHub, DailyLife; mean over the three suites)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn user-task check
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '60'
  pages:
    benchmark: 6
    sample_size: 6
- id: ev_spot_cl35_att
  defence_id: spot_cl35
  benchmark: AgentDyn, 560 injection cases with the important_instructions attack (mean over the three suites)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn user-task check; attacker goal executed
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  attacker_budget: AgentDojo important_instructions template
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 6
    attacker_budget: 6
    sample_size: 6
- id: ev_spot_l33_clean
  defence_id: spot_l33
  benchmark: AgentDyn, 60 user tasks without injection (Shopping, GitHub, DailyLife; mean over the three suites)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn user-task check
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '60'
  pages:
    benchmark: 6
    sample_size: 6
- id: ev_spot_l33_att
  defence_id: spot_l33
  benchmark: AgentDyn, 560 injection cases with the important_instructions attack (mean over the three suites)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn user-task check; attacker goal executed
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  attacker_budget: AgentDojo important_instructions template
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 6
    attacker_budget: 6
    sample_size: 6
- id: ev_none_msa_clean
  defence_id: msa70b
  benchmark: AgentDyn, 60 user tasks without injection (Shopping, GitHub, DailyLife; mean over the three suites)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDyn user-task check
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '60'
  pages:
    benchmark: 6
    sample_size: 6
- id: ev_none_msa_att
  defence_id: msa70b
  benchmark: AgentDyn, 560 injection cases with the important_instructions attack (mean over the three suites)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDyn user-task check; attacker goal executed
  layout_same_as_defence: 'yes'
  measured_by: agentdyn_arxiv2026
  page: 7
  attacker_budget: AgentDojo important_instructions template
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 6
    attacker_budget: 6
    sample_size: 6
results:
- id: r_none_g4o_u
  evaluation_id: ev_none_g4o_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility (no attack)
  value: '53.33'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_none_g4o_ua
  evaluation_id: ev_none_g4o_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (under attack)
  value: '55.52'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_none_g4o_asr
  evaluation_id: ev_none_g4o_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '37.80'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_none_gem_u
  evaluation_id: ev_none_gem_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility (no attack)
  value: '51.67'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_none_gem_ua
  evaluation_id: ev_none_gem_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (under attack)
  value: '56.95'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_none_gem_asr
  evaluation_id: ev_none_gem_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '20.61'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_none_cl35_u
  evaluation_id: ev_none_cl35_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility (no attack)
  value: '60.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_none_cl35_ua
  evaluation_id: ev_none_cl35_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (under attack)
  value: '55.43'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_none_cl35_asr
  evaluation_id: ev_none_cl35_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '11.89'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_none_l33_u
  evaluation_id: ev_none_l33_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility (no attack)
  value: '10.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_none_l33_ua
  evaluation_id: ev_none_l33_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (under attack)
  value: '6.15'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_none_l33_asr
  evaluation_id: ev_none_l33_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '11.91'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_sand_g4o_u
  evaluation_id: ev_sand_g4o_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility (no attack)
  value: '63.33'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_sand_g4o_ua
  evaluation_id: ev_sand_g4o_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (under attack)
  value: '56.13'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_sand_g4o_asr
  evaluation_id: ev_sand_g4o_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '31.17'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_sand_gem_u
  evaluation_id: ev_sand_gem_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility (no attack)
  value: '51.67'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_sand_gem_ua
  evaluation_id: ev_sand_gem_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (under attack)
  value: '49.08'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_sand_gem_asr
  evaluation_id: ev_sand_gem_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '23.94'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_sand_cl35_u
  evaluation_id: ev_sand_cl35_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility (no attack)
  value: '56.67'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_sand_cl35_ua
  evaluation_id: ev_sand_cl35_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (under attack)
  value: '55.67'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_sand_cl35_asr
  evaluation_id: ev_sand_cl35_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '11.26'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_sand_l33_u
  evaluation_id: ev_sand_l33_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility (no attack)
  value: '5.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_sand_l33_ua
  evaluation_id: ev_sand_l33_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (under attack)
  value: '7.35'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_sand_l33_asr
  evaluation_id: ev_sand_l33_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '9.96'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_spot_g4o_u
  evaluation_id: ev_spot_g4o_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility (no attack)
  value: '55.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_spot_g4o_ua
  evaluation_id: ev_spot_g4o_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (under attack)
  value: '52.24'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_spot_g4o_asr
  evaluation_id: ev_spot_g4o_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '27.61'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_spot_gem_u
  evaluation_id: ev_spot_gem_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility (no attack)
  value: '58.33'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_spot_gem_ua
  evaluation_id: ev_spot_gem_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (under attack)
  value: '52.61'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_spot_gem_asr
  evaluation_id: ev_spot_gem_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '16.87'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_spot_cl35_u
  evaluation_id: ev_spot_cl35_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility (no attack)
  value: '58.33'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_spot_cl35_ua
  evaluation_id: ev_spot_cl35_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (under attack)
  value: '57.09'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_spot_cl35_asr
  evaluation_id: ev_spot_cl35_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '3.80'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_spot_l33_u
  evaluation_id: ev_spot_l33_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility (no attack)
  value: '10.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_spot_l33_ua
  evaluation_id: ev_spot_l33_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (under attack)
  value: '6.78'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_spot_l33_asr
  evaluation_id: ev_spot_l33_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '14.85'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_none_msa_u
  evaluation_id: ev_none_msa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility (no attack)
  value: '55.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_none_msa_ua
  evaluation_id: ev_none_msa_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (under attack)
  value: '53.35'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_none_msa_asr
  evaluation_id: ev_none_msa_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '8.98'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
comparisons:
- id: c_sand_g4o_clean
  left: ev_sand_g4o_clean
  right: ev_none_g4o_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sand_g4o_att
  left: ev_sand_g4o_att
  right: ev_none_g4o_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_spot_g4o_clean
  left: ev_spot_g4o_clean
  right: ev_none_g4o_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_spot_g4o_att
  left: ev_spot_g4o_att
  right: ev_none_g4o_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sand_gem_clean
  left: ev_sand_gem_clean
  right: ev_none_gem_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sand_gem_att
  left: ev_sand_gem_att
  right: ev_none_gem_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_spot_gem_clean
  left: ev_spot_gem_clean
  right: ev_none_gem_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_spot_gem_att
  left: ev_spot_gem_att
  right: ev_none_gem_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sand_cl35_clean
  left: ev_sand_cl35_clean
  right: ev_none_cl35_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sand_cl35_att
  left: ev_sand_cl35_att
  right: ev_none_cl35_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_spot_cl35_clean
  left: ev_spot_cl35_clean
  right: ev_none_cl35_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_spot_cl35_att
  left: ev_spot_cl35_att
  right: ev_none_cl35_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sand_l33_clean
  left: ev_sand_l33_clean
  right: ev_none_l33_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sand_l33_att
  left: ev_sand_l33_att
  right: ev_none_l33_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_spot_l33_clean
  left: ev_spot_l33_clean
  right: ev_none_l33_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_spot_l33_att
  left: ev_spot_l33_att
  right: ev_none_l33_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_msa_vs_l33_clean
  left: ev_none_msa_clean
  right: ev_none_l33_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: Meta SecAlign 70B against its base Llama 3.3 70B
  page: 7
- id: c_msa_vs_l33_att
  left: ev_none_msa_att
  right: ev_none_l33_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: Meta SecAlign 70B against its base Llama 3.3 70B
  page: 7
claims:
- id: k1
  text: almost all existing defences are either not secure enough or over-defend
  location: abstract
  anchor: almost all existing defenses are either not secure enough
  jobs:
  - R1
  - R2
  supported_by:
  - c_sand_g4o_clean
  - c_sand_g4o_att
  - c_spot_g4o_clean
  - c_spot_g4o_att
  - c_sand_gem_clean
  - c_sand_gem_att
  - c_spot_gem_clean
  - c_spot_gem_att
  - c_sand_cl35_clean
  - c_sand_cl35_att
  - c_spot_cl35_clean
  - c_spot_cl35_att
  - c_sand_l33_clean
  - c_sand_l33_att
  - c_spot_l33_clean
  - c_spot_l33_att
  - c_msa_vs_l33_clean
  - c_msa_vs_l33_att
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 3 (ASR)
    R2clean: Table 3 (utility, no attack)
    R2attack: Table 3 (utility under attack)
schema_gaps:
- where: defences spot_*
  note: the Spotlighting variant (delimiting, datamarking or encoding) is not stated; coded delimiters_markers as in the other records
- where: evaluations ev_*
  note: the authors add 'Complete all tasks automatically without requesting user confirmation' to the system message for every agent (p. 8)

`icon_arxiv2026` (2602.20708)

In [ ]:
%%writefile reports/records/icon_arxiv2026.yaml
report:
  id: icon_arxiv2026
  slug: '2602.20708'
  title: 'ICON: Indirect Prompt Injection Defense for Agents based on Inference-Time Correction'
  first_author: Che Wang
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2602.20708'
  version_read: arXiv v1
  first_date: '2026-02-24'
  read: F
  role: proposes
  code_release: none found in the paper
  extraction_scope: 'whole report (11 pages): Table 2 columns No Defense, Repeat Prompt, Delimiting and ICON for the three attacks on the three models (the Avg rows, which pool the attacks, not coded) and the BU column; the
    abstract claim (MELON, a tool filter, and the LLM-Detector Qwen3Guard, a detector, are left out; Table 3 detection rates, Table 4 cost and the multimodal results not coded)'
defences:
- id: icon_q3
  name: ICON
  origin: this_report
  base_checkpoint: Qwen-3-8B
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 1
  components:
  - 'Latent Space Trace Prober: a small CNN and MLP classifier on attention focus features from selected layers and heads, trained on 255 TrojanTools samples; the LLM is not trained'
  - 'Mitigating Rectifier: on detection, the top attention weights of flagged heads are scaled down before the softmax'
  layout: ReAct agent; tool outputs carry the injection
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: a Mitigating Rectifier performs surgical attention steering
  pages:
    components: 6
- id: rep_q3
  name: Repeat Prompt
  origin: other_report
  base_checkpoint: Qwen-3-8B
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  layout: ReAct agent; tool outputs carry the injection
  layout_compat: *id001
  anchor: which reiterates system instructions
- id: del_q3
  name: Delimiting
  origin: other_report
  base_checkpoint: Qwen-3-8B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  layout: ReAct agent; tool outputs carry the injection
  layout_compat: *id001
  anchor: which uses special tokens to isolate untrusted content
- id: icon_l31
  name: ICON
  origin: this_report
  base_checkpoint: Llama-3.1-8B
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 1
  components:
  - 'Latent Space Trace Prober: a small CNN and MLP classifier on attention focus features from selected layers and heads, trained on 255 TrojanTools samples; the LLM is not trained'
  - 'Mitigating Rectifier: on detection, the top attention weights of flagged heads are scaled down before the softmax'
  layout: ReAct agent; tool outputs carry the injection
  layout_compat: *id001
  anchor: a Mitigating Rectifier performs surgical attention steering
  pages:
    components: 6
- id: rep_l31
  name: Repeat Prompt
  origin: other_report
  base_checkpoint: Llama-3.1-8B
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  layout: ReAct agent; tool outputs carry the injection
  layout_compat: *id001
  anchor: which reiterates system instructions
- id: del_l31
  name: Delimiting
  origin: other_report
  base_checkpoint: Llama-3.1-8B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  layout: ReAct agent; tool outputs carry the injection
  layout_compat: *id001
  anchor: which uses special tokens to isolate untrusted content
- id: icon_m8
  name: ICON
  origin: this_report
  base_checkpoint: Mistral-8B (Ministral)
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 1
  components:
  - 'Latent Space Trace Prober: a small CNN and MLP classifier on attention focus features from selected layers and heads, trained on 255 TrojanTools samples; the LLM is not trained'
  - 'Mitigating Rectifier: on detection, the top attention weights of flagged heads are scaled down before the softmax'
  layout: ReAct agent; tool outputs carry the injection
  layout_compat: *id001
  anchor: a Mitigating Rectifier performs surgical attention steering
  pages:
    components: 6
- id: rep_m8
  name: Repeat Prompt
  origin: other_report
  base_checkpoint: Mistral-8B (Ministral)
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  layout: ReAct agent; tool outputs carry the injection
  layout_compat: *id001
  anchor: which reiterates system instructions
- id: del_m8
  name: Delimiting
  origin: other_report
  base_checkpoint: Mistral-8B (Ministral)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  layout: ReAct agent; tool outputs carry the injection
  layout_compat: *id001
  anchor: which uses special tokens to isolate untrusted content
evaluations:
- id: ev_q3_none_clean
  defence_id: none:Qwen-3-8B
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_q3_none_ign
  defence_id: none:Qwen-3-8B
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), Ignore Instruction injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: Ignore Instruction attack prompts
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_q3_rep_ign
  defence_id: rep_q3
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), Ignore Instruction injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: Ignore Instruction attack prompts
  decoding: not reported
- id: ev_q3_del_ign
  defence_id: del_q3
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), Ignore Instruction injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: Ignore Instruction attack prompts
  decoding: not reported
- id: ev_q3_icon_ign
  defence_id: icon_q3
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), Ignore Instruction injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: Ignore Instruction attack prompts
  decoding: not reported
- id: ev_q3_none_comb
  defence_id: none:Qwen-3-8B
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), Combined Attacks injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: Combined Attacks attack prompts
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_q3_rep_comb
  defence_id: rep_q3
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), Combined Attacks injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: Combined Attacks attack prompts
  decoding: not reported
- id: ev_q3_del_comb
  defence_id: del_q3
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), Combined Attacks injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: Combined Attacks attack prompts
  decoding: not reported
- id: ev_q3_icon_comb
  defence_id: icon_q3
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), Combined Attacks injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: Combined Attacks attack prompts
  decoding: not reported
- id: ev_q3_none_troj
  defence_id: none:Qwen-3-8B
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), TrojanTools injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: TrojanTools attack prompts
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_q3_rep_troj
  defence_id: rep_q3
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), TrojanTools injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: TrojanTools attack prompts
  decoding: not reported
- id: ev_q3_del_troj
  defence_id: del_q3
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), TrojanTools injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: TrojanTools attack prompts
  decoding: not reported
- id: ev_q3_icon_troj
  defence_id: icon_q3
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), TrojanTools injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: TrojanTools attack prompts
  decoding: not reported
- id: ev_l31_none_clean
  defence_id: none:Llama-3.1-8B
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_l31_none_ign
  defence_id: none:Llama-3.1-8B
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), Ignore Instruction injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: Ignore Instruction attack prompts
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_l31_rep_ign
  defence_id: rep_l31
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), Ignore Instruction injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: Ignore Instruction attack prompts
  decoding: not reported
- id: ev_l31_del_ign
  defence_id: del_l31
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), Ignore Instruction injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: Ignore Instruction attack prompts
  decoding: not reported
- id: ev_l31_icon_ign
  defence_id: icon_l31
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), Ignore Instruction injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: Ignore Instruction attack prompts
  decoding: not reported
- id: ev_l31_none_comb
  defence_id: none:Llama-3.1-8B
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), Combined Attacks injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: Combined Attacks attack prompts
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_l31_rep_comb
  defence_id: rep_l31
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), Combined Attacks injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: Combined Attacks attack prompts
  decoding: not reported
- id: ev_l31_del_comb
  defence_id: del_l31
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), Combined Attacks injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: Combined Attacks attack prompts
  decoding: not reported
- id: ev_l31_icon_comb
  defence_id: icon_l31
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), Combined Attacks injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: Combined Attacks attack prompts
  decoding: not reported
- id: ev_l31_none_troj
  defence_id: none:Llama-3.1-8B
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), TrojanTools injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: TrojanTools attack prompts
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_l31_rep_troj
  defence_id: rep_l31
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), TrojanTools injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: TrojanTools attack prompts
  decoding: not reported
- id: ev_l31_del_troj
  defence_id: del_l31
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), TrojanTools injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: TrojanTools attack prompts
  decoding: not reported
- id: ev_l31_icon_troj
  defence_id: icon_l31
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), TrojanTools injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: TrojanTools attack prompts
  decoding: not reported
- id: ev_m8_none_clean
  defence_id: none:Mistral-8B (Ministral)
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_m8_none_ign
  defence_id: none:Mistral-8B (Ministral)
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), Ignore Instruction injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: Ignore Instruction attack prompts
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_m8_rep_ign
  defence_id: rep_m8
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), Ignore Instruction injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: Ignore Instruction attack prompts
  decoding: not reported
- id: ev_m8_del_ign
  defence_id: del_m8
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), Ignore Instruction injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: Ignore Instruction attack prompts
  decoding: not reported
- id: ev_m8_icon_ign
  defence_id: icon_m8
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), Ignore Instruction injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: Ignore Instruction attack prompts
  decoding: not reported
- id: ev_m8_none_comb
  defence_id: none:Mistral-8B (Ministral)
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), Combined Attacks injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: Combined Attacks attack prompts
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_m8_rep_comb
  defence_id: rep_m8
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), Combined Attacks injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: Combined Attacks attack prompts
  decoding: not reported
- id: ev_m8_del_comb
  defence_id: del_m8
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), Combined Attacks injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: Combined Attacks attack prompts
  decoding: not reported
- id: ev_m8_icon_comb
  defence_id: icon_m8
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), Combined Attacks injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: Combined Attacks attack prompts
  decoding: not reported
- id: ev_m8_none_troj
  defence_id: none:Mistral-8B (Ministral)
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), TrojanTools injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: TrojanTools attack prompts
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_m8_rep_troj
  defence_id: rep_m8
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), TrojanTools injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: TrojanTools attack prompts
  decoding: not reported
- id: ev_m8_del_troj
  defence_id: del_m8
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), TrojanTools injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: TrojanTools attack prompts
  decoding: not reported
- id: ev_m8_icon_troj
  defence_id: icon_m8
  benchmark: agent tool-use tasks (the paper evaluates on InjecAgent and AgentDojo; Table 2 does not say which), TrojanTools injections
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met; user task completed
  layout_same_as_defence: 'yes'
  measured_by: icon_arxiv2026
  page: 7
  attacker_budget: TrojanTools attack prompts
  decoding: not reported
results:
- id: r_q3_bu
  evaluation_id: ev_q3_none_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '69.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_q3_none_ign_asr
  evaluation_id: ev_q3_none_ign
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore Instruction)
  value: '25.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_q3_none_ign_ua
  evaluation_id: ev_q3_none_ign
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (Ignore Instruction)
  value: '49.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_q3_rep_ign_asr
  evaluation_id: ev_q3_rep_ign
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore Instruction)
  value: '11.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_q3_rep_ign_ua
  evaluation_id: ev_q3_rep_ign
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (Ignore Instruction)
  value: '54.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_q3_del_ign_asr
  evaluation_id: ev_q3_del_ign
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore Instruction)
  value: '15.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_q3_del_ign_ua
  evaluation_id: ev_q3_del_ign
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (Ignore Instruction)
  value: '52.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_q3_icon_ign_asr
  evaluation_id: ev_q3_icon_ign
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore Instruction)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_q3_icon_ign_ua
  evaluation_id: ev_q3_icon_ign
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (Ignore Instruction)
  value: '58.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_q3_none_comb_asr
  evaluation_id: ev_q3_none_comb
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Combined Attacks)
  value: '30.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_q3_none_comb_ua
  evaluation_id: ev_q3_none_comb
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (Combined Attacks)
  value: '44.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_q3_rep_comb_asr
  evaluation_id: ev_q3_rep_comb
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Combined Attacks)
  value: '13.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_q3_rep_comb_ua
  evaluation_id: ev_q3_rep_comb
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (Combined Attacks)
  value: '53.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_q3_del_comb_asr
  evaluation_id: ev_q3_del_comb
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Combined Attacks)
  value: '18.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_q3_del_comb_ua
  evaluation_id: ev_q3_del_comb
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (Combined Attacks)
  value: '50.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_q3_icon_comb_asr
  evaluation_id: ev_q3_icon_comb
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Combined Attacks)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_q3_icon_comb_ua
  evaluation_id: ev_q3_icon_comb
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (Combined Attacks)
  value: '52.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_q3_none_troj_asr
  evaluation_id: ev_q3_none_troj
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (TrojanTools)
  value: '42.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_q3_none_troj_ua
  evaluation_id: ev_q3_none_troj
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (TrojanTools)
  value: '35.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_q3_rep_troj_asr
  evaluation_id: ev_q3_rep_troj
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (TrojanTools)
  value: '25.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_q3_rep_troj_ua
  evaluation_id: ev_q3_rep_troj
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (TrojanTools)
  value: '45.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_q3_del_troj_asr
  evaluation_id: ev_q3_del_troj
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (TrojanTools)
  value: '31.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_q3_del_troj_ua
  evaluation_id: ev_q3_del_troj
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (TrojanTools)
  value: '43.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_q3_icon_troj_asr
  evaluation_id: ev_q3_icon_troj
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (TrojanTools)
  value: '1.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_q3_icon_troj_ua
  evaluation_id: ev_q3_icon_troj
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (TrojanTools)
  value: '50.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_bu
  evaluation_id: ev_l31_none_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '34.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_none_ign_asr
  evaluation_id: ev_l31_none_ign
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore Instruction)
  value: '23.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_none_ign_ua
  evaluation_id: ev_l31_none_ign
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (Ignore Instruction)
  value: '25.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_rep_ign_asr
  evaluation_id: ev_l31_rep_ign
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore Instruction)
  value: '14.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_rep_ign_ua
  evaluation_id: ev_l31_rep_ign
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (Ignore Instruction)
  value: '30.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_del_ign_asr
  evaluation_id: ev_l31_del_ign
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore Instruction)
  value: '16.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_del_ign_ua
  evaluation_id: ev_l31_del_ign
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (Ignore Instruction)
  value: '28.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_icon_ign_asr
  evaluation_id: ev_l31_icon_ign
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore Instruction)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_icon_ign_ua
  evaluation_id: ev_l31_icon_ign
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (Ignore Instruction)
  value: '32.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_none_comb_asr
  evaluation_id: ev_l31_none_comb
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Combined Attacks)
  value: '36.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_none_comb_ua
  evaluation_id: ev_l31_none_comb
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (Combined Attacks)
  value: '22.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_rep_comb_asr
  evaluation_id: ev_l31_rep_comb
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Combined Attacks)
  value: '20.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_rep_comb_ua
  evaluation_id: ev_l31_rep_comb
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (Combined Attacks)
  value: '28.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_del_comb_asr
  evaluation_id: ev_l31_del_comb
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Combined Attacks)
  value: '22.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_del_comb_ua
  evaluation_id: ev_l31_del_comb
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (Combined Attacks)
  value: '25.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_icon_comb_asr
  evaluation_id: ev_l31_icon_comb
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Combined Attacks)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_icon_comb_ua
  evaluation_id: ev_l31_icon_comb
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (Combined Attacks)
  value: '29.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_none_troj_asr
  evaluation_id: ev_l31_none_troj
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (TrojanTools)
  value: '47.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_none_troj_ua
  evaluation_id: ev_l31_none_troj
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (TrojanTools)
  value: '14.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_rep_troj_asr
  evaluation_id: ev_l31_rep_troj
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (TrojanTools)
  value: '28.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_rep_troj_ua
  evaluation_id: ev_l31_rep_troj
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (TrojanTools)
  value: '25.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_del_troj_asr
  evaluation_id: ev_l31_del_troj
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (TrojanTools)
  value: '30.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_del_troj_ua
  evaluation_id: ev_l31_del_troj
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (TrojanTools)
  value: '24.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_icon_troj_asr
  evaluation_id: ev_l31_icon_troj
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (TrojanTools)
  value: '0.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_l31_icon_troj_ua
  evaluation_id: ev_l31_icon_troj
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (TrojanTools)
  value: '27.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_bu
  evaluation_id: ev_m8_none_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '20.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_none_ign_asr
  evaluation_id: ev_m8_none_ign
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore Instruction)
  value: '16.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_none_ign_ua
  evaluation_id: ev_m8_none_ign
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (Ignore Instruction)
  value: '12.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_rep_ign_asr
  evaluation_id: ev_m8_rep_ign
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore Instruction)
  value: '10.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_rep_ign_ua
  evaluation_id: ev_m8_rep_ign
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (Ignore Instruction)
  value: '17.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_del_ign_asr
  evaluation_id: ev_m8_del_ign
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore Instruction)
  value: '12.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_del_ign_ua
  evaluation_id: ev_m8_del_ign
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (Ignore Instruction)
  value: '16.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_icon_ign_asr
  evaluation_id: ev_m8_icon_ign
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore Instruction)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_icon_ign_ua
  evaluation_id: ev_m8_icon_ign
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (Ignore Instruction)
  value: '17.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_none_comb_asr
  evaluation_id: ev_m8_none_comb
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Combined Attacks)
  value: '33.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_none_comb_ua
  evaluation_id: ev_m8_none_comb
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (Combined Attacks)
  value: '11.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_rep_comb_asr
  evaluation_id: ev_m8_rep_comb
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Combined Attacks)
  value: '19.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_rep_comb_ua
  evaluation_id: ev_m8_rep_comb
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (Combined Attacks)
  value: '15.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_del_comb_asr
  evaluation_id: ev_m8_del_comb
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Combined Attacks)
  value: '23.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_del_comb_ua
  evaluation_id: ev_m8_del_comb
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (Combined Attacks)
  value: '14.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_icon_comb_asr
  evaluation_id: ev_m8_icon_comb
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Combined Attacks)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_icon_comb_ua
  evaluation_id: ev_m8_icon_comb
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (Combined Attacks)
  value: '16.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_none_troj_asr
  evaluation_id: ev_m8_none_troj
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (TrojanTools)
  value: '42.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_none_troj_ua
  evaluation_id: ev_m8_none_troj
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (TrojanTools)
  value: '8.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_rep_troj_asr
  evaluation_id: ev_m8_rep_troj
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (TrojanTools)
  value: '23.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_rep_troj_ua
  evaluation_id: ev_m8_rep_troj
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (TrojanTools)
  value: '12.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_del_troj_asr
  evaluation_id: ev_m8_del_troj
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (TrojanTools)
  value: '27.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_del_troj_ua
  evaluation_id: ev_m8_del_troj
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (TrojanTools)
  value: '10.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_icon_troj_asr
  evaluation_id: ev_m8_icon_troj
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (TrojanTools)
  value: '0.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_m8_icon_troj_ua
  evaluation_id: ev_m8_icon_troj
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA (TrojanTools)
  value: '13.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
comparisons:
- id: c_q3_rep_ign
  left: ev_q3_rep_ign
  right: ev_q3_none_ign
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q3_del_ign
  left: ev_q3_del_ign
  right: ev_q3_none_ign
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q3_icon_ign
  left: ev_q3_icon_ign
  right: ev_q3_none_ign
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q3_rep_comb
  left: ev_q3_rep_comb
  right: ev_q3_none_comb
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q3_del_comb
  left: ev_q3_del_comb
  right: ev_q3_none_comb
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q3_icon_comb
  left: ev_q3_icon_comb
  right: ev_q3_none_comb
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q3_rep_troj
  left: ev_q3_rep_troj
  right: ev_q3_none_troj
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q3_del_troj
  left: ev_q3_del_troj
  right: ev_q3_none_troj
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q3_icon_troj
  left: ev_q3_icon_troj
  right: ev_q3_none_troj
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l31_rep_ign
  left: ev_l31_rep_ign
  right: ev_l31_none_ign
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l31_del_ign
  left: ev_l31_del_ign
  right: ev_l31_none_ign
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l31_icon_ign
  left: ev_l31_icon_ign
  right: ev_l31_none_ign
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l31_rep_comb
  left: ev_l31_rep_comb
  right: ev_l31_none_comb
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l31_del_comb
  left: ev_l31_del_comb
  right: ev_l31_none_comb
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l31_icon_comb
  left: ev_l31_icon_comb
  right: ev_l31_none_comb
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l31_rep_troj
  left: ev_l31_rep_troj
  right: ev_l31_none_troj
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l31_del_troj
  left: ev_l31_del_troj
  right: ev_l31_none_troj
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l31_icon_troj
  left: ev_l31_icon_troj
  right: ev_l31_none_troj
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m8_rep_ign
  left: ev_m8_rep_ign
  right: ev_m8_none_ign
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m8_del_ign
  left: ev_m8_del_ign
  right: ev_m8_none_ign
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m8_icon_ign
  left: ev_m8_icon_ign
  right: ev_m8_none_ign
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m8_rep_comb
  left: ev_m8_rep_comb
  right: ev_m8_none_comb
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m8_del_comb
  left: ev_m8_del_comb
  right: ev_m8_none_comb
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m8_icon_comb
  left: ev_m8_icon_comb
  right: ev_m8_none_comb
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m8_rep_troj
  left: ev_m8_rep_troj
  right: ev_m8_none_troj
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m8_del_troj
  left: ev_m8_del_troj
  right: ev_m8_none_troj
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m8_icon_troj
  left: ev_m8_icon_troj
  right: ev_m8_none_troj
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
claims:
- id: k1
  text: ICON reaches 0.4% attack success, matching commercial detectors, with a large gain in task utility
  location: abstract
  anchor: ICON achieves a competitive 0.4% ASR, matching commercial grade detectors
  jobs:
  - R1
  - R2
  supported_by:
  - c_q3_icon_ign
  - c_q3_icon_comb
  - c_q3_icon_troj
  - c_l31_icon_ign
  - c_l31_icon_comb
  - c_l31_icon_troj
  - c_m8_icon_ign
  - c_m8_icon_comb
  - c_m8_icon_troj
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 2 (ASR)
    R2clean: Table 2 gives BU for the undefended model only; not reported for any defence
    R2attack: Table 2 (UA)
schema_gaps:
- where: defences icon_*
  note: the prober is a small classifier fitted on 255 samples and the LLM is not trained, so axis 2 is fixed_inference_rule and both training fields are not_applicable, as coding rule 11 treats a direction fitted from examples
- where: evaluations ev_*_troj
  note: ICON's prober is trained on TrojanTools (p. 6), and TrojanTools is also one of the three attacks in Table 2, so that row is not out of distribution for ICON
- where: evaluations ev_*
  note: Table 2 gives benign utility (BU) for the undefended model only, so the defences' clean utility is not reported

`ihchallenge_arxiv2026` (2603.10521)

In [ ]:
%%writefile reports/records/ihchallenge_arxiv2026.yaml
report:
  id: ihchallenge_arxiv2026
  slug: '2603.10521'
  title: 'IH-Challenge: A Training Dataset to Improve Instruction Hierarchy on Frontier LLMs'
  first_author: Chuan Guo
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2603.10521'
  version_read: arXiv v1
  first_date: '2026-03-11'
  read: F
  role: proposes
  code_release: dataset released (huggingface.co/datasets/openai/ih-challenge, linked from the abstract); training code not released
  extraction_scope: 'Sections 3 and 4: Table 5 (prompt injection) and Table 2 (overrefusal and capability); the abstract claim (Tables 3 and 4, which test conflicts between system, developer and user messages rather than injected
    data, Figures 4 to 6, Table 6 and Table 7 not coded)'
defences:
- id: ihc_g5m
  name: GPT-5-Mini-R (RL on IH-Challenge with online adversarial attacks)
  origin: this_report
  base_checkpoint: GPT-5-Mini
  axis1:
  - chat_roles
  axis2:
  - adversarial_rl
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  components:
  - task skeletons with higher-priority instructions and Python graders (Single-Constraint, Multi-Constraint, Input-Conditioned, Anti-Overrefusal)
  - a frozen attacker model writes the lower-priority conflicting message online against the current defender, under a tool-call budget
  - policy-gradient updates on grader rewards, mixed with capability tasks
  layout: priority order system, developer, user, tool
  released_weights: no (closed model)
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: supported
    agent_tool_result: supported
  anchor: We train the defender using RL, where at each step we synthesize
  pages:
    layout: 4
    layout_compat: 10
evaluations:
- id: ev_int_base
  defence_id: none:GPT-5-Mini
  benchmark: 'Internal PI Benchmark: static adversarially generated agentic injections'
  attacker_type: static
  attacker_access: black_box
  judge: not described
  layout_same_as_defence: 'yes'
  measured_by: ihchallenge_arxiv2026
  page: 10
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_int_r
  defence_id: ihc_g5m
  benchmark: 'Internal PI Benchmark: static adversarially generated agentic injections'
  attacker_type: static
  attacker_access: black_box
  judge: not described
  layout_same_as_defence: 'yes'
  measured_by: ihchallenge_arxiv2026
  page: 10
  decoding: not reported
- id: ev_cse_base
  defence_id: none:GPT-5-Mini
  benchmark: CyberSecEval 2 (instructions in tool output)
  attacker_type: static
  attacker_access: black_box
  judge: not described
  layout_same_as_defence: 'yes'
  measured_by: ihchallenge_arxiv2026
  page: 10
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_cse_r
  defence_id: ihc_g5m
  benchmark: CyberSecEval 2 (instructions in tool output)
  attacker_type: static
  attacker_access: black_box
  judge: not described
  layout_same_as_defence: 'yes'
  measured_by: ihchallenge_arxiv2026
  page: 10
  decoding: not reported
- id: ev_ihor_base
  defence_id: none:GPT-5-Mini
  benchmark: 'IH-Challenge (overrefusal): higher-tier instruction changed so the request is allowed'
  attacker_type: none
  attacker_access: not_applicable
  judge: python grader
  layout_same_as_defence: 'yes'
  measured_by: ihchallenge_arxiv2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_ihor_r
  defence_id: ihc_g5m
  benchmark: 'IH-Challenge (overrefusal): higher-tier instruction changed so the request is allowed'
  attacker_type: none
  attacker_access: not_applicable
  judge: python grader
  layout_same_as_defence: 'yes'
  measured_by: ihchallenge_arxiv2026
  page: 8
  decoding: not reported
- id: ev_ttor_base
  defence_id: none:GPT-5-Mini
  benchmark: TensorTrust (overrefusal)
  attacker_type: none
  attacker_access: not_applicable
  judge: not described
  layout_same_as_defence: 'yes'
  measured_by: ihchallenge_arxiv2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_ttor_r
  defence_id: ihc_g5m
  benchmark: TensorTrust (overrefusal)
  attacker_type: none
  attacker_access: not_applicable
  judge: not described
  layout_same_as_defence: 'yes'
  measured_by: ihchallenge_arxiv2026
  page: 8
  decoding: not reported
- id: ev_gpqa_base
  defence_id: none:GPT-5-Mini
  benchmark: GPQA Diamond
  attacker_type: none
  attacker_access: not_applicable
  judge: answer key
  layout_same_as_defence: 'yes'
  measured_by: ihchallenge_arxiv2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_gpqa_r
  defence_id: ihc_g5m
  benchmark: GPQA Diamond
  attacker_type: none
  attacker_access: not_applicable
  judge: answer key
  layout_same_as_defence: 'yes'
  measured_by: ihchallenge_arxiv2026
  page: 8
  decoding: not reported
- id: ev_aime_base
  defence_id: none:GPT-5-Mini
  benchmark: AIME 2024
  attacker_type: none
  attacker_access: not_applicable
  judge: answer key
  layout_same_as_defence: 'yes'
  measured_by: ihchallenge_arxiv2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_aime_r
  defence_id: ihc_g5m
  benchmark: AIME 2024
  attacker_type: none
  attacker_access: not_applicable
  judge: answer key
  layout_same_as_defence: 'yes'
  measured_by: ihchallenge_arxiv2026
  page: 8
  decoding: not reported
- id: ev_win_base
  defence_id: none:GPT-5-Mini
  benchmark: private chat questions, win rate against o1
  attacker_type: none
  attacker_access: not_applicable
  judge: not described
  layout_same_as_defence: 'yes'
  measured_by: ihchallenge_arxiv2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_win_r
  defence_id: ihc_g5m
  benchmark: private chat questions, win rate against o1
  attacker_type: none
  attacker_access: not_applicable
  judge: not described
  layout_same_as_defence: 'yes'
  measured_by: ihchallenge_arxiv2026
  page: 8
  decoding: not reported
results:
- id: r_int_base
  evaluation_id: ev_int_base
  quantity: R1
  coverage: direct
  metric_as_printed: robustness (Table 5)
  value: '0.44'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 5
- id: r_int_r
  evaluation_id: ev_int_r
  quantity: R1
  coverage: direct
  metric_as_printed: robustness (Table 5)
  value: '1.00'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 5
- id: r_cse_base
  evaluation_id: ev_cse_base
  quantity: R1
  coverage: direct
  metric_as_printed: robustness (Table 5)
  value: '0.88'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 5
- id: r_cse_r
  evaluation_id: ev_cse_r
  quantity: R1
  coverage: direct
  metric_as_printed: robustness (Table 5)
  value: '0.91'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 5
- id: r_ihor_base
  evaluation_id: ev_ihor_base
  quantity: other
  coverage: direct
  metric_as_printed: IH-Challenge (overrefusal)
  value: '0.79'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_ihor_r
  evaluation_id: ev_ihor_r
  quantity: other
  coverage: direct
  metric_as_printed: IH-Challenge (overrefusal)
  value: '1.00'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_ttor_base
  evaluation_id: ev_ttor_base
  quantity: other
  coverage: direct
  metric_as_printed: TensorTrust (overrefusal)
  value: '0.91'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_ttor_r
  evaluation_id: ev_ttor_r
  quantity: other
  coverage: direct
  metric_as_printed: TensorTrust (overrefusal)
  value: '0.90'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_gpqa_base
  evaluation_id: ev_gpqa_base
  quantity: general_utility
  coverage: direct
  metric_as_printed: GPQA Diamond
  value: '0.83'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_gpqa_r
  evaluation_id: ev_gpqa_r
  quantity: general_utility
  coverage: direct
  metric_as_printed: GPQA Diamond
  value: '0.83'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_aime_base
  evaluation_id: ev_aime_base
  quantity: general_utility
  coverage: direct
  metric_as_printed: AIME 2024
  value: '0.93'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_aime_r
  evaluation_id: ev_aime_r
  quantity: general_utility
  coverage: direct
  metric_as_printed: AIME 2024
  value: '0.94'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_win_base
  evaluation_id: ev_win_base
  quantity: general_utility
  coverage: direct
  metric_as_printed: Chat WinRate vs. o1
  value: '0.71'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_win_r
  evaluation_id: ev_win_r
  quantity: general_utility
  coverage: direct
  metric_as_printed: Chat WinRate vs. o1
  value: '0.66'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
comparisons:
- id: c_int
  left: ev_int_r
  right: ev_int_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_cse
  left: ev_cse_r
  right: ev_cse_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ihor
  left: ev_ihor_r
  right: ev_ihor_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_ttor
  left: ev_ttor_r
  right: ev_ttor_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_gpqa
  left: ev_gpqa_r
  right: ev_gpqa_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_aime
  left: ev_aime_r
  right: ev_aime_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_win
  left: ev_win_r
  right: ev_win_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
claims:
- id: k1
  text: training on IH-Challenge saturates an internal agentic prompt injection test with minimal capability loss
  location: abstract
  anchor: saturates an internal static agentic prompt injection evaluation
  jobs:
  - R1
  - general_utility
  supported_by:
  - c_int
  - c_cse
  - c_gpqa
  - c_aime
  - c_win
  page: 1
  contradicted_by:
  - r_win_r
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 5 (robustness on prompt injection evaluations)
    general_utility: Table 2
schema_gaps:
- where: defence ihc_g5m
  note: the Anti-Overrefusal split trains the model to follow lower-tier requests that only look forbidden; the paper does not show the same strings used both as attacks and as allowed requests, so trains_instruction_pair
    is no
- where: results r_int_*
  note: the internal prompt injection set is not public and its grading is not described

`pismith_colm2026` (2603.13026)

In [ ]:
%%writefile reports/records/pismith_colm2026.yaml
report:
  id: pismith_colm2026
  slug: '2603.13026'
  title: 'PISmith: Reinforcement Learning-based Red Teaming for Prompt Injection Defenses'
  first_author: Chenlong Yin
  venue: COLM 2026
  peer_reviewed: 'yes'
  arxiv: '2603.13026'
  version_read: arXiv v2 (carries the COLM 2026 header)
  first_date: '2026-03-13'
  read: F
  role: re_measures
  code_release: https://github.com/albert-y1n/PISmith (linked from the abstract)
  extraction_scope: 'Sections 4.1 to 4.3 and Appendix E: Table 7 rows No Defense, Sandwich and Instructional (Qwen3-4B-Instruct-2507, means over the 13 PIArena benchmarks) and the Average row of Table 1 (Meta-SecAlign-8B,
    which has no undefended arm in the paper); the abstract claim (PromptArmor and DataFilter, which rewrite the data, and PIGuard, PromptGuard and DataSentinel, which are detectors, are left out; per-benchmark Tables 8 to
    15, the agent results of Tables 2, 3 and 5 and the ablations not coded)'
defences:
- id: sand_q
  name: Sandwich
  origin: other_report
  base_checkpoint: Qwen3-4B-Instruct-2507
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 23
  anchor: appends a reminder of the original instruction after the data
- id: instr_q
  name: Instructional
  origin: other_report
  base_checkpoint: Qwen3-4B-Instruct-2507
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 23
  anchor: prepends an explicit instruction warning the LLM to ignore
- id: msa8b
  name: Meta-SecAlign-8B (released model, re-measured)
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct (base of Meta-SecAlign-8B; not run here)
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  anchor: We first evaluate Meta-SecAlign-8B (Chen et al., 2025c), a state-of-the-art
evaluations:
- id: ev_none_clean
  defence_id: none:Qwen3-4B-Instruct-2507
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task accuracy against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 23
  decoding: not reported
  pages:
    benchmark: 6
  baseline_kind: off_the_shelf
- id: ev_none_direct
  defence_id: none:Qwen3-4B-Instruct-2507
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, Direct (naive) injection
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 23
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_none_comb
  defence_id: none:Qwen3-4B-Instruct-2507
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, Combined attack templates
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 23
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_none_tap
  defence_id: none:Qwen3-4B-Instruct-2507
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, TAP search per instance
  attacker_type: adaptive
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 23
  attacker_budget: per-instance search budget
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_none_pair
  defence_id: none:Qwen3-4B-Instruct-2507
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, PAIR search per instance
  attacker_type: adaptive
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 23
  attacker_budget: per-instance search budget
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_none_strat
  defence_id: none:Qwen3-4B-Instruct-2507
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, Strategy search per instance
  attacker_type: adaptive
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 23
  attacker_budget: per-instance search budget
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_none_pismith
  defence_id: none:Qwen3-4B-Instruct-2507
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, PISmith RL attacker (Qwen3-4B, trained on 100 Dolly Closed QA samples against the target)
  attacker_type: rl
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 23
  attacker_budget: 10 injected prompts per instance
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_sand_clean
  defence_id: sand_q
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task accuracy against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 23
  decoding: not reported
  pages:
    benchmark: 6
- id: ev_sand_direct
  defence_id: sand_q
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, Direct (naive) injection
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 23
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_sand_comb
  defence_id: sand_q
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, Combined attack templates
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 23
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_sand_tap
  defence_id: sand_q
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, TAP search per instance
  attacker_type: adaptive
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 23
  attacker_budget: per-instance search budget
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_sand_pair
  defence_id: sand_q
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, PAIR search per instance
  attacker_type: adaptive
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 23
  attacker_budget: per-instance search budget
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_sand_strat
  defence_id: sand_q
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, Strategy search per instance
  attacker_type: adaptive
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 23
  attacker_budget: per-instance search budget
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_sand_pismith
  defence_id: sand_q
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, PISmith RL attacker (Qwen3-4B, trained on 100 Dolly Closed QA samples against the target)
  attacker_type: rl
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 23
  attacker_budget: 10 injected prompts per instance
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_instr_clean
  defence_id: instr_q
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task accuracy against the reference answers
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 23
  decoding: not reported
  pages:
    benchmark: 6
- id: ev_instr_direct
  defence_id: instr_q
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, Direct (naive) injection
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 23
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_instr_comb
  defence_id: instr_q
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, Combined attack templates
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 23
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_instr_tap
  defence_id: instr_q
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, TAP search per instance
  attacker_type: adaptive
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 23
  attacker_budget: per-instance search budget
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_instr_pair
  defence_id: instr_q
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, PAIR search per instance
  attacker_type: adaptive
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 23
  attacker_budget: per-instance search budget
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_instr_strat
  defence_id: instr_q
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, Strategy search per instance
  attacker_type: adaptive
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 23
  attacker_budget: per-instance search budget
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_instr_pismith
  defence_id: instr_q
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, PISmith RL attacker (Qwen3-4B, trained on 100 Dolly Closed QA samples against the target)
  attacker_type: rl
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 23
  attacker_budget: 10 injected prompts per instance
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_msa_direct
  defence_id: msa8b
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, Direct (naive) injection
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_msa_comb
  defence_id: msa8b
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, Combined attack templates
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_msa_tap
  defence_id: msa8b
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, TAP search per instance
  attacker_type: adaptive
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 7
  attacker_budget: per-instance search budget
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_msa_pair
  defence_id: msa8b
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, PAIR search per instance
  attacker_type: adaptive
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 7
  attacker_budget: per-instance search budget
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_msa_strat
  defence_id: msa8b
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, Strategy search per instance
  attacker_type: adaptive
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 7
  attacker_budget: per-instance search budget
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_msa_rlh
  defence_id: msa8b
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, RL-Hammer attacker (same base, data and protocol as PISmith)
  attacker_type: rl
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 7
  attacker_budget: 10 injected prompts per instance
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_msa_pismith
  defence_id: msa8b
  benchmark: PIArena benchmarks (13 QA, RAG and long-context datasets), mean over benchmarks, PISmith RL attacker (Qwen3-4B, trained on 100 Dolly Closed QA samples against the target)
  attacker_type: rl
  attacker_access: black_box
  judge: GPT-4o-mini as judge of attack success
  layout_same_as_defence: 'yes'
  measured_by: pismith_colm2026
  page: 7
  attacker_budget: 10 injected prompts per instance
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
results:
- id: r_none_u
  evaluation_id: ev_none_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility
  value: '0.74'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_none_direct
  evaluation_id: ev_none_direct
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (Direct)
  value: '0.63'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_none_comb
  evaluation_id: ev_none_comb
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (Combined)
  value: '0.85'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_none_tap
  evaluation_id: ev_none_tap
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (TAP)
  value: '0.68'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_none_pair
  evaluation_id: ev_none_pair
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (PAIR)
  value: '0.68'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_none_strat
  evaluation_id: ev_none_strat
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (Strategy)
  value: '0.99'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_none_pismith_at10
  evaluation_id: ev_none_pismith
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@10 (PISmith)
  value: '1.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_none_pismith
  evaluation_id: ev_none_pismith
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (PISmith)
  value: '0.92'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_sand_u
  evaluation_id: ev_sand_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility
  value: '0.74'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_sand_direct
  evaluation_id: ev_sand_direct
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (Direct)
  value: '0.56'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_sand_comb
  evaluation_id: ev_sand_comb
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (Combined)
  value: '0.76'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_sand_tap
  evaluation_id: ev_sand_tap
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (TAP)
  value: '0.61'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_sand_pair
  evaluation_id: ev_sand_pair
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (PAIR)
  value: '0.62'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_sand_strat
  evaluation_id: ev_sand_strat
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (Strategy)
  value: '0.97'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_sand_pismith_at10
  evaluation_id: ev_sand_pismith
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@10 (PISmith)
  value: '0.99'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_sand_pismith
  evaluation_id: ev_sand_pismith
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (PISmith)
  value: '0.91'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_instr_u
  evaluation_id: ev_instr_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility
  value: '0.73'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_instr_direct
  evaluation_id: ev_instr_direct
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (Direct)
  value: '0.61'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_instr_comb
  evaluation_id: ev_instr_comb
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (Combined)
  value: '0.83'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_instr_tap
  evaluation_id: ev_instr_tap
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (TAP)
  value: '0.65'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_instr_pair
  evaluation_id: ev_instr_pair
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (PAIR)
  value: '0.67'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_instr_strat
  evaluation_id: ev_instr_strat
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (Strategy)
  value: '0.98'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_instr_pismith_at10
  evaluation_id: ev_instr_pismith
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@10 (PISmith)
  value: '0.99'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_instr_pismith
  evaluation_id: ev_instr_pismith
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (PISmith)
  value: '0.92'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_msa_direct
  evaluation_id: ev_msa_direct
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (Average)
  value: '0.04'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1, Average row
- id: r_msa_comb
  evaluation_id: ev_msa_comb
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (Average)
  value: '0.07'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1, Average row
- id: r_msa_tap
  evaluation_id: ev_msa_tap
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (Average)
  value: '0.11'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1, Average row
- id: r_msa_pair
  evaluation_id: ev_msa_pair
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (Average)
  value: '0.16'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1, Average row
- id: r_msa_strat
  evaluation_id: ev_msa_strat
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (Average)
  value: '0.21'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1, Average row
- id: r_msa_rlh
  evaluation_id: ev_msa_rlh
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (Average)
  value: '0.48'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1, Average row
- id: r_msa_pismith
  evaluation_id: ev_msa_pismith
  quantity: R1
  coverage: direct
  metric_as_printed: ASR@1 (Average)
  value: '0.87'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1, Average row
comparisons:
- id: c_sand_clean
  left: ev_sand_clean
  right: ev_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_sand_direct
  left: ev_sand_direct
  right: ev_none_direct
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_sand_comb
  left: ev_sand_comb
  right: ev_none_comb
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_sand_tap
  left: ev_sand_tap
  right: ev_none_tap
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_sand_pair
  left: ev_sand_pair
  right: ev_none_pair
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_sand_strat
  left: ev_sand_strat
  right: ev_none_strat
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_sand_pismith
  left: ev_sand_pismith
  right: ev_none_pismith
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_instr_clean
  left: ev_instr_clean
  right: ev_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_instr_direct
  left: ev_instr_direct
  right: ev_none_direct
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_instr_comb
  left: ev_instr_comb
  right: ev_none_comb
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_instr_tap
  left: ev_instr_tap
  right: ev_none_tap
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_instr_pair
  left: ev_instr_pair
  right: ev_none_pair
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_instr_strat
  left: ev_instr_strat
  right: ev_none_strat
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_instr_pismith
  left: ev_instr_pismith
  right: ev_none_pismith
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
claims:
- id: k1
  text: state-of-the-art defences remain vulnerable to adaptive attacks
  location: abstract
  anchor: state-of-the-art prompt injection defenses remain vulnerable to adaptive attacks
  jobs:
  - R1
  supported_by:
  - c_sand_pismith
  - c_instr_pismith
  - c_sand_tap
  - c_instr_tap
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 7 (means over 13 benchmarks; per-benchmark values in Tables 8 to 10) and Table 1
    R2clean: Table 7 (utility without attack)
schema_gaps:
- where: results r_msa_*
  note: Meta-SecAlign-8B is measured without its base model in the same protocol, so its results carry no comparison

`autonomytax_arxiv2026` (2603.19423)

In [ ]:
%%writefile reports/records/autonomytax_arxiv2026.yaml
report:
  id: autonomytax_arxiv2026
  slug: '2603.19423'
  title: 'The Autonomy Tax: Defense Training Breaks LLM Agents'
  first_author: Shawn Li
  venue: arXiv preprint
  peer_reviewed: 'no'
  doi: 10.18653/2026.emnlp-main.52
  arxiv: '2603.19423'
  version_read: arXiv v3
  first_date: '2026-03-19'
  read: F
  role: re_measures
  code_release: not checked
  later_version: EMNLP 2026 (from the search record; not read)
  extraction_scope: 'Sections 5 and 6 and Appendices A.3, A.6 and A.7: Tables 1 and 2 (step-1 behaviour and completion on 97 AgentDojo tasks), Table 3 (refusal of attacks and of benign documents on the curated subset) and
    Table 8 (Meta SecAlign with and without the input role); the abstract claim (Tables 4 to 7 and the depth analysis not coded)'
defences:
- id: struq_l3
  name: StruQ (re-implemented by the authors)
  origin: other_report
  base_checkpoint: Llama-3-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 17
  components:
  - <inst> and <obs> XML delimiters
  - fine-tuning with instruction tokens weighted 2.0
  anchor: StruQ wraps these formats with XML delimiters
- id: secalign_l3
  name: SecAlign (re-implemented by the authors)
  origin: other_report
  base_checkpoint: Llama-3-8B-Instruct
  axis1:
  - no_signal
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 17
  components:
  - 'DPO on 30K preference triples from agent tasks: a refusal to follow file instructions is preferred, completing the attack is not'
  anchor: Training data consists of 30K preference triples
- id: struq_mi
  name: StruQ (re-implemented by the authors)
  origin: other_report
  base_checkpoint: Mistral-7B-Instruct-v0.3
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 17
  components:
  - <inst> and <obs> XML delimiters
  - fine-tuning with instruction tokens weighted 2.0
  anchor: StruQ wraps these formats with XML delimiters
- id: secalign_mi
  name: SecAlign (re-implemented by the authors)
  origin: other_report
  base_checkpoint: Mistral-7B-Instruct-v0.3
  axis1:
  - no_signal
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 17
  components:
  - 'DPO on 30K preference triples from agent tasks: a refusal to follow file instructions is preferred, completing the attack is not'
  anchor: Training data consists of 30K preference triples
- id: msa_l31
  name: Meta SecAlign (official model)
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: measured
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  anchor: Meta SecAlign (Meta's official DPO-trained defense with
  layout_compat:
    user_turn: supported
    separate_role: supported
    system_context: untested
    agent_tool_result: supported
  pages:
    layout_compat: 19
evaluations:
- id: ev_ag_l3_none
  defence_id: none:Llama-3-8B-Instruct
  benchmark: AgentDojo, 97 tasks, attack-like trigger phrases added to 30% of tool observations
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task check; step-1 action classified as normal, refusal or invalid
  layout_same_as_defence: 'yes'
  measured_by: autonomytax_arxiv2026
  page: 5
  decoding: greedy decoding
  sample_size: '97'
  pages:
    benchmark: 4
  baseline_kind: off_the_shelf
- id: ev_ag_l3_struq
  defence_id: struq_l3
  benchmark: AgentDojo, 97 tasks, attack-like trigger phrases added to 30% of tool observations
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task check; step-1 action classified as normal, refusal or invalid
  layout_same_as_defence: 'yes'
  measured_by: autonomytax_arxiv2026
  page: 5
  decoding: greedy decoding
  sample_size: '97'
  pages:
    benchmark: 4
- id: ev_ag_l3_secalign
  defence_id: secalign_l3
  benchmark: AgentDojo, 97 tasks, attack-like trigger phrases added to 30% of tool observations
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task check; step-1 action classified as normal, refusal or invalid
  layout_same_as_defence: 'yes'
  measured_by: autonomytax_arxiv2026
  page: 5
  decoding: greedy decoding
  sample_size: '97'
  pages:
    benchmark: 4
- id: ev_ag_l31_msa
  defence_id: msa_l31
  benchmark: AgentDojo, 97 tasks, attack-like trigger phrases added to 30% of tool observations
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task check; step-1 action classified as normal, refusal or invalid
  layout_same_as_defence: 'yes'
  measured_by: autonomytax_arxiv2026
  page: 5
  decoding: greedy decoding
  sample_size: '97'
  pages:
    benchmark: 4
- id: ev_ag_mi_none
  defence_id: none:Mistral-7B-Instruct-v0.3
  benchmark: AgentDojo, 97 tasks, attack-like trigger phrases added to 30% of tool observations
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task check; step-1 action classified as normal, refusal or invalid
  layout_same_as_defence: 'yes'
  measured_by: autonomytax_arxiv2026
  page: 5
  decoding: greedy decoding
  sample_size: '97'
  pages:
    benchmark: 4
  baseline_kind: off_the_shelf
- id: ev_ag_mi_struq
  defence_id: struq_mi
  benchmark: AgentDojo, 97 tasks, attack-like trigger phrases added to 30% of tool observations
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task check; step-1 action classified as normal, refusal or invalid
  layout_same_as_defence: 'yes'
  measured_by: autonomytax_arxiv2026
  page: 5
  decoding: greedy decoding
  sample_size: '97'
  pages:
    benchmark: 4
- id: ev_ag_mi_secalign
  defence_id: secalign_mi
  benchmark: AgentDojo, 97 tasks, attack-like trigger phrases added to 30% of tool observations
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task check; step-1 action classified as normal, refusal or invalid
  layout_same_as_defence: 'yes'
  measured_by: autonomytax_arxiv2026
  page: 5
  decoding: greedy decoding
  sample_size: '97'
  pages:
    benchmark: 4
- id: ev_cs_l3_none_adv
  defence_id: none:Llama-3-8B-Instruct
  benchmark: 'curated challenging subset: 289 adversarial prompts (social engineering, obfuscation, instruction override) and 61 benign technical documents; adversarial part'
  attacker_type: static
  attacker_access: black_box
  judge: refusal of the attack (TPR)
  layout_same_as_defence: 'yes'
  measured_by: autonomytax_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: '289'
  pages:
    benchmark: 4
    decoding: 5
  baseline_kind: off_the_shelf
- id: ev_cs_l3_none_ben
  defence_id: none:Llama-3-8B-Instruct
  benchmark: 'curated challenging subset: 289 adversarial prompts (social engineering, obfuscation, instruction override) and 61 benign technical documents; benign part'
  attacker_type: none
  attacker_access: not_applicable
  judge: refusal of benign content (FPR)
  layout_same_as_defence: 'yes'
  measured_by: autonomytax_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: '61'
  pages:
    benchmark: 4
    decoding: 5
  baseline_kind: off_the_shelf
- id: ev_cs_l3_struq_adv
  defence_id: struq_l3
  benchmark: 'curated challenging subset: 289 adversarial prompts (social engineering, obfuscation, instruction override) and 61 benign technical documents; adversarial part'
  attacker_type: static
  attacker_access: black_box
  judge: refusal of the attack (TPR)
  layout_same_as_defence: 'yes'
  measured_by: autonomytax_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: '289'
  pages:
    benchmark: 4
    decoding: 5
- id: ev_cs_l3_struq_ben
  defence_id: struq_l3
  benchmark: 'curated challenging subset: 289 adversarial prompts (social engineering, obfuscation, instruction override) and 61 benign technical documents; benign part'
  attacker_type: none
  attacker_access: not_applicable
  judge: refusal of benign content (FPR)
  layout_same_as_defence: 'yes'
  measured_by: autonomytax_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: '61'
  pages:
    benchmark: 4
    decoding: 5
- id: ev_cs_l3_secalign_adv
  defence_id: secalign_l3
  benchmark: 'curated challenging subset: 289 adversarial prompts (social engineering, obfuscation, instruction override) and 61 benign technical documents; adversarial part'
  attacker_type: static
  attacker_access: black_box
  judge: refusal of the attack (TPR)
  layout_same_as_defence: 'yes'
  measured_by: autonomytax_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: '289'
  pages:
    benchmark: 4
    decoding: 5
- id: ev_cs_l3_secalign_ben
  defence_id: secalign_l3
  benchmark: 'curated challenging subset: 289 adversarial prompts (social engineering, obfuscation, instruction override) and 61 benign technical documents; benign part'
  attacker_type: none
  attacker_access: not_applicable
  judge: refusal of benign content (FPR)
  layout_same_as_defence: 'yes'
  measured_by: autonomytax_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: '61'
  pages:
    benchmark: 4
    decoding: 5
- id: ev_cs_l31_none_adv
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: 'curated challenging subset: 289 adversarial prompts (social engineering, obfuscation, instruction override) and 61 benign technical documents; adversarial part'
  attacker_type: static
  attacker_access: black_box
  judge: refusal of the attack (TPR)
  layout_same_as_defence: 'yes'
  measured_by: autonomytax_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: '289'
  pages:
    benchmark: 4
    decoding: 5
  baseline_kind: off_the_shelf
- id: ev_cs_l31_none_ben
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: 'curated challenging subset: 289 adversarial prompts (social engineering, obfuscation, instruction override) and 61 benign technical documents; benign part'
  attacker_type: none
  attacker_access: not_applicable
  judge: refusal of benign content (FPR)
  layout_same_as_defence: 'yes'
  measured_by: autonomytax_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: '61'
  pages:
    benchmark: 4
    decoding: 5
  baseline_kind: off_the_shelf
- id: ev_cs_l31_msa_adv
  defence_id: msa_l31
  benchmark: 'curated challenging subset: 289 adversarial prompts (social engineering, obfuscation, instruction override) and 61 benign technical documents; adversarial part'
  attacker_type: static
  attacker_access: black_box
  judge: refusal of the attack (TPR)
  layout_same_as_defence: 'yes'
  measured_by: autonomytax_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: '289'
  pages:
    benchmark: 4
    decoding: 5
- id: ev_cs_l31_msa_ben
  defence_id: msa_l31
  benchmark: 'curated challenging subset: 289 adversarial prompts (social engineering, obfuscation, instruction override) and 61 benign technical documents; benign part'
  attacker_type: none
  attacker_access: not_applicable
  judge: refusal of benign content (FPR)
  layout_same_as_defence: 'yes'
  measured_by: autonomytax_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: '61'
  pages:
    benchmark: 4
    decoding: 5
- id: ev_cs_mi_none_adv
  defence_id: none:Mistral-7B-Instruct-v0.3
  benchmark: 'curated challenging subset: 289 adversarial prompts (social engineering, obfuscation, instruction override) and 61 benign technical documents; adversarial part'
  attacker_type: static
  attacker_access: black_box
  judge: refusal of the attack (TPR)
  layout_same_as_defence: 'yes'
  measured_by: autonomytax_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: '289'
  pages:
    benchmark: 4
    decoding: 5
  baseline_kind: off_the_shelf
- id: ev_cs_mi_none_ben
  defence_id: none:Mistral-7B-Instruct-v0.3
  benchmark: 'curated challenging subset: 289 adversarial prompts (social engineering, obfuscation, instruction override) and 61 benign technical documents; benign part'
  attacker_type: none
  attacker_access: not_applicable
  judge: refusal of benign content (FPR)
  layout_same_as_defence: 'yes'
  measured_by: autonomytax_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: '61'
  pages:
    benchmark: 4
    decoding: 5
  baseline_kind: off_the_shelf
- id: ev_cs_mi_struq_adv
  defence_id: struq_mi
  benchmark: 'curated challenging subset: 289 adversarial prompts (social engineering, obfuscation, instruction override) and 61 benign technical documents; adversarial part'
  attacker_type: static
  attacker_access: black_box
  judge: refusal of the attack (TPR)
  layout_same_as_defence: 'yes'
  measured_by: autonomytax_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: '289'
  pages:
    benchmark: 4
    decoding: 5
- id: ev_cs_mi_struq_ben
  defence_id: struq_mi
  benchmark: 'curated challenging subset: 289 adversarial prompts (social engineering, obfuscation, instruction override) and 61 benign technical documents; benign part'
  attacker_type: none
  attacker_access: not_applicable
  judge: refusal of benign content (FPR)
  layout_same_as_defence: 'yes'
  measured_by: autonomytax_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: '61'
  pages:
    benchmark: 4
    decoding: 5
- id: ev_cs_mi_secalign_adv
  defence_id: secalign_mi
  benchmark: 'curated challenging subset: 289 adversarial prompts (social engineering, obfuscation, instruction override) and 61 benign technical documents; adversarial part'
  attacker_type: static
  attacker_access: black_box
  judge: refusal of the attack (TPR)
  layout_same_as_defence: 'yes'
  measured_by: autonomytax_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: '289'
  pages:
    benchmark: 4
    decoding: 5
- id: ev_cs_mi_secalign_ben
  defence_id: secalign_mi
  benchmark: 'curated challenging subset: 289 adversarial prompts (social engineering, obfuscation, instruction override) and 61 benign technical documents; benign part'
  attacker_type: none
  attacker_access: not_applicable
  judge: refusal of benign content (FPR)
  layout_same_as_defence: 'yes'
  measured_by: autonomytax_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: '61'
  pages:
    benchmark: 4
    decoding: 5
- id: ev_t8_user
  defence_id: msa_l31
  benchmark: 1,000 single-turn prompts (500 adversarial, 500 benign), standard user role for all content
  attacker_type: static
  attacker_access: black_box
  judge: refusal of the attack; refusal of benign content
  layout_same_as_defence: 'no'
  measured_by: autonomytax_arxiv2026
  page: 19
  decoding: greedy decoding
  sample_size: 1,000
  pages:
    benchmark: 19
    sample_size: 18
    decoding: 5
- id: ev_t8_input
  defence_id: msa_l31
  benchmark: 1,000 single-turn prompts (500 adversarial, 500 benign), adversarial content in the input role
  attacker_type: static
  attacker_access: black_box
  judge: refusal of the attack; refusal of benign content
  layout_same_as_defence: 'yes'
  measured_by: autonomytax_arxiv2026
  page: 19
  decoding: greedy decoding
  sample_size: 1,000
  pages:
    benchmark: 19
    sample_size: 18
    decoding: 5
results:
- id: r_ag_l3_none_normal
  evaluation_id: ev_ag_l3_none
  quantity: R2clean
  coverage: proxy
  metric_as_printed: Normal (valid action at step 1, before any tool output)
  value: '96.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_ag_l3_none_cr
  evaluation_id: ev_ag_l3_none
  quantity: other
  coverage: direct
  metric_as_printed: CR (completion rate)
  value: '86.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_ag_l3_struq_normal
  evaluation_id: ev_ag_l3_struq
  quantity: R2clean
  coverage: proxy
  metric_as_printed: Normal (valid action at step 1, before any tool output)
  value: '22.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_ag_l3_struq_cr
  evaluation_id: ev_ag_l3_struq
  quantity: other
  coverage: direct
  metric_as_printed: CR (completion rate)
  value: '80.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_ag_l3_secalign_normal
  evaluation_id: ev_ag_l3_secalign
  quantity: R2clean
  coverage: proxy
  metric_as_printed: Normal (valid action at step 1, before any tool output)
  value: '52.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_ag_l3_secalign_cr
  evaluation_id: ev_ag_l3_secalign
  quantity: other
  coverage: direct
  metric_as_printed: CR (completion rate)
  value: '63.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_ag_l31_msa_normal
  evaluation_id: ev_ag_l31_msa
  quantity: R2clean
  coverage: proxy
  metric_as_printed: Normal (valid action at step 1, before any tool output)
  value: '94.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_ag_l31_msa_cr
  evaluation_id: ev_ag_l31_msa
  quantity: other
  coverage: direct
  metric_as_printed: CR (completion rate)
  value: '85.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_ag_mi_none_normal
  evaluation_id: ev_ag_mi_none
  quantity: R2clean
  coverage: proxy
  metric_as_printed: Normal (valid action at step 1, before any tool output)
  value: '97.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_ag_mi_none_cr
  evaluation_id: ev_ag_mi_none
  quantity: other
  coverage: direct
  metric_as_printed: CR (completion rate)
  value: '50.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_ag_mi_struq_normal
  evaluation_id: ev_ag_mi_struq
  quantity: R2clean
  coverage: proxy
  metric_as_printed: Normal (valid action at step 1, before any tool output)
  value: '23.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_ag_mi_struq_cr
  evaluation_id: ev_ag_mi_struq
  quantity: other
  coverage: direct
  metric_as_printed: CR (completion rate)
  value: '14.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_ag_mi_secalign_normal
  evaluation_id: ev_ag_mi_secalign
  quantity: R2clean
  coverage: proxy
  metric_as_printed: Normal (valid action at step 1, before any tool output)
  value: '53.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_ag_mi_secalign_cr
  evaluation_id: ev_ag_mi_secalign
  quantity: other
  coverage: direct
  metric_as_printed: CR (completion rate)
  value: '1.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_cs_l3_none_tpr
  evaluation_id: ev_cs_l3_none_adv
  quantity: R1
  coverage: proxy
  metric_as_printed: TPR (correctly refusing attacks)
  value: '82.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_cs_l3_none_fpr
  evaluation_id: ev_cs_l3_none_ben
  quantity: other
  coverage: direct
  metric_as_printed: FPR (incorrectly refusing benign content)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_cs_l3_struq_tpr
  evaluation_id: ev_cs_l3_struq_adv
  quantity: R1
  coverage: proxy
  metric_as_printed: TPR (correctly refusing attacks)
  value: '37.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_cs_l3_struq_fpr
  evaluation_id: ev_cs_l3_struq_ben
  quantity: other
  coverage: direct
  metric_as_printed: FPR (incorrectly refusing benign content)
  value: '70.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_cs_l3_secalign_tpr
  evaluation_id: ev_cs_l3_secalign_adv
  quantity: R1
  coverage: proxy
  metric_as_printed: TPR (correctly refusing attacks)
  value: '26.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_cs_l3_secalign_fpr
  evaluation_id: ev_cs_l3_secalign_ben
  quantity: other
  coverage: direct
  metric_as_printed: FPR (incorrectly refusing benign content)
  value: '24.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_cs_l31_none_tpr
  evaluation_id: ev_cs_l31_none_adv
  quantity: R1
  coverage: proxy
  metric_as_printed: TPR (correctly refusing attacks)
  value: '79.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_cs_l31_none_fpr
  evaluation_id: ev_cs_l31_none_ben
  quantity: other
  coverage: direct
  metric_as_printed: FPR (incorrectly refusing benign content)
  value: '0.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_cs_l31_msa_tpr
  evaluation_id: ev_cs_l31_msa_adv
  quantity: R1
  coverage: proxy
  metric_as_printed: TPR (correctly refusing attacks)
  value: '68.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_cs_l31_msa_fpr
  evaluation_id: ev_cs_l31_msa_ben
  quantity: other
  coverage: direct
  metric_as_printed: FPR (incorrectly refusing benign content)
  value: '0.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_cs_mi_none_tpr
  evaluation_id: ev_cs_mi_none_adv
  quantity: R1
  coverage: proxy
  metric_as_printed: TPR (correctly refusing attacks)
  value: '12.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_cs_mi_none_fpr
  evaluation_id: ev_cs_mi_none_ben
  quantity: other
  coverage: direct
  metric_as_printed: FPR (incorrectly refusing benign content)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_cs_mi_struq_tpr
  evaluation_id: ev_cs_mi_struq_adv
  quantity: R1
  coverage: proxy
  metric_as_printed: TPR (correctly refusing attacks)
  value: '18.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_cs_mi_struq_fpr
  evaluation_id: ev_cs_mi_struq_ben
  quantity: other
  coverage: direct
  metric_as_printed: FPR (incorrectly refusing benign content)
  value: '54.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_cs_mi_secalign_tpr
  evaluation_id: ev_cs_mi_secalign_adv
  quantity: R1
  coverage: proxy
  metric_as_printed: TPR (correctly refusing attacks)
  value: '6.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_cs_mi_secalign_fpr
  evaluation_id: ev_cs_mi_secalign_ben
  quantity: other
  coverage: direct
  metric_as_printed: FPR (incorrectly refusing benign content)
  value: '36.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 3
- id: r_t8_user_asr
  evaluation_id: ev_t8_user
  quantity: R1
  coverage: proxy
  metric_as_printed: ASR
  value: '32.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 8
- id: r_t8_user_fpr
  evaluation_id: ev_t8_user
  quantity: other
  coverage: direct
  metric_as_printed: FPR
  value: '0.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 8
- id: r_t8_input_asr
  evaluation_id: ev_t8_input
  quantity: R1
  coverage: proxy
  metric_as_printed: ASR
  value: '20.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 8
- id: r_t8_input_fpr
  evaluation_id: ev_t8_input
  quantity: other
  coverage: direct
  metric_as_printed: FPR
  value: '3.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 8
comparisons:
- id: c_ag_l3_struq
  left: ev_ag_l3_struq
  right: ev_ag_l3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_ag_l3_secalign
  left: ev_ag_l3_secalign
  right: ev_ag_l3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_ag_mi_struq
  left: ev_ag_mi_struq
  right: ev_ag_mi_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_ag_mi_secalign
  left: ev_ag_mi_secalign
  right: ev_ag_mi_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_ag_msa
  left: ev_ag_l31_msa
  right: ev_ag_l3_none
  purpose: W
  c1_base: unmatched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: Meta SecAlign on Llama-3.1-8B against undefended Llama-3-8B, as Table 2 compares them
  page: 5
- id: c_cs_l3_struq_adv
  left: ev_cs_l3_struq_adv
  right: ev_cs_l3_none_adv
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_cs_l3_struq_ben
  left: ev_cs_l3_struq_ben
  right: ev_cs_l3_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_cs_l3_secalign_adv
  left: ev_cs_l3_secalign_adv
  right: ev_cs_l3_none_adv
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_cs_l3_secalign_ben
  left: ev_cs_l3_secalign_ben
  right: ev_cs_l3_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_cs_mi_struq_adv
  left: ev_cs_mi_struq_adv
  right: ev_cs_mi_none_adv
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_cs_mi_struq_ben
  left: ev_cs_mi_struq_ben
  right: ev_cs_mi_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_cs_mi_secalign_adv
  left: ev_cs_mi_secalign_adv
  right: ev_cs_mi_none_adv
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_cs_mi_secalign_ben
  left: ev_cs_mi_secalign_ben
  right: ev_cs_mi_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_cs_msa_adv
  left: ev_cs_l31_msa_adv
  right: ev_cs_l31_none_adv
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_cs_msa_ben
  left: ev_cs_l31_msa_ben
  right: ev_cs_l31_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: unreported
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_t8_role
  left: ev_t8_input
  right: ev_t8_user
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: adversarial content in the input role against the user role
  page: 19
claims:
- id: k1
  text: defence training destroys agent competence while failing to stop sophisticated attacks
  location: abstract
  anchor: defense training designed to improve safety systematically destroys agent competence
  jobs:
  - R1
  - R2
  supported_by:
  - c_ag_l3_struq
  - c_ag_l3_secalign
  - c_ag_mi_struq
  - c_ag_mi_secalign
  - c_cs_l3_struq_adv
  - c_cs_l3_secalign_adv
  page: 1
  contradicted_by:
  - r_ag_l31_msa_normal
  - r_ag_l31_msa_cr
coverage:
  R1: proxy
  R2clean: proxy
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 3 and 8 (refusal of a mixed set of direct requests and embedded instructions)
    R2clean: Table 1 (valid first action before any tool output); completion with trigger phrases is coded other
schema_gaps:
- where: evaluations ev_ag_*
  note: completion rate is measured with attack-like trigger phrases, not attacks, added to 30% of tool observations, so it is coded other; the step-1 rate is measured before any tool output and is coded as a clean-utility
    proxy
- where: evaluations ev_cs_*, ev_t8_*
  note: the adversarial prompts are mostly direct requests (social engineering, obfuscation) with some instructions embedded in task text, and success is counted as not refusing, so R1 is a proxy (pp. 4, 16)
- where: comparisons c_cs_msa_*
  note: Table 3 lists Meta SecAlign and Base-Llama3.1-8B in the n = 350 subset table, but Appendix A.7 says the Meta SecAlign runs used a 1,000-prompt set, Table 4 uses another attack taxonomy for it, and Table 8's input-role
    TPR (79.6%) equals Base-Llama3.1-8B's Table 3 TPR; the denominator is marked unreported (pp. 6, 18, 19)
- where: defences struq_*, secalign_*
  note: 'both are the authors'' re-implementations: StruQ with <inst>/<obs> tags and weighted instruction tokens; SecAlign trained to prefer refusals on 30K agent triples, not the original SecAlign objective (p. 17)'

`piarena_arxiv2026` (2604.08499)

In [ ]:
%%writefile reports/records/piarena_arxiv2026.yaml
report:
  id: piarena_arxiv2026
  slug: '2604.08499'
  title: 'PIArena: A Platform for Prompt Injection Evaluation'
  first_author: Runpeng Geng
  venue: arXiv preprint
  peer_reviewed: 'no'
  doi: 10.18653/v1/2026.acl-long.1533
  arxiv: '2604.08499'
  version_read: arXiv v1
  first_date: '2026-04-09'
  read: F
  role: benchmark
  code_release: https://github.com/sleeepeer/PIArena (linked from the abstract)
  later_version: ACL 2026 (from the search record; not read)
  extraction_scope: 'Sections 4 and 5 and Appendix B: the Average rows of Table 2 and Table 7 (GCG on MultiNews) for No Defense and SecAlign++; the abstract claim (PISanitizer, DataFilter and PromptArmor, which rewrite the
    context, and DataSentinel, PromptGuard, AttentionTracker and PIGuard, which are detectors, are left out; per-dataset rows, Tables 3 to 6 and 9 to 11 not coded)'
defences:
- id: secalignpp
  name: SecAlign++
  origin: other_report
  base_checkpoint: not stated (the checkpoint used is not named; the default backend is Qwen3-4B-Instruct)
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  anchor: All defenses use their default settings and open-source implementations
evaluations:
- id: ev_noatt_none
  defence_id: none:Qwen3-4B-Instruct
  benchmark: PIArena, 13 datasets (QA, RAG, long-context, code), mean over datasets, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judge of the injected task; task-specific utility (LLM judge or LongBench metrics)
  layout_same_as_defence: 'yes'
  measured_by: piarena_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_noatt_sapp
  defence_id: secalignpp
  benchmark: PIArena, 13 datasets (QA, RAG, long-context, code), mean over datasets, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judge of the injected task; task-specific utility (LLM judge or LongBench metrics)
  layout_same_as_defence: 'yes'
  measured_by: piarena_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_comb_none
  defence_id: none:Qwen3-4B-Instruct
  benchmark: PIArena, 13 datasets (QA, RAG, long-context, code), mean over datasets, Combined attack
  attacker_type: static
  attacker_access: black_box
  judge: LLM judge of the injected task; task-specific utility (LLM judge or LongBench metrics)
  layout_same_as_defence: 'yes'
  measured_by: piarena_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_comb_sapp
  defence_id: secalignpp
  benchmark: PIArena, 13 datasets (QA, RAG, long-context, code), mean over datasets, Combined attack
  attacker_type: static
  attacker_access: black_box
  judge: LLM judge of the injected task; task-specific utility (LLM judge or LongBench metrics)
  layout_same_as_defence: 'yes'
  measured_by: piarena_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_direct_none
  defence_id: none:Qwen3-4B-Instruct
  benchmark: PIArena, 13 datasets (QA, RAG, long-context, code), mean over datasets, Direct attack (the injected task itself)
  attacker_type: static
  attacker_access: black_box
  judge: LLM judge of the injected task; task-specific utility (LLM judge or LongBench metrics)
  layout_same_as_defence: 'yes'
  measured_by: piarena_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_direct_sapp
  defence_id: secalignpp
  benchmark: PIArena, 13 datasets (QA, RAG, long-context, code), mean over datasets, Direct attack (the injected task itself)
  attacker_type: static
  attacker_access: black_box
  judge: LLM judge of the injected task; task-specific utility (LLM judge or LongBench metrics)
  layout_same_as_defence: 'yes'
  measured_by: piarena_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_strat_none
  defence_id: none:Qwen3-4B-Instruct
  benchmark: PIArena, 13 datasets (QA, RAG, long-context, code), mean over datasets, Strategy attack (Qwen3-4B attacker refining on defence feedback)
  attacker_type: adaptive
  attacker_access: black_box
  judge: LLM judge of the injected task; task-specific utility (LLM judge or LongBench metrics)
  layout_same_as_defence: 'yes'
  measured_by: piarena_arxiv2026
  page: 7
  attacker_budget: iterative refinement on defence feedback
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_strat_sapp
  defence_id: secalignpp
  benchmark: PIArena, 13 datasets (QA, RAG, long-context, code), mean over datasets, Strategy attack (Qwen3-4B attacker refining on defence feedback)
  attacker_type: adaptive
  attacker_access: black_box
  judge: LLM judge of the injected task; task-specific utility (LLM judge or LongBench metrics)
  layout_same_as_defence: 'yes'
  measured_by: piarena_arxiv2026
  page: 7
  attacker_budget: iterative refinement on defence feedback
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_noatt7_none
  defence_id: none:Qwen3-4B-Instruct
  benchmark: MultiNews, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judge of the injected task; task-specific utility (LLM judge or LongBench metrics)
  layout_same_as_defence: 'yes'
  measured_by: piarena_arxiv2026
  page: 16
  decoding: not reported
  pages:
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_noatt7_sapp
  defence_id: secalignpp
  benchmark: MultiNews, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judge of the injected task; task-specific utility (LLM judge or LongBench metrics)
  layout_same_as_defence: 'yes'
  measured_by: piarena_arxiv2026
  page: 16
  decoding: not reported
  pages:
    judge: 6
- id: ev_gcg_none
  defence_id: none:Qwen3-4B-Instruct
  benchmark: MultiNews, GCG optimisation attack
  attacker_type: optimisation
  attacker_access: white_box
  judge: LLM judge of the injected task; task-specific utility (LLM judge or LongBench metrics)
  layout_same_as_defence: 'yes'
  measured_by: piarena_arxiv2026
  page: 16
  decoding: not reported
  pages:
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_gcg_sapp
  defence_id: secalignpp
  benchmark: MultiNews, GCG optimisation attack
  attacker_type: optimisation
  attacker_access: white_box
  judge: LLM judge of the injected task; task-specific utility (LLM judge or LongBench metrics)
  layout_same_as_defence: 'yes'
  measured_by: piarena_arxiv2026
  page: 16
  decoding: not reported
  pages:
    judge: 6
results:
- id: r_noatt_none_u
  evaluation_id: ev_noatt_none
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility (Average row)
  value: '0.74'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_noatt_none_asr
  evaluation_id: ev_noatt_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Average row)
  value: '0.01'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_noatt_sapp_u
  evaluation_id: ev_noatt_sapp
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility (Average row)
  value: '0.58'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_noatt_sapp_asr
  evaluation_id: ev_noatt_sapp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Average row)
  value: '0.01'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_comb_none_u
  evaluation_id: ev_comb_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (Average row)
  value: '0.50'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_comb_none_asr
  evaluation_id: ev_comb_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Average row)
  value: '0.72'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_comb_sapp_u
  evaluation_id: ev_comb_sapp
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (Average row)
  value: '0.58'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_comb_sapp_asr
  evaluation_id: ev_comb_sapp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Average row)
  value: '0.07'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_direct_none_u
  evaluation_id: ev_direct_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (Average row)
  value: '0.57'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_direct_none_asr
  evaluation_id: ev_direct_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Average row)
  value: '0.56'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_direct_sapp_u
  evaluation_id: ev_direct_sapp
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (Average row)
  value: '0.59'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_direct_sapp_asr
  evaluation_id: ev_direct_sapp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Average row)
  value: '0.04'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_strat_none_u
  evaluation_id: ev_strat_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (Average row)
  value: '0.19'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_strat_none_asr
  evaluation_id: ev_strat_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Average row)
  value: '0.99'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_strat_sapp_u
  evaluation_id: ev_strat_sapp
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (Average row)
  value: '0.45'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_strat_sapp_asr
  evaluation_id: ev_strat_sapp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Average row)
  value: '0.21'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_noatt7_none_u
  evaluation_id: ev_noatt7_none
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility
  value: '0.19'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 7
- id: r_noatt7_none_asr
  evaluation_id: ev_noatt7_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.0'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 7
- id: r_noatt7_sapp_u
  evaluation_id: ev_noatt7_sapp
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility
  value: '0.20'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 7
- id: r_noatt7_sapp_asr
  evaluation_id: ev_noatt7_sapp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.02'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 7
- id: r_gcg_none_u
  evaluation_id: ev_gcg_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility
  value: '0.11'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 7
- id: r_gcg_none_asr
  evaluation_id: ev_gcg_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.63'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 7
- id: r_gcg_sapp_u
  evaluation_id: ev_gcg_sapp
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility
  value: '0.20'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 7
- id: r_gcg_sapp_asr
  evaluation_id: ev_gcg_sapp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.86'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 7
comparisons:
- id: c_noatt
  left: ev_noatt_sapp
  right: ev_noatt_none
  purpose: W
  c1_base: unreported
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_comb
  left: ev_comb_sapp
  right: ev_comb_none
  purpose: W
  c1_base: unreported
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_direct
  left: ev_direct_sapp
  right: ev_direct_none
  purpose: W
  c1_base: unreported
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_strat
  left: ev_strat_sapp
  right: ev_strat_none
  purpose: W
  c1_base: unreported
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_noatt7
  left: ev_noatt7_sapp
  right: ev_noatt7_none
  purpose: W
  c1_base: unreported
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_gcg
  left: ev_gcg_sapp
  right: ev_gcg_none
  purpose: W
  c1_base: unreported
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
claims:
- id: k1
  text: state-of-the-art defences generalise poorly across tasks and are vulnerable to adaptive attacks
  location: abstract
  anchor: limited generalizability across tasks, vulnerability to adaptive attacks
  jobs:
  - R1
  - R2
  supported_by:
  - c_strat
  - c_gcg
  - c_noatt
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 2 (Average rows) and Table 7
    R2clean: Table 2 and Table 7, no attack
    R2attack: Table 2 and Table 7, under attack
schema_gaps:
- where: defence secalignpp
  note: the paper does not name the SecAlign++ checkpoint or say whether it was trained on the default Qwen3-4B-Instruct backend, so the base condition of its comparisons is unreported

`localalign_ccs2026` (2605.01462)

In [ ]:
%%writefile reports/records/localalign_ccs2026.yaml
report:
  id: localalign_ccs2026
  slug: '2605.01462'
  title: 'LocalAlign: Enabling Generalizable Prompt Injection Defense via Generation of Near-Target Adversarial Examples for Alignment Training'
  first_author: Yuyang Gong
  venue: CCS 2026
  peer_reviewed: 'yes'
  doi: 10.1145/3830454.3846553
  arxiv: '2605.01462'
  version_read: arXiv v2 (carries the CCS 2026 header)
  first_date: '2026-05-02'
  read: F
  role: proposes
  code_release: GitHub repository mentioned in Appendix B; not checked
  extraction_scope: 'Sections 4 and 5 and Appendix B: Table 1 (out-of-distribution ASR, both models), Table 2 (AlpacaFarm ASR) and Table 3 (utility); the abstract claims (the ablations of Tables 4 to 6, Figure 3 and the PISmith
    averages in Appendix B not coded)'
defences:
- id: la_l
  name: LocalAlign
  origin: this_report
  base_checkpoint: Llama3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 8
  components:
  - warm-up with Meta-SecAlign-style preference data from Cleaned Alpaca
  - near-target injected commands written by the defended model itself with one prompt (same task family, same data, a different answer)
  - 'margin-aware DPO: per-batch DPO coefficient raised for samples whose wrong answer is close to the correct one'
  - LoRA rank 64
  anchor: we fine-tune the model on the generated adversarial examples
  pages:
    components: 14
- id: la_q
  name: LocalAlign
  origin: this_report
  base_checkpoint: Qwen3-4B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 8
  components:
  - warm-up with Meta-SecAlign-style preference data from Cleaned Alpaca
  - near-target injected commands written by the defended model itself with one prompt (same task family, same data, a different answer)
  - 'margin-aware DPO: per-batch DPO coefficient raised for samples whose wrong answer is close to the correct one'
  - LoRA rank 64
  anchor: we fine-tune the model on the generated adversarial examples
  pages:
    components: 14
- id: msa_l
  name: Meta-SecAlign (released Llama3.1 checkpoint)
  origin: other_report
  base_checkpoint: Llama3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 14
  anchor: we use the authors’ released Llama3.1 check
- id: msa_q
  name: Meta-SecAlign (reproduced by the authors with the released recipe)
  origin: other_report
  base_checkpoint: Qwen3-4B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 14
  anchor: reproduce the Qwen3-4B baseline using their training
- id: struq_l
  name: StruQ (reproduced with the authors' released code)
  origin: other_report
  base_checkpoint: Llama3.1-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 14
  anchor: We reproduce StruQ using the authors’ released code
evaluations:
- id: ev_af_none_of
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: AlpacaFarm (i.i.d.), 'Print exactly Hacked!' at the end of the data; maximum over five optimization-free attacks (direct, ignore, fake completion, escape, combined)
  attacker_type: static
  attacker_access: black_box
  judge: response begins with 'Hacked' (string match)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
    judge: 14
    attacker_budget: 14
  baseline_kind: off_the_shelf
- id: ev_af_none_gcg
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: AlpacaFarm (i.i.d.), 'Print exactly Hacked!' at the end of the data; GCG suffix, 20 tokens, 500 steps
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with 'Hacked' (string match)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: 20-token suffix, 500 steps, batch 512
  decoding: not reported
  pages:
    benchmark: 14
    judge: 14
    attacker_budget: 14
  baseline_kind: off_the_shelf
- id: ev_af_none_agcg
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: AlpacaFarm (i.i.d.), 'Print exactly Hacked!' at the end of the data; adaptive GCG with fake delimiters searched in embedding space
  attacker_type: adaptive
  attacker_access: white_box
  judge: response begins with 'Hacked' (string match)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones plus GCG
  decoding: not reported
  pages:
    benchmark: 14
    judge: 14
    attacker_budget: 14
  baseline_kind: off_the_shelf
- id: ev_af_struq_of
  defence_id: struq_l
  benchmark: AlpacaFarm (i.i.d.), 'Print exactly Hacked!' at the end of the data; maximum over five optimization-free attacks (direct, ignore, fake completion, escape, combined)
  attacker_type: static
  attacker_access: black_box
  judge: response begins with 'Hacked' (string match)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
    judge: 14
    attacker_budget: 14
- id: ev_af_struq_gcg
  defence_id: struq_l
  benchmark: AlpacaFarm (i.i.d.), 'Print exactly Hacked!' at the end of the data; GCG suffix, 20 tokens, 500 steps
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with 'Hacked' (string match)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: 20-token suffix, 500 steps, batch 512
  decoding: not reported
  pages:
    benchmark: 14
    judge: 14
    attacker_budget: 14
- id: ev_af_struq_agcg
  defence_id: struq_l
  benchmark: AlpacaFarm (i.i.d.), 'Print exactly Hacked!' at the end of the data; adaptive GCG with fake delimiters searched in embedding space
  attacker_type: adaptive
  attacker_access: white_box
  judge: response begins with 'Hacked' (string match)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones plus GCG
  decoding: not reported
  pages:
    benchmark: 14
    judge: 14
    attacker_budget: 14
- id: ev_af_msa_of
  defence_id: msa_l
  benchmark: AlpacaFarm (i.i.d.), 'Print exactly Hacked!' at the end of the data; maximum over five optimization-free attacks (direct, ignore, fake completion, escape, combined)
  attacker_type: static
  attacker_access: black_box
  judge: response begins with 'Hacked' (string match)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
    judge: 14
    attacker_budget: 14
- id: ev_af_msa_gcg
  defence_id: msa_l
  benchmark: AlpacaFarm (i.i.d.), 'Print exactly Hacked!' at the end of the data; GCG suffix, 20 tokens, 500 steps
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with 'Hacked' (string match)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: 20-token suffix, 500 steps, batch 512
  decoding: not reported
  pages:
    benchmark: 14
    judge: 14
    attacker_budget: 14
- id: ev_af_msa_agcg
  defence_id: msa_l
  benchmark: AlpacaFarm (i.i.d.), 'Print exactly Hacked!' at the end of the data; adaptive GCG with fake delimiters searched in embedding space
  attacker_type: adaptive
  attacker_access: white_box
  judge: response begins with 'Hacked' (string match)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones plus GCG
  decoding: not reported
  pages:
    benchmark: 14
    judge: 14
    attacker_budget: 14
- id: ev_af_la_of
  defence_id: la_l
  benchmark: AlpacaFarm (i.i.d.), 'Print exactly Hacked!' at the end of the data; maximum over five optimization-free attacks (direct, ignore, fake completion, escape, combined)
  attacker_type: static
  attacker_access: black_box
  judge: response begins with 'Hacked' (string match)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
    judge: 14
    attacker_budget: 14
- id: ev_af_la_gcg
  defence_id: la_l
  benchmark: AlpacaFarm (i.i.d.), 'Print exactly Hacked!' at the end of the data; GCG suffix, 20 tokens, 500 steps
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with 'Hacked' (string match)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: 20-token suffix, 500 steps, batch 512
  decoding: not reported
  pages:
    benchmark: 14
    judge: 14
    attacker_budget: 14
- id: ev_af_la_agcg
  defence_id: la_l
  benchmark: AlpacaFarm (i.i.d.), 'Print exactly Hacked!' at the end of the data; adaptive GCG with fake delimiters searched in embedding space
  attacker_type: adaptive
  attacker_access: white_box
  judge: response begins with 'Hacked' (string match)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones plus GCG
  decoding: not reported
  pages:
    benchmark: 14
    judge: 14
    attacker_budget: 14
- id: ev_ood_l_none_of_hotpotqa
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: HotpotQA (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_ood_l_none_of_qasper
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: Qasper (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_ood_l_none_of_injecagent
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: InjecAgent (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_ood_l_none_of_sep
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: SEP (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_ood_l_none_of_mmlu
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: MMLU (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_ood_l_none_of_openprompt
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: Open-Prompt (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_ood_l_msa_of_hotpotqa
  defence_id: msa_l
  benchmark: HotpotQA (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_l_msa_of_qasper
  defence_id: msa_l
  benchmark: Qasper (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_l_msa_of_injecagent
  defence_id: msa_l
  benchmark: InjecAgent (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_l_msa_of_sep
  defence_id: msa_l
  benchmark: SEP (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_l_msa_of_mmlu
  defence_id: msa_l
  benchmark: MMLU (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_l_msa_of_openprompt
  defence_id: msa_l
  benchmark: Open-Prompt (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_l_la_of_hotpotqa
  defence_id: la_l
  benchmark: HotpotQA (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_l_la_of_qasper
  defence_id: la_l
  benchmark: Qasper (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_l_la_of_injecagent
  defence_id: la_l
  benchmark: InjecAgent (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_l_la_of_sep
  defence_id: la_l
  benchmark: SEP (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_l_la_of_mmlu
  defence_id: la_l
  benchmark: MMLU (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_l_la_of_openprompt
  defence_id: la_l
  benchmark: Open-Prompt (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_l_none_aof_hotpotqa
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: HotpotQA (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_ood_l_none_aof_qasper
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: Qasper (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_ood_l_none_aof_injecagent
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: InjecAgent (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_ood_l_none_aof_sep
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: SEP (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_ood_l_none_aof_mmlu
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: MMLU (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_ood_l_none_aof_openprompt
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: Open-Prompt (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_ood_l_msa_aof_hotpotqa
  defence_id: msa_l
  benchmark: HotpotQA (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_l_msa_aof_qasper
  defence_id: msa_l
  benchmark: Qasper (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_l_msa_aof_injecagent
  defence_id: msa_l
  benchmark: InjecAgent (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_l_msa_aof_sep
  defence_id: msa_l
  benchmark: SEP (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_l_msa_aof_mmlu
  defence_id: msa_l
  benchmark: MMLU (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_l_msa_aof_openprompt
  defence_id: msa_l
  benchmark: Open-Prompt (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_l_la_aof_hotpotqa
  defence_id: la_l
  benchmark: HotpotQA (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_l_la_aof_qasper
  defence_id: la_l
  benchmark: Qasper (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_l_la_aof_injecagent
  defence_id: la_l
  benchmark: InjecAgent (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_l_la_aof_sep
  defence_id: la_l
  benchmark: SEP (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_l_la_aof_mmlu
  defence_id: la_l
  benchmark: MMLU (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_l_la_aof_openprompt
  defence_id: la_l
  benchmark: Open-Prompt (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_q_none_of_hotpotqa
  defence_id: none:Qwen3-4B-Instruct
  benchmark: HotpotQA (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_ood_q_none_of_qasper
  defence_id: none:Qwen3-4B-Instruct
  benchmark: Qasper (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_ood_q_none_of_injecagent
  defence_id: none:Qwen3-4B-Instruct
  benchmark: InjecAgent (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_ood_q_none_of_sep
  defence_id: none:Qwen3-4B-Instruct
  benchmark: SEP (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_ood_q_none_of_mmlu
  defence_id: none:Qwen3-4B-Instruct
  benchmark: MMLU (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_ood_q_none_of_openprompt
  defence_id: none:Qwen3-4B-Instruct
  benchmark: Open-Prompt (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_ood_q_msa_of_hotpotqa
  defence_id: msa_q
  benchmark: HotpotQA (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_q_msa_of_qasper
  defence_id: msa_q
  benchmark: Qasper (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_q_msa_of_injecagent
  defence_id: msa_q
  benchmark: InjecAgent (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_q_msa_of_sep
  defence_id: msa_q
  benchmark: SEP (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_q_msa_of_mmlu
  defence_id: msa_q
  benchmark: MMLU (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_q_msa_of_openprompt
  defence_id: msa_q
  benchmark: Open-Prompt (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_q_la_of_hotpotqa
  defence_id: la_q
  benchmark: HotpotQA (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_q_la_of_qasper
  defence_id: la_q
  benchmark: Qasper (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_q_la_of_injecagent
  defence_id: la_q
  benchmark: InjecAgent (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_q_la_of_sep
  defence_id: la_q
  benchmark: SEP (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_q_la_of_mmlu
  defence_id: la_q
  benchmark: MMLU (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_q_la_of_openprompt
  defence_id: la_q
  benchmark: Open-Prompt (out of distribution); maximum over the optimization-free attacks
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_q_none_aof_hotpotqa
  defence_id: none:Qwen3-4B-Instruct
  benchmark: HotpotQA (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_ood_q_none_aof_qasper
  defence_id: none:Qwen3-4B-Instruct
  benchmark: Qasper (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_ood_q_none_aof_injecagent
  defence_id: none:Qwen3-4B-Instruct
  benchmark: InjecAgent (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_ood_q_none_aof_sep
  defence_id: none:Qwen3-4B-Instruct
  benchmark: SEP (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_ood_q_none_aof_mmlu
  defence_id: none:Qwen3-4B-Instruct
  benchmark: MMLU (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_ood_q_none_aof_openprompt
  defence_id: none:Qwen3-4B-Instruct
  benchmark: Open-Prompt (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_ood_q_msa_aof_hotpotqa
  defence_id: msa_q
  benchmark: HotpotQA (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_q_msa_aof_qasper
  defence_id: msa_q
  benchmark: Qasper (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_q_msa_aof_injecagent
  defence_id: msa_q
  benchmark: InjecAgent (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_q_msa_aof_sep
  defence_id: msa_q
  benchmark: SEP (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_q_msa_aof_mmlu
  defence_id: msa_q
  benchmark: MMLU (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_q_msa_aof_openprompt
  defence_id: msa_q
  benchmark: Open-Prompt (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_q_la_aof_hotpotqa
  defence_id: la_q
  benchmark: HotpotQA (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_q_la_aof_qasper
  defence_id: la_q
  benchmark: Qasper (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_q_la_aof_injecagent
  defence_id: la_q
  benchmark: InjecAgent (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_q_la_aof_sep
  defence_id: la_q
  benchmark: SEP (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_q_la_aof_mmlu
  defence_id: la_q
  benchmark: MMLU (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_ood_q_la_aof_openprompt
  defence_id: la_q
  benchmark: Open-Prompt (out of distribution); adaptive optimization-free attacks with fake delimiters
  attacker_type: adaptive
  attacker_access: white_box
  judge: attack success as defined for each dataset (README of the authors' repository)
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 10
  attacker_budget: fake delimiters near the official ones
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_u_l_none_bbh
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: BBH
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_u_l_none_ifeval
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: IFEval
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_u_l_none_mmlupro
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: MMLU-Pro
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_u_l_none_mmlu
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: MMLU
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_u_l_none_alpaca
  defence_id: none:Llama3.1-8B-Instruct
  benchmark: AlpacaEval2
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_u_l_msa_bbh
  defence_id: msa_l
  benchmark: BBH
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
- id: ev_u_l_msa_ifeval
  defence_id: msa_l
  benchmark: IFEval
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
- id: ev_u_l_msa_mmlupro
  defence_id: msa_l
  benchmark: MMLU-Pro
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
- id: ev_u_l_msa_mmlu
  defence_id: msa_l
  benchmark: MMLU
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
- id: ev_u_l_msa_alpaca
  defence_id: msa_l
  benchmark: AlpacaEval2
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
- id: ev_u_l_la_bbh
  defence_id: la_l
  benchmark: BBH
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
- id: ev_u_l_la_ifeval
  defence_id: la_l
  benchmark: IFEval
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
- id: ev_u_l_la_mmlupro
  defence_id: la_l
  benchmark: MMLU-Pro
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
- id: ev_u_l_la_mmlu
  defence_id: la_l
  benchmark: MMLU
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
- id: ev_u_l_la_alpaca
  defence_id: la_l
  benchmark: AlpacaEval2
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
- id: ev_u_q_none_bbh
  defence_id: none:Qwen3-4B-Instruct
  benchmark: BBH
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_u_q_none_ifeval
  defence_id: none:Qwen3-4B-Instruct
  benchmark: IFEval
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_u_q_none_mmlupro
  defence_id: none:Qwen3-4B-Instruct
  benchmark: MMLU-Pro
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_u_q_none_mmlu
  defence_id: none:Qwen3-4B-Instruct
  benchmark: MMLU
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_u_q_none_alpaca
  defence_id: none:Qwen3-4B-Instruct
  benchmark: AlpacaEval2
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_u_q_msa_bbh
  defence_id: msa_q
  benchmark: BBH
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
- id: ev_u_q_msa_ifeval
  defence_id: msa_q
  benchmark: IFEval
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
- id: ev_u_q_msa_mmlupro
  defence_id: msa_q
  benchmark: MMLU-Pro
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
- id: ev_u_q_msa_mmlu
  defence_id: msa_q
  benchmark: MMLU
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
- id: ev_u_q_msa_alpaca
  defence_id: msa_q
  benchmark: AlpacaEval2
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
- id: ev_u_q_la_bbh
  defence_id: la_q
  benchmark: BBH
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
- id: ev_u_q_la_ifeval
  defence_id: la_q
  benchmark: IFEval
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
- id: ev_u_q_la_mmlupro
  defence_id: la_q
  benchmark: MMLU-Pro
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
- id: ev_u_q_la_mmlu
  defence_id: la_q
  benchmark: MMLU
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
- id: ev_u_q_la_alpaca
  defence_id: la_q
  benchmark: AlpacaEval2
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: localalign_ccs2026
  page: 11
  decoding: not reported
results:
- id: r_af_none_of
  evaluation_id: ev_af_none_of
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '89.42'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_af_none_gcg
  evaluation_id: ev_af_none_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (GCG)
  value: '26.92'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_af_none_agcg
  evaluation_id: ev_af_none_agcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive GCG)
  value: '92.31'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_af_struq_of
  evaluation_id: ev_af_struq_of
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '9.13'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_af_struq_gcg
  evaluation_id: ev_af_struq_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (GCG)
  value: '16.35'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_af_struq_agcg
  evaluation_id: ev_af_struq_agcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive GCG)
  value: '11.54'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_af_msa_of
  evaluation_id: ev_af_msa_of
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '1.44'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_af_msa_gcg
  evaluation_id: ev_af_msa_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (GCG)
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_af_msa_agcg
  evaluation_id: ev_af_msa_agcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive GCG)
  value: '2.40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_af_la_of
  evaluation_id: ev_af_la_of
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_af_la_gcg
  evaluation_id: ev_af_la_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (GCG)
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_af_la_agcg
  evaluation_id: ev_af_la_agcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive GCG)
  value: '0.48'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 2
- id: r_ood_l_none_of_hotpotqa
  evaluation_id: ev_ood_l_none_of_hotpotqa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '74.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_none_of_qasper
  evaluation_id: ev_ood_l_none_of_qasper
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '83.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_none_of_injecagent
  evaluation_id: ev_ood_l_none_of_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '81.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_none_of_sep
  evaluation_id: ev_ood_l_none_of_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '91.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_none_of_mmlu
  evaluation_id: ev_ood_l_none_of_mmlu
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '90.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_none_of_openprompt
  evaluation_id: ev_ood_l_none_of_openprompt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '87.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_msa_of_hotpotqa
  evaluation_id: ev_ood_l_msa_of_hotpotqa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '44.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_msa_of_qasper
  evaluation_id: ev_ood_l_msa_of_qasper
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '45.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_msa_of_injecagent
  evaluation_id: ev_ood_l_msa_of_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '13.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_msa_of_sep
  evaluation_id: ev_ood_l_msa_of_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_msa_of_mmlu
  evaluation_id: ev_ood_l_msa_of_mmlu
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '8.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_msa_of_openprompt
  evaluation_id: ev_ood_l_msa_of_openprompt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '0.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_la_of_hotpotqa
  evaluation_id: ev_ood_l_la_of_hotpotqa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '8.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_la_of_qasper
  evaluation_id: ev_ood_l_la_of_qasper
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '7.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_la_of_injecagent
  evaluation_id: ev_ood_l_la_of_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_la_of_sep
  evaluation_id: ev_ood_l_la_of_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_la_of_mmlu
  evaluation_id: ev_ood_l_la_of_mmlu
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_la_of_openprompt
  evaluation_id: ev_ood_l_la_of_openprompt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_none_aof_hotpotqa
  evaluation_id: ev_ood_l_none_aof_hotpotqa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '78.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_none_aof_qasper
  evaluation_id: ev_ood_l_none_aof_qasper
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '87.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_none_aof_injecagent
  evaluation_id: ev_ood_l_none_aof_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '73.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_none_aof_sep
  evaluation_id: ev_ood_l_none_aof_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '85.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_none_aof_mmlu
  evaluation_id: ev_ood_l_none_aof_mmlu
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '92.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_none_aof_openprompt
  evaluation_id: ev_ood_l_none_aof_openprompt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '98.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_msa_aof_hotpotqa
  evaluation_id: ev_ood_l_msa_aof_hotpotqa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '55.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_msa_aof_qasper
  evaluation_id: ev_ood_l_msa_aof_qasper
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '56.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_msa_aof_injecagent
  evaluation_id: ev_ood_l_msa_aof_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '6.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_msa_aof_sep
  evaluation_id: ev_ood_l_msa_aof_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '2.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_msa_aof_mmlu
  evaluation_id: ev_ood_l_msa_aof_mmlu
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '24.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_msa_aof_openprompt
  evaluation_id: ev_ood_l_msa_aof_openprompt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '8.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_la_aof_hotpotqa
  evaluation_id: ev_ood_l_la_aof_hotpotqa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '23.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_la_aof_qasper
  evaluation_id: ev_ood_l_la_aof_qasper
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '9.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_la_aof_injecagent
  evaluation_id: ev_ood_l_la_aof_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_la_aof_sep
  evaluation_id: ev_ood_l_la_aof_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '0.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_la_aof_mmlu
  evaluation_id: ev_ood_l_la_aof_mmlu
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '0.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_l_la_aof_openprompt
  evaluation_id: ev_ood_l_la_aof_openprompt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_none_of_hotpotqa
  evaluation_id: ev_ood_q_none_of_hotpotqa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '22.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_none_of_qasper
  evaluation_id: ev_ood_q_none_of_qasper
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '11.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_none_of_injecagent
  evaluation_id: ev_ood_q_none_of_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '76.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_none_of_sep
  evaluation_id: ev_ood_q_none_of_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '94.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_none_of_mmlu
  evaluation_id: ev_ood_q_none_of_mmlu
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '99.90'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_none_of_openprompt
  evaluation_id: ev_ood_q_none_of_openprompt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '99.20'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_msa_of_hotpotqa
  evaluation_id: ev_ood_q_msa_of_hotpotqa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '16.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_msa_of_qasper
  evaluation_id: ev_ood_q_msa_of_qasper
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '4.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_msa_of_injecagent
  evaluation_id: ev_ood_q_msa_of_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '43.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_msa_of_sep
  evaluation_id: ev_ood_q_msa_of_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '0.90'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_msa_of_mmlu
  evaluation_id: ev_ood_q_msa_of_mmlu
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '11.60'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_msa_of_openprompt
  evaluation_id: ev_ood_q_msa_of_openprompt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '3.84'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_la_of_hotpotqa
  evaluation_id: ev_ood_q_la_of_hotpotqa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '2.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_la_of_qasper
  evaluation_id: ev_ood_q_la_of_qasper
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '2.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_la_of_injecagent
  evaluation_id: ev_ood_q_la_of_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '4.60'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_la_of_sep
  evaluation_id: ev_ood_q_la_of_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_la_of_mmlu
  evaluation_id: ev_ood_q_la_of_mmlu
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_la_of_openprompt
  evaluation_id: ev_ood_q_la_of_openprompt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Optimization-Free)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_none_aof_hotpotqa
  evaluation_id: ev_ood_q_none_aof_hotpotqa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '30.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_none_aof_qasper
  evaluation_id: ev_ood_q_none_aof_qasper
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '44.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_none_aof_injecagent
  evaluation_id: ev_ood_q_none_aof_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '80.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_none_aof_sep
  evaluation_id: ev_ood_q_none_aof_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '96.39'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_none_aof_mmlu
  evaluation_id: ev_ood_q_none_aof_mmlu
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '99.70'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_none_aof_openprompt
  evaluation_id: ev_ood_q_none_aof_openprompt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '99.76'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_msa_aof_hotpotqa
  evaluation_id: ev_ood_q_msa_aof_hotpotqa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '14.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_msa_aof_qasper
  evaluation_id: ev_ood_q_msa_aof_qasper
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '14.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_msa_aof_injecagent
  evaluation_id: ev_ood_q_msa_aof_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '68.20'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_msa_aof_sep
  evaluation_id: ev_ood_q_msa_aof_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '20.70'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_msa_aof_mmlu
  evaluation_id: ev_ood_q_msa_aof_mmlu
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '89.40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_msa_aof_openprompt
  evaluation_id: ev_ood_q_msa_aof_openprompt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '1.80'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_la_aof_hotpotqa
  evaluation_id: ev_ood_q_la_aof_hotpotqa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '4.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_la_aof_qasper
  evaluation_id: ev_ood_q_la_aof_qasper
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '6.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_la_aof_injecagent
  evaluation_id: ev_ood_q_la_aof_injecagent
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '15.70'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_la_aof_sep
  evaluation_id: ev_ood_q_la_aof_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '2.15'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_la_aof_mmlu
  evaluation_id: ev_ood_q_la_aof_mmlu
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '9.10'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_ood_q_la_aof_openprompt
  evaluation_id: ev_ood_q_la_aof_openprompt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Adaptive Optimization-Free)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 1
- id: r_u_l_none_bbh
  evaluation_id: ev_u_l_none_bbh
  quantity: general_utility
  coverage: direct
  metric_as_printed: BBH
  value: '71.63'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_l_none_ifeval
  evaluation_id: ev_u_l_none_ifeval
  quantity: general_utility
  coverage: direct
  metric_as_printed: IFEval
  value: '80.34'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_l_none_mmlupro
  evaluation_id: ev_u_l_none_mmlupro
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro
  value: '42.25'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_l_none_mmlu
  evaluation_id: ev_u_l_none_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '68.23'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_l_none_alpaca
  evaluation_id: ev_u_l_none_alpaca
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval2
  value: '85.90'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_l_msa_bbh
  evaluation_id: ev_u_l_msa_bbh
  quantity: general_utility
  coverage: direct
  metric_as_printed: BBH
  value: '71.03'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_l_msa_ifeval
  evaluation_id: ev_u_l_msa_ifeval
  quantity: general_utility
  coverage: direct
  metric_as_printed: IFEval
  value: '75.23'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_l_msa_mmlupro
  evaluation_id: ev_u_l_msa_mmlupro
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro
  value: '42.24'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_l_msa_mmlu
  evaluation_id: ev_u_l_msa_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '67.48'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_l_msa_alpaca
  evaluation_id: ev_u_l_msa_alpaca
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval2
  value: '85.59'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_l_la_bbh
  evaluation_id: ev_u_l_la_bbh
  quantity: general_utility
  coverage: direct
  metric_as_printed: BBH
  value: '71.37'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_l_la_ifeval
  evaluation_id: ev_u_l_la_ifeval
  quantity: general_utility
  coverage: direct
  metric_as_printed: IFEval
  value: '73.93'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_l_la_mmlupro
  evaluation_id: ev_u_l_la_mmlupro
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro
  value: '44.12'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_l_la_mmlu
  evaluation_id: ev_u_l_la_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '65.66'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_l_la_alpaca
  evaluation_id: ev_u_l_la_alpaca
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval2
  value: '84.16'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_q_none_bbh
  evaluation_id: ev_u_q_none_bbh
  quantity: general_utility
  coverage: direct
  metric_as_printed: BBH
  value: '76.64'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_q_none_ifeval
  evaluation_id: ev_u_q_none_ifeval
  quantity: general_utility
  coverage: direct
  metric_as_printed: IFEval
  value: '86.75'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_q_none_mmlupro
  evaluation_id: ev_u_q_none_mmlupro
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro
  value: '56.74'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_q_none_mmlu
  evaluation_id: ev_u_q_none_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '72.58'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_q_none_alpaca
  evaluation_id: ev_u_q_none_alpaca
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval2
  value: '90.19'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_q_msa_bbh
  evaluation_id: ev_u_q_msa_bbh
  quantity: general_utility
  coverage: direct
  metric_as_printed: BBH
  value: '77.66'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_q_msa_ifeval
  evaluation_id: ev_u_q_msa_ifeval
  quantity: general_utility
  coverage: direct
  metric_as_printed: IFEval
  value: '85.74'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_q_msa_mmlupro
  evaluation_id: ev_u_q_msa_mmlupro
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro
  value: '50.19'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_q_msa_mmlu
  evaluation_id: ev_u_q_msa_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '72.11'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_q_msa_alpaca
  evaluation_id: ev_u_q_msa_alpaca
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval2
  value: '90.34'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_q_la_bbh
  evaluation_id: ev_u_q_la_bbh
  quantity: general_utility
  coverage: direct
  metric_as_printed: BBH
  value: '75.66'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_q_la_ifeval
  evaluation_id: ev_u_q_la_ifeval
  quantity: general_utility
  coverage: direct
  metric_as_printed: IFEval
  value: '83.75'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_q_la_mmlupro
  evaluation_id: ev_u_q_la_mmlupro
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU-Pro
  value: '49.35'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_q_la_mmlu
  evaluation_id: ev_u_q_la_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '72.12'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
- id: r_u_q_la_alpaca
  evaluation_id: ev_u_q_la_alpaca
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval2
  value: '89.63'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 3
comparisons:
- id: c_af_struq_of
  left: ev_af_struq_of
  right: ev_af_none_of
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_af_struq_gcg
  left: ev_af_struq_gcg
  right: ev_af_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_af_struq_agcg
  left: ev_af_struq_agcg
  right: ev_af_none_agcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_af_msa_of
  left: ev_af_msa_of
  right: ev_af_none_of
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_af_msa_gcg
  left: ev_af_msa_gcg
  right: ev_af_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_af_msa_agcg
  left: ev_af_msa_agcg
  right: ev_af_none_agcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_af_la_of
  left: ev_af_la_of
  right: ev_af_none_of
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_af_la_gcg
  left: ev_af_la_gcg
  right: ev_af_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_af_la_agcg
  left: ev_af_la_agcg
  right: ev_af_none_agcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_af_la_msa_of
  left: ev_af_la_of
  right: ev_af_msa_of
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_af_la_msa_gcg
  left: ev_af_la_gcg
  right: ev_af_msa_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_af_la_msa_agcg
  left: ev_af_la_agcg
  right: ev_af_msa_agcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_msa_of_hotpotqa
  left: ev_ood_l_msa_of_hotpotqa
  right: ev_ood_l_none_of_hotpotqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_la_of_hotpotqa
  left: ev_ood_l_la_of_hotpotqa
  right: ev_ood_l_none_of_hotpotqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_la_msa_of_hotpotqa
  left: ev_ood_l_la_of_hotpotqa
  right: ev_ood_l_msa_of_hotpotqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_msa_of_qasper
  left: ev_ood_l_msa_of_qasper
  right: ev_ood_l_none_of_qasper
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_la_of_qasper
  left: ev_ood_l_la_of_qasper
  right: ev_ood_l_none_of_qasper
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_la_msa_of_qasper
  left: ev_ood_l_la_of_qasper
  right: ev_ood_l_msa_of_qasper
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_msa_of_injecagent
  left: ev_ood_l_msa_of_injecagent
  right: ev_ood_l_none_of_injecagent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_la_of_injecagent
  left: ev_ood_l_la_of_injecagent
  right: ev_ood_l_none_of_injecagent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_la_msa_of_injecagent
  left: ev_ood_l_la_of_injecagent
  right: ev_ood_l_msa_of_injecagent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_msa_of_sep
  left: ev_ood_l_msa_of_sep
  right: ev_ood_l_none_of_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_la_of_sep
  left: ev_ood_l_la_of_sep
  right: ev_ood_l_none_of_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_la_msa_of_sep
  left: ev_ood_l_la_of_sep
  right: ev_ood_l_msa_of_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_msa_of_mmlu
  left: ev_ood_l_msa_of_mmlu
  right: ev_ood_l_none_of_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_la_of_mmlu
  left: ev_ood_l_la_of_mmlu
  right: ev_ood_l_none_of_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_la_msa_of_mmlu
  left: ev_ood_l_la_of_mmlu
  right: ev_ood_l_msa_of_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_msa_of_openprompt
  left: ev_ood_l_msa_of_openprompt
  right: ev_ood_l_none_of_openprompt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_la_of_openprompt
  left: ev_ood_l_la_of_openprompt
  right: ev_ood_l_none_of_openprompt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_la_msa_of_openprompt
  left: ev_ood_l_la_of_openprompt
  right: ev_ood_l_msa_of_openprompt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_msa_aof_hotpotqa
  left: ev_ood_l_msa_aof_hotpotqa
  right: ev_ood_l_none_aof_hotpotqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_la_aof_hotpotqa
  left: ev_ood_l_la_aof_hotpotqa
  right: ev_ood_l_none_aof_hotpotqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_la_msa_aof_hotpotqa
  left: ev_ood_l_la_aof_hotpotqa
  right: ev_ood_l_msa_aof_hotpotqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_msa_aof_qasper
  left: ev_ood_l_msa_aof_qasper
  right: ev_ood_l_none_aof_qasper
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_la_aof_qasper
  left: ev_ood_l_la_aof_qasper
  right: ev_ood_l_none_aof_qasper
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_la_msa_aof_qasper
  left: ev_ood_l_la_aof_qasper
  right: ev_ood_l_msa_aof_qasper
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_msa_aof_injecagent
  left: ev_ood_l_msa_aof_injecagent
  right: ev_ood_l_none_aof_injecagent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_la_aof_injecagent
  left: ev_ood_l_la_aof_injecagent
  right: ev_ood_l_none_aof_injecagent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_la_msa_aof_injecagent
  left: ev_ood_l_la_aof_injecagent
  right: ev_ood_l_msa_aof_injecagent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_msa_aof_sep
  left: ev_ood_l_msa_aof_sep
  right: ev_ood_l_none_aof_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_la_aof_sep
  left: ev_ood_l_la_aof_sep
  right: ev_ood_l_none_aof_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_la_msa_aof_sep
  left: ev_ood_l_la_aof_sep
  right: ev_ood_l_msa_aof_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_msa_aof_mmlu
  left: ev_ood_l_msa_aof_mmlu
  right: ev_ood_l_none_aof_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_la_aof_mmlu
  left: ev_ood_l_la_aof_mmlu
  right: ev_ood_l_none_aof_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_la_msa_aof_mmlu
  left: ev_ood_l_la_aof_mmlu
  right: ev_ood_l_msa_aof_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_msa_aof_openprompt
  left: ev_ood_l_msa_aof_openprompt
  right: ev_ood_l_none_aof_openprompt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_la_aof_openprompt
  left: ev_ood_l_la_aof_openprompt
  right: ev_ood_l_none_aof_openprompt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_l_la_msa_aof_openprompt
  left: ev_ood_l_la_aof_openprompt
  right: ev_ood_l_msa_aof_openprompt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_msa_of_hotpotqa
  left: ev_ood_q_msa_of_hotpotqa
  right: ev_ood_q_none_of_hotpotqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_la_of_hotpotqa
  left: ev_ood_q_la_of_hotpotqa
  right: ev_ood_q_none_of_hotpotqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_la_msa_of_hotpotqa
  left: ev_ood_q_la_of_hotpotqa
  right: ev_ood_q_msa_of_hotpotqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_msa_of_qasper
  left: ev_ood_q_msa_of_qasper
  right: ev_ood_q_none_of_qasper
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_la_of_qasper
  left: ev_ood_q_la_of_qasper
  right: ev_ood_q_none_of_qasper
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_la_msa_of_qasper
  left: ev_ood_q_la_of_qasper
  right: ev_ood_q_msa_of_qasper
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_msa_of_injecagent
  left: ev_ood_q_msa_of_injecagent
  right: ev_ood_q_none_of_injecagent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_la_of_injecagent
  left: ev_ood_q_la_of_injecagent
  right: ev_ood_q_none_of_injecagent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_la_msa_of_injecagent
  left: ev_ood_q_la_of_injecagent
  right: ev_ood_q_msa_of_injecagent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_msa_of_sep
  left: ev_ood_q_msa_of_sep
  right: ev_ood_q_none_of_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_la_of_sep
  left: ev_ood_q_la_of_sep
  right: ev_ood_q_none_of_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_la_msa_of_sep
  left: ev_ood_q_la_of_sep
  right: ev_ood_q_msa_of_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_msa_of_mmlu
  left: ev_ood_q_msa_of_mmlu
  right: ev_ood_q_none_of_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_la_of_mmlu
  left: ev_ood_q_la_of_mmlu
  right: ev_ood_q_none_of_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_la_msa_of_mmlu
  left: ev_ood_q_la_of_mmlu
  right: ev_ood_q_msa_of_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_msa_of_openprompt
  left: ev_ood_q_msa_of_openprompt
  right: ev_ood_q_none_of_openprompt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_la_of_openprompt
  left: ev_ood_q_la_of_openprompt
  right: ev_ood_q_none_of_openprompt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_la_msa_of_openprompt
  left: ev_ood_q_la_of_openprompt
  right: ev_ood_q_msa_of_openprompt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_msa_aof_hotpotqa
  left: ev_ood_q_msa_aof_hotpotqa
  right: ev_ood_q_none_aof_hotpotqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_la_aof_hotpotqa
  left: ev_ood_q_la_aof_hotpotqa
  right: ev_ood_q_none_aof_hotpotqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_la_msa_aof_hotpotqa
  left: ev_ood_q_la_aof_hotpotqa
  right: ev_ood_q_msa_aof_hotpotqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_msa_aof_qasper
  left: ev_ood_q_msa_aof_qasper
  right: ev_ood_q_none_aof_qasper
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_la_aof_qasper
  left: ev_ood_q_la_aof_qasper
  right: ev_ood_q_none_aof_qasper
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_la_msa_aof_qasper
  left: ev_ood_q_la_aof_qasper
  right: ev_ood_q_msa_aof_qasper
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_msa_aof_injecagent
  left: ev_ood_q_msa_aof_injecagent
  right: ev_ood_q_none_aof_injecagent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_la_aof_injecagent
  left: ev_ood_q_la_aof_injecagent
  right: ev_ood_q_none_aof_injecagent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_la_msa_aof_injecagent
  left: ev_ood_q_la_aof_injecagent
  right: ev_ood_q_msa_aof_injecagent
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_msa_aof_sep
  left: ev_ood_q_msa_aof_sep
  right: ev_ood_q_none_aof_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_la_aof_sep
  left: ev_ood_q_la_aof_sep
  right: ev_ood_q_none_aof_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_la_msa_aof_sep
  left: ev_ood_q_la_aof_sep
  right: ev_ood_q_msa_aof_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_msa_aof_mmlu
  left: ev_ood_q_msa_aof_mmlu
  right: ev_ood_q_none_aof_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_la_aof_mmlu
  left: ev_ood_q_la_aof_mmlu
  right: ev_ood_q_none_aof_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_la_msa_aof_mmlu
  left: ev_ood_q_la_aof_mmlu
  right: ev_ood_q_msa_aof_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_msa_aof_openprompt
  left: ev_ood_q_msa_aof_openprompt
  right: ev_ood_q_none_aof_openprompt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_la_aof_openprompt
  left: ev_ood_q_la_aof_openprompt
  right: ev_ood_q_none_aof_openprompt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_ood_q_la_msa_aof_openprompt
  left: ev_ood_q_la_aof_openprompt
  right: ev_ood_q_msa_aof_openprompt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_u_l_msa_bbh
  left: ev_u_l_msa_bbh
  right: ev_u_l_none_bbh
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_u_l_la_bbh
  left: ev_u_l_la_bbh
  right: ev_u_l_none_bbh
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_u_l_msa_ifeval
  left: ev_u_l_msa_ifeval
  right: ev_u_l_none_ifeval
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_u_l_la_ifeval
  left: ev_u_l_la_ifeval
  right: ev_u_l_none_ifeval
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_u_l_msa_mmlupro
  left: ev_u_l_msa_mmlupro
  right: ev_u_l_none_mmlupro
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_u_l_la_mmlupro
  left: ev_u_l_la_mmlupro
  right: ev_u_l_none_mmlupro
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_u_l_msa_mmlu
  left: ev_u_l_msa_mmlu
  right: ev_u_l_none_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_u_l_la_mmlu
  left: ev_u_l_la_mmlu
  right: ev_u_l_none_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_u_l_msa_alpaca
  left: ev_u_l_msa_alpaca
  right: ev_u_l_none_alpaca
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_u_l_la_alpaca
  left: ev_u_l_la_alpaca
  right: ev_u_l_none_alpaca
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_u_q_msa_bbh
  left: ev_u_q_msa_bbh
  right: ev_u_q_none_bbh
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_u_q_la_bbh
  left: ev_u_q_la_bbh
  right: ev_u_q_none_bbh
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_u_q_msa_ifeval
  left: ev_u_q_msa_ifeval
  right: ev_u_q_none_ifeval
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_u_q_la_ifeval
  left: ev_u_q_la_ifeval
  right: ev_u_q_none_ifeval
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_u_q_msa_mmlupro
  left: ev_u_q_msa_mmlupro
  right: ev_u_q_none_mmlupro
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_u_q_la_mmlupro
  left: ev_u_q_la_mmlupro
  right: ev_u_q_none_mmlupro
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_u_q_msa_mmlu
  left: ev_u_q_msa_mmlu
  right: ev_u_q_none_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_u_q_la_mmlu
  left: ev_u_q_la_mmlu
  right: ev_u_q_none_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_u_q_msa_alpaca
  left: ev_u_q_msa_alpaca
  right: ev_u_q_none_alpaca
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_u_q_la_alpaca
  left: ev_u_q_la_alpaca
  right: ev_u_q_none_alpaca
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
claims:
- id: k1
  text: LocalAlign is the first method to keep attack success below 10% in most settings, including stronger attacks
  location: abstract
  anchor: reduces the attack success rates of prompt injections to below 10%
  jobs:
  - R1
  supported_by:
  - c_af_la_of
  - c_af_la_gcg
  - c_af_la_agcg
  - c_af_la_msa_of
  - c_af_la_msa_gcg
  - c_af_la_msa_agcg
  - c_ood_l_la_of_hotpotqa
  - c_ood_l_la_msa_of_hotpotqa
  - c_ood_l_la_of_qasper
  - c_ood_l_la_msa_of_qasper
  - c_ood_l_la_of_injecagent
  - c_ood_l_la_msa_of_injecagent
  - c_ood_l_la_of_sep
  - c_ood_l_la_msa_of_sep
  - c_ood_l_la_of_mmlu
  - c_ood_l_la_msa_of_mmlu
  - c_ood_l_la_of_openprompt
  - c_ood_l_la_msa_of_openprompt
  - c_ood_l_la_aof_hotpotqa
  - c_ood_l_la_msa_aof_hotpotqa
  - c_ood_l_la_aof_qasper
  - c_ood_l_la_msa_aof_qasper
  - c_ood_l_la_aof_injecagent
  - c_ood_l_la_msa_aof_injecagent
  - c_ood_l_la_aof_sep
  - c_ood_l_la_msa_aof_sep
  - c_ood_l_la_aof_mmlu
  - c_ood_l_la_msa_aof_mmlu
  - c_ood_l_la_aof_openprompt
  - c_ood_l_la_msa_aof_openprompt
  - c_ood_q_la_of_hotpotqa
  - c_ood_q_la_msa_of_hotpotqa
  - c_ood_q_la_of_qasper
  - c_ood_q_la_msa_of_qasper
  - c_ood_q_la_of_injecagent
  - c_ood_q_la_msa_of_injecagent
  - c_ood_q_la_of_sep
  - c_ood_q_la_msa_of_sep
  - c_ood_q_la_of_mmlu
  - c_ood_q_la_msa_of_mmlu
  - c_ood_q_la_of_openprompt
  - c_ood_q_la_msa_of_openprompt
  - c_ood_q_la_aof_hotpotqa
  - c_ood_q_la_msa_aof_hotpotqa
  - c_ood_q_la_aof_qasper
  - c_ood_q_la_msa_aof_qasper
  - c_ood_q_la_aof_injecagent
  - c_ood_q_la_msa_aof_injecagent
  - c_ood_q_la_aof_sep
  - c_ood_q_la_msa_aof_sep
  - c_ood_q_la_aof_mmlu
  - c_ood_q_la_msa_aof_mmlu
  - c_ood_q_la_aof_openprompt
  - c_ood_q_la_msa_aof_openprompt
  page: 1
- id: k2
  text: LocalAlign keeps utility comparable to the model before defence fine-tuning
  location: abstract
  anchor: maintains utility comparable to the model before fine-tuning for defense
  jobs:
  - general_utility
  supported_by:
  - c_u_l_la_bbh
  - c_u_l_la_ifeval
  - c_u_l_la_mmlupro
  - c_u_l_la_mmlu
  - c_u_l_la_alpaca
  - c_u_q_la_bbh
  - c_u_q_la_ifeval
  - c_u_q_la_mmlupro
  - c_u_q_la_mmlu
  - c_u_q_la_alpaca
  page: 1
  contradicted_by:
  - r_u_l_la_ifeval
  - r_u_q_la_mmlupro
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 1 and 2
    general_utility: Table 3
schema_gaps:
- where: defences la_*
  note: the input format of LocalAlign is not restated; it warms up with the Meta-SecAlign recipe and is attacked with fake delimiters aimed at structured separation, so it is coded with Meta-SecAlign's separate data role
- where: results r_af_*, r_ood_*
  note: Optimization-Free is the maximum ASR over five attack templates, taken per dataset after evaluation (p. 14)

`alwaysfall_arxiv2026` (2605.17634)

In [ ]:
%%writefile reports/records/alwaysfall_arxiv2026.yaml
report:
  id: alwaysfall_arxiv2026
  slug: '2605.17634'
  title: AI Agents May Always Fall for Prompt Injections
  first_author: Sahar Abdelnabi
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2605.17634'
  version_read: arXiv v2
  first_date: '2026-05-17'
  read: F
  role: re_measures
  code_release: not checked
  extraction_scope: 'Section 2: Figure 2 panels (c) and (d) (LLaMA-3.3-70B against SecAlign-70B on the paired email set and on LLMail-Inject); the abstract claim (the Prompt Guard classifiers, which are detectors, the frontier-model
    scenarios of Section 4 and the appendices not coded)'
defences:
- id: msa70b
  name: SecAlign-70B (Meta SecAlign 70B, re-measured)
  origin: other_report
  base_checkpoint: LLaMA-3.3-70B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  name_as_printed: SecAlign-70B
  anchor: SecAlign-70B [Chen et al., 2025], which is a fine-tuned LLaMA model to resist
evaluations:
- id: ev_rneu_none
  defence_id: none:LLaMA-3.3-70B-Instruct
  benchmark: paired email set (4,200 scenarios; one clause differs between an authority-asserting and a neutral variant), email assistant with a send_email tool; restrictive (read-only) user prompt, neutral email
  attacker_type: none
  attacker_access: not_applicable
  judge: send_email tool called
  layout_same_as_defence: 'yes'
  measured_by: alwaysfall_arxiv2026
  page: 4
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_rneu_msa
  defence_id: msa70b
  benchmark: paired email set (4,200 scenarios; one clause differs between an authority-asserting and a neutral variant), email assistant with a send_email tool; restrictive (read-only) user prompt, neutral email
  attacker_type: none
  attacker_access: not_applicable
  judge: send_email tool called
  layout_same_as_defence: 'yes'
  measured_by: alwaysfall_arxiv2026
  page: 4
  decoding: not reported
- id: ev_rauth_none
  defence_id: none:LLaMA-3.3-70B-Instruct
  benchmark: paired email set (4,200 scenarios; one clause differs between an authority-asserting and a neutral variant), email assistant with a send_email tool; restrictive user prompt, authority-asserting email
  attacker_type: static
  attacker_access: black_box
  judge: send_email tool called
  layout_same_as_defence: 'yes'
  measured_by: alwaysfall_arxiv2026
  page: 4
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_rauth_msa
  defence_id: msa70b
  benchmark: paired email set (4,200 scenarios; one clause differs between an authority-asserting and a neutral variant), email assistant with a send_email tool; restrictive user prompt, authority-asserting email
  attacker_type: static
  attacker_access: black_box
  judge: send_email tool called
  layout_same_as_defence: 'yes'
  measured_by: alwaysfall_arxiv2026
  page: 4
  decoding: not reported
- id: ev_pneu_none
  defence_id: none:LLaMA-3.3-70B-Instruct
  benchmark: paired email set (4,200 scenarios; one clause differs between an authority-asserting and a neutral variant), email assistant with a send_email tool; permissive user prompt, neutral email
  attacker_type: none
  attacker_access: not_applicable
  judge: send_email tool called
  layout_same_as_defence: 'yes'
  measured_by: alwaysfall_arxiv2026
  page: 4
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_pneu_msa
  defence_id: msa70b
  benchmark: paired email set (4,200 scenarios; one clause differs between an authority-asserting and a neutral variant), email assistant with a send_email tool; permissive user prompt, neutral email
  attacker_type: none
  attacker_access: not_applicable
  judge: send_email tool called
  layout_same_as_defence: 'yes'
  measured_by: alwaysfall_arxiv2026
  page: 4
  decoding: not reported
- id: ev_pauth_none
  defence_id: none:LLaMA-3.3-70B-Instruct
  benchmark: paired email set (4,200 scenarios; one clause differs between an authority-asserting and a neutral variant), email assistant with a send_email tool; permissive user prompt, authority-asserting email
  attacker_type: none
  attacker_access: not_applicable
  judge: send_email tool called
  layout_same_as_defence: 'yes'
  measured_by: alwaysfall_arxiv2026
  page: 4
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_pauth_msa
  defence_id: msa70b
  benchmark: paired email set (4,200 scenarios; one clause differs between an authority-asserting and a neutral variant), email assistant with a send_email tool; permissive user prompt, authority-asserting email
  attacker_type: none
  attacker_access: not_applicable
  judge: send_email tool called
  layout_same_as_defence: 'yes'
  measured_by: alwaysfall_arxiv2026
  page: 4
  decoding: not reported
- id: ev_lm_none
  defence_id: none:LLaMA-3.3-70B-Instruct
  benchmark: LLMail-Inject (4,300 human-written injections in benign emails); user asks for a summary and a SECURITY ALERT on suspicious content
  attacker_type: human
  attacker_access: black_box
  judge: send_email tool called; injection reported in a SECURITY ALERT
  layout_same_as_defence: 'yes'
  measured_by: alwaysfall_arxiv2026
  page: 4
  decoding: not reported
  sample_size: 4,300
  baseline_kind: off_the_shelf
- id: ev_lm_msa
  defence_id: msa70b
  benchmark: LLMail-Inject (4,300 human-written injections in benign emails); user asks for a summary and a SECURITY ALERT on suspicious content
  attacker_type: human
  attacker_access: black_box
  judge: send_email tool called; injection reported in a SECURITY ALERT
  layout_same_as_defence: 'yes'
  measured_by: alwaysfall_arxiv2026
  page: 4
  decoding: not reported
  sample_size: 4,300
results:
- id: r_rneu_none
  evaluation_id: ev_rneu_none
  quantity: other
  coverage: direct
  metric_as_printed: Send rate (restrictive, neutral)
  value: '49.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 4
  location: Figure 2(c) and Section 2 text
- id: r_rneu_msa
  evaluation_id: ev_rneu_msa
  quantity: other
  coverage: direct
  metric_as_printed: Send rate (restrictive, neutral)
  value: '85.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 4
  location: Figure 2(c) and Section 2 text
- id: r_rauth_none
  evaluation_id: ev_rauth_none
  quantity: R1
  coverage: direct
  metric_as_printed: Send rate (restrictive, authority)
  value: '54.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 4
  location: Figure 2(c) and Section 2 text
- id: r_rauth_msa
  evaluation_id: ev_rauth_msa
  quantity: R1
  coverage: direct
  metric_as_printed: Send rate (restrictive, authority)
  value: '88.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 4
  location: Figure 2(c) and Section 2 text
- id: r_pneu_none
  evaluation_id: ev_pneu_none
  quantity: R2clean
  coverage: proxy
  metric_as_printed: Send rate (permissive, neutral)
  value: '97.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 4
  location: Figure 2(c)
- id: r_pneu_msa
  evaluation_id: ev_pneu_msa
  quantity: R2clean
  coverage: proxy
  metric_as_printed: Send rate (permissive, neutral)
  value: '99.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 4
  location: Figure 2(c)
- id: r_pauth_none
  evaluation_id: ev_pauth_none
  quantity: R2clean
  coverage: proxy
  metric_as_printed: Send rate (permissive, authority)
  value: '97.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 4
  location: Figure 2(c)
- id: r_pauth_msa
  evaluation_id: ev_pauth_msa
  quantity: R2clean
  coverage: proxy
  metric_as_printed: Send rate (permissive, authority)
  value: '99.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 4
  location: Figure 2(c)
- id: r_lm_none_send
  evaluation_id: ev_lm_none
  quantity: R1
  coverage: direct
  metric_as_printed: Send rate
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 4
  location: Figure 2(d) and Section 2 text
- id: r_lm_none_flag
  evaluation_id: ev_lm_none
  quantity: R2attack
  coverage: proxy
  metric_as_printed: Injections flagged
  value: '89.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 4
  location: Figure 2(d) and Section 2 text
- id: r_lm_msa_send
  evaluation_id: ev_lm_msa
  quantity: R1
  coverage: direct
  metric_as_printed: Send rate
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 4
  location: Figure 2(d) and Section 2 text
- id: r_lm_msa_flag
  evaluation_id: ev_lm_msa
  quantity: R2attack
  coverage: proxy
  metric_as_printed: Injections flagged
  value: '54.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 4
  location: Figure 2(d) and Section 2 text
comparisons:
- id: c_rneu
  left: ev_rneu_msa
  right: ev_rneu_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_rauth
  left: ev_rauth_msa
  right: ev_rauth_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_pneu
  left: ev_pneu_msa
  right: ev_pneu_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_pauth
  left: ev_pauth_msa
  right: ev_pauth_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_lm
  left: ev_lm_msa
  right: ev_lm_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 4
claims:
- id: k1
  text: data-instruction separation fails on contextual manipulation and degrades contextually appropriate behaviour
  location: abstract
  anchor: fails to detect attacks that operate through contextual manipulation
  jobs:
  - R1
  - R2
  supported_by:
  - c_rauth
  - c_rneu
  - c_lm
  page: 1
coverage:
  R1: direct
  R2clean: proxy
  R2attack: proxy
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Figure 2(c) restrictive authority cell; Figure 2(d) send rate
    R2clean: Figure 2(c) permissive cells (a send counts, its content is not judged)
    R2attack: Figure 2(d) injections flagged as the user asked
schema_gaps:
- where: results r_pneu_*, r_pauth_*
  note: the permissive send rates are printed only inside Figure 2(c), which is text in the PDF; they are coded figure_only with the printed value kept (checked against the page text by the batch notebook)
- where: evaluations ev_rneu_*
  note: under the restrictive prompt a send on the neutral email is an unauthorised action without any injected claim, so it is coded other

`actconsistency_arxiv2026` (2605.28467)

In [ ]:
%%writefile reports/records/actconsistency_arxiv2026.yaml
report:
  id: actconsistency_arxiv2026
  slug: '2605.28467'
  title: Mitigating Adaptive Attacks against Reasoning Models with Activation Consistency Training
  first_author: Avidan Shah
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2605.28467'
  version_read: arXiv v1
  first_date: '2026-05-27'
  read: F
  role: proposes
  code_release: not checked
  extraction_scope: 'Section 4 and Appendices A.4 and A.5: Table 1 (static prompt injection and utility, five models), Table 3 test split (adaptive GRPO attacker) and the Baseline and SecAlign rows of Table 4; the abstract
    claim (PromptArmor, a detector, is left out; the jailbreak results of Section 5 and Tables 5 to 8, the steering analysis and the train-split columns not coded)'
defences:
- id: act_q17
  name: ACT (activation consistency training)
  origin: this_report
  base_checkpoint: Qwen3-1.7B
  axis1:
  - no_signal
  axis2:
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  components:
  - pairs of a clean prompt and the same prompt with an injection wrapped into the data
  - LoRA training so that hidden states on the wrapped prompt match the frozen base model's hidden states on the clean prompt at shared suffix positions, every layer
  anchor: ACT minimizes
- id: bct_q17
  name: BCT (output-level consistency training)
  origin: this_report
  base_checkpoint: Qwen3-1.7B
  axis1:
  - no_signal
  axis2:
  - sft
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  components:
  - cross-entropy fine-tuning on the wrapped prompt with the base model's own completion (with its reasoning) of the clean prompt as target
  anchor: BCT requires no additional formalism beyond standard cross entropy
- id: act_q8
  name: ACT (activation consistency training)
  origin: this_report
  base_checkpoint: Qwen3-8B
  axis1:
  - no_signal
  axis2:
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  components:
  - pairs of a clean prompt and the same prompt with an injection wrapped into the data
  - LoRA training so that hidden states on the wrapped prompt match the frozen base model's hidden states on the clean prompt at shared suffix positions, every layer
  anchor: ACT minimizes
- id: bct_q8
  name: BCT (output-level consistency training)
  origin: this_report
  base_checkpoint: Qwen3-8B
  axis1:
  - no_signal
  axis2:
  - sft
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  components:
  - cross-entropy fine-tuning on the wrapped prompt with the base model's own completion (with its reasoning) of the clean prompt as target
  anchor: BCT requires no additional formalism beyond standard cross entropy
- id: act_phi
  name: ACT (activation consistency training)
  origin: this_report
  base_checkpoint: Phi-4-reasoning
  axis1:
  - no_signal
  axis2:
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  components:
  - pairs of a clean prompt and the same prompt with an injection wrapped into the data
  - LoRA training so that hidden states on the wrapped prompt match the frozen base model's hidden states on the clean prompt at shared suffix positions, every layer
  anchor: ACT minimizes
- id: bct_phi
  name: BCT (output-level consistency training)
  origin: this_report
  base_checkpoint: Phi-4-reasoning
  axis1:
  - no_signal
  axis2:
  - sft
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  components:
  - cross-entropy fine-tuning on the wrapped prompt with the base model's own completion (with its reasoning) of the clean prompt as target
  anchor: BCT requires no additional formalism beyond standard cross entropy
- id: act_oss
  name: ACT (activation consistency training)
  origin: this_report
  base_checkpoint: GPT-OSS-20B
  axis1:
  - no_signal
  axis2:
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  components:
  - pairs of a clean prompt and the same prompt with an injection wrapped into the data
  - LoRA training so that hidden states on the wrapped prompt match the frozen base model's hidden states on the clean prompt at shared suffix positions, every layer
  anchor: ACT minimizes
- id: bct_oss
  name: BCT (output-level consistency training)
  origin: this_report
  base_checkpoint: GPT-OSS-20B
  axis1:
  - no_signal
  axis2:
  - sft
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  components:
  - cross-entropy fine-tuning on the wrapped prompt with the base model's own completion (with its reasoning) of the clean prompt as target
  anchor: BCT requires no additional formalism beyond standard cross entropy
- id: act_gem
  name: ACT (activation consistency training)
  origin: this_report
  base_checkpoint: Gemma-4-E4B-it
  axis1:
  - no_signal
  axis2:
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  components:
  - pairs of a clean prompt and the same prompt with an injection wrapped into the data
  - LoRA training so that hidden states on the wrapped prompt match the frozen base model's hidden states on the clean prompt at shared suffix positions, every layer
  anchor: ACT minimizes
- id: bct_gem
  name: BCT (output-level consistency training)
  origin: this_report
  base_checkpoint: Gemma-4-E4B-it
  axis1:
  - no_signal
  axis2:
  - sft
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  components:
  - cross-entropy fine-tuning on the wrapped prompt with the base model's own completion (with its reasoning) of the clean prompt as target
  anchor: BCT requires no additional formalism beyond standard cross entropy
- id: secalign_q8
  name: SecAlign (DPO, applied by the authors to Qwen3-8B)
  origin: other_report
  base_checkpoint: Qwen3-8B
  axis1:
  - no_signal
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 19
  anchor: Hyperparameters follow the SecAlign paper
evaluations:
- id: ev_s_q17_none_opi
  defence_id: none:Qwen3-1.7B
  benchmark: Open-Prompt-Injection, 680 test examples, five attack formats
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined by the benchmark
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '680'
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_q17_none_af
  defence_id: none:Qwen3-1.7B
  benchmark: AlpacaFarm, 208 examples, 'Print exactly Hacked!', four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response starts with 'Hacked'
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '208'
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_q17_none_sep
  defence_id: none:Qwen3-1.7B
  benchmark: SEP, injection at start, middle or end
  attacker_type: static
  attacker_access: black_box
  judge: SecAlign's SEP protocol
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_q17_none_opiclean
  defence_id: none:Qwen3-1.7B
  benchmark: Open-Prompt-Injection clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: task accuracy
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_q17_none_mmlu
  defence_id: none:Qwen3-1.7B
  benchmark: MMLU
  attacker_type: none
  attacker_access: not_applicable
  judge: answer key
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_q17_act_opi
  defence_id: act_q17
  benchmark: Open-Prompt-Injection, 680 test examples, five attack formats
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined by the benchmark
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '680'
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_q17_act_af
  defence_id: act_q17
  benchmark: AlpacaFarm, 208 examples, 'Print exactly Hacked!', four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response starts with 'Hacked'
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '208'
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_q17_act_sep
  defence_id: act_q17
  benchmark: SEP, injection at start, middle or end
  attacker_type: static
  attacker_access: black_box
  judge: SecAlign's SEP protocol
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_q17_act_opiclean
  defence_id: act_q17
  benchmark: Open-Prompt-Injection clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: task accuracy
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_q17_act_mmlu
  defence_id: act_q17
  benchmark: MMLU
  attacker_type: none
  attacker_access: not_applicable
  judge: answer key
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_q17_bct_opi
  defence_id: bct_q17
  benchmark: Open-Prompt-Injection, 680 test examples, five attack formats
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined by the benchmark
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '680'
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_q17_bct_af
  defence_id: bct_q17
  benchmark: AlpacaFarm, 208 examples, 'Print exactly Hacked!', four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response starts with 'Hacked'
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '208'
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_q17_bct_sep
  defence_id: bct_q17
  benchmark: SEP, injection at start, middle or end
  attacker_type: static
  attacker_access: black_box
  judge: SecAlign's SEP protocol
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_q17_bct_opiclean
  defence_id: bct_q17
  benchmark: Open-Prompt-Injection clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: task accuracy
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_q17_bct_mmlu
  defence_id: bct_q17
  benchmark: MMLU
  attacker_type: none
  attacker_access: not_applicable
  judge: answer key
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_q8_none_opi
  defence_id: none:Qwen3-8B
  benchmark: Open-Prompt-Injection, 680 test examples, five attack formats
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined by the benchmark
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '680'
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_q8_none_af
  defence_id: none:Qwen3-8B
  benchmark: AlpacaFarm, 208 examples, 'Print exactly Hacked!', four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response starts with 'Hacked'
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '208'
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_q8_none_sep
  defence_id: none:Qwen3-8B
  benchmark: SEP, injection at start, middle or end
  attacker_type: static
  attacker_access: black_box
  judge: SecAlign's SEP protocol
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_q8_none_opiclean
  defence_id: none:Qwen3-8B
  benchmark: Open-Prompt-Injection clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: task accuracy
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_q8_none_mmlu
  defence_id: none:Qwen3-8B
  benchmark: MMLU
  attacker_type: none
  attacker_access: not_applicable
  judge: answer key
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_q8_act_opi
  defence_id: act_q8
  benchmark: Open-Prompt-Injection, 680 test examples, five attack formats
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined by the benchmark
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '680'
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_q8_act_af
  defence_id: act_q8
  benchmark: AlpacaFarm, 208 examples, 'Print exactly Hacked!', four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response starts with 'Hacked'
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '208'
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_q8_act_sep
  defence_id: act_q8
  benchmark: SEP, injection at start, middle or end
  attacker_type: static
  attacker_access: black_box
  judge: SecAlign's SEP protocol
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_q8_act_opiclean
  defence_id: act_q8
  benchmark: Open-Prompt-Injection clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: task accuracy
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_q8_act_mmlu
  defence_id: act_q8
  benchmark: MMLU
  attacker_type: none
  attacker_access: not_applicable
  judge: answer key
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_q8_bct_opi
  defence_id: bct_q8
  benchmark: Open-Prompt-Injection, 680 test examples, five attack formats
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined by the benchmark
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '680'
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_q8_bct_af
  defence_id: bct_q8
  benchmark: AlpacaFarm, 208 examples, 'Print exactly Hacked!', four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response starts with 'Hacked'
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '208'
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_q8_bct_sep
  defence_id: bct_q8
  benchmark: SEP, injection at start, middle or end
  attacker_type: static
  attacker_access: black_box
  judge: SecAlign's SEP protocol
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_q8_bct_opiclean
  defence_id: bct_q8
  benchmark: Open-Prompt-Injection clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: task accuracy
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_q8_bct_mmlu
  defence_id: bct_q8
  benchmark: MMLU
  attacker_type: none
  attacker_access: not_applicable
  judge: answer key
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_phi_none_opi
  defence_id: none:Phi-4-reasoning
  benchmark: Open-Prompt-Injection, 680 test examples, five attack formats
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined by the benchmark
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '680'
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_phi_none_af
  defence_id: none:Phi-4-reasoning
  benchmark: AlpacaFarm, 208 examples, 'Print exactly Hacked!', four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response starts with 'Hacked'
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '208'
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_phi_none_sep
  defence_id: none:Phi-4-reasoning
  benchmark: SEP, injection at start, middle or end
  attacker_type: static
  attacker_access: black_box
  judge: SecAlign's SEP protocol
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_phi_none_opiclean
  defence_id: none:Phi-4-reasoning
  benchmark: Open-Prompt-Injection clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: task accuracy
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_phi_none_mmlu
  defence_id: none:Phi-4-reasoning
  benchmark: MMLU
  attacker_type: none
  attacker_access: not_applicable
  judge: answer key
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_phi_act_opi
  defence_id: act_phi
  benchmark: Open-Prompt-Injection, 680 test examples, five attack formats
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined by the benchmark
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '680'
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_phi_act_af
  defence_id: act_phi
  benchmark: AlpacaFarm, 208 examples, 'Print exactly Hacked!', four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response starts with 'Hacked'
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '208'
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_phi_act_sep
  defence_id: act_phi
  benchmark: SEP, injection at start, middle or end
  attacker_type: static
  attacker_access: black_box
  judge: SecAlign's SEP protocol
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_phi_act_opiclean
  defence_id: act_phi
  benchmark: Open-Prompt-Injection clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: task accuracy
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_phi_act_mmlu
  defence_id: act_phi
  benchmark: MMLU
  attacker_type: none
  attacker_access: not_applicable
  judge: answer key
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_phi_bct_opi
  defence_id: bct_phi
  benchmark: Open-Prompt-Injection, 680 test examples, five attack formats
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined by the benchmark
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '680'
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_phi_bct_af
  defence_id: bct_phi
  benchmark: AlpacaFarm, 208 examples, 'Print exactly Hacked!', four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response starts with 'Hacked'
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '208'
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_phi_bct_sep
  defence_id: bct_phi
  benchmark: SEP, injection at start, middle or end
  attacker_type: static
  attacker_access: black_box
  judge: SecAlign's SEP protocol
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_phi_bct_opiclean
  defence_id: bct_phi
  benchmark: Open-Prompt-Injection clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: task accuracy
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_phi_bct_mmlu
  defence_id: bct_phi
  benchmark: MMLU
  attacker_type: none
  attacker_access: not_applicable
  judge: answer key
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_oss_none_opi
  defence_id: none:GPT-OSS-20B
  benchmark: Open-Prompt-Injection, 680 test examples, five attack formats
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined by the benchmark
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '680'
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_oss_none_af
  defence_id: none:GPT-OSS-20B
  benchmark: AlpacaFarm, 208 examples, 'Print exactly Hacked!', four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response starts with 'Hacked'
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '208'
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_oss_none_sep
  defence_id: none:GPT-OSS-20B
  benchmark: SEP, injection at start, middle or end
  attacker_type: static
  attacker_access: black_box
  judge: SecAlign's SEP protocol
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_oss_none_opiclean
  defence_id: none:GPT-OSS-20B
  benchmark: Open-Prompt-Injection clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: task accuracy
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_oss_none_mmlu
  defence_id: none:GPT-OSS-20B
  benchmark: MMLU
  attacker_type: none
  attacker_access: not_applicable
  judge: answer key
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_oss_act_opi
  defence_id: act_oss
  benchmark: Open-Prompt-Injection, 680 test examples, five attack formats
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined by the benchmark
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '680'
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_oss_act_af
  defence_id: act_oss
  benchmark: AlpacaFarm, 208 examples, 'Print exactly Hacked!', four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response starts with 'Hacked'
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '208'
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_oss_act_sep
  defence_id: act_oss
  benchmark: SEP, injection at start, middle or end
  attacker_type: static
  attacker_access: black_box
  judge: SecAlign's SEP protocol
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_oss_act_opiclean
  defence_id: act_oss
  benchmark: Open-Prompt-Injection clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: task accuracy
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_oss_act_mmlu
  defence_id: act_oss
  benchmark: MMLU
  attacker_type: none
  attacker_access: not_applicable
  judge: answer key
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_oss_bct_opi
  defence_id: bct_oss
  benchmark: Open-Prompt-Injection, 680 test examples, five attack formats
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined by the benchmark
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '680'
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_oss_bct_af
  defence_id: bct_oss
  benchmark: AlpacaFarm, 208 examples, 'Print exactly Hacked!', four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response starts with 'Hacked'
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '208'
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_oss_bct_sep
  defence_id: bct_oss
  benchmark: SEP, injection at start, middle or end
  attacker_type: static
  attacker_access: black_box
  judge: SecAlign's SEP protocol
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_oss_bct_opiclean
  defence_id: bct_oss
  benchmark: Open-Prompt-Injection clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: task accuracy
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_oss_bct_mmlu
  defence_id: bct_oss
  benchmark: MMLU
  attacker_type: none
  attacker_access: not_applicable
  judge: answer key
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_gem_none_opi
  defence_id: none:Gemma-4-E4B-it
  benchmark: Open-Prompt-Injection, 680 test examples, five attack formats
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined by the benchmark
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '680'
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_gem_none_af
  defence_id: none:Gemma-4-E4B-it
  benchmark: AlpacaFarm, 208 examples, 'Print exactly Hacked!', four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response starts with 'Hacked'
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '208'
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_gem_none_sep
  defence_id: none:Gemma-4-E4B-it
  benchmark: SEP, injection at start, middle or end
  attacker_type: static
  attacker_access: black_box
  judge: SecAlign's SEP protocol
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_gem_none_opiclean
  defence_id: none:Gemma-4-E4B-it
  benchmark: Open-Prompt-Injection clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: task accuracy
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_gem_none_mmlu
  defence_id: none:Gemma-4-E4B-it
  benchmark: MMLU
  attacker_type: none
  attacker_access: not_applicable
  judge: answer key
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_s_gem_act_opi
  defence_id: act_gem
  benchmark: Open-Prompt-Injection, 680 test examples, five attack formats
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined by the benchmark
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '680'
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_gem_act_af
  defence_id: act_gem
  benchmark: AlpacaFarm, 208 examples, 'Print exactly Hacked!', four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response starts with 'Hacked'
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '208'
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_gem_act_sep
  defence_id: act_gem
  benchmark: SEP, injection at start, middle or end
  attacker_type: static
  attacker_access: black_box
  judge: SecAlign's SEP protocol
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_gem_act_opiclean
  defence_id: act_gem
  benchmark: Open-Prompt-Injection clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: task accuracy
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_gem_act_mmlu
  defence_id: act_gem
  benchmark: MMLU
  attacker_type: none
  attacker_access: not_applicable
  judge: answer key
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_gem_bct_opi
  defence_id: bct_gem
  benchmark: Open-Prompt-Injection, 680 test examples, five attack formats
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined by the benchmark
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '680'
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_gem_bct_af
  defence_id: bct_gem
  benchmark: AlpacaFarm, 208 examples, 'Print exactly Hacked!', four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response starts with 'Hacked'
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '208'
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_gem_bct_sep
  defence_id: bct_gem
  benchmark: SEP, injection at start, middle or end
  attacker_type: static
  attacker_access: black_box
  judge: SecAlign's SEP protocol
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_gem_bct_opiclean
  defence_id: bct_gem
  benchmark: Open-Prompt-Injection clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: task accuracy
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_s_gem_bct_mmlu
  defence_id: bct_gem
  benchmark: MMLU
  attacker_type: none
  attacker_access: not_applicable
  judge: answer key
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 17
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  pages:
    benchmark: 5
    decoding: 24
- id: ev_a_q17_none
  defence_id: none:Qwen3-1.7B
  benchmark: AlpacaFarm test split (n = 108), suffixes from a GRPO attacker trained against this target, best of 8
  attacker_type: rl
  attacker_access: black_box
  judge: response starts with 'Hacked!'; user task completed
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 18
  attacker_budget: 8 suffixes per example
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '108'
  pages:
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_a_q17_act
  defence_id: act_q17
  benchmark: AlpacaFarm test split (n = 108), suffixes from a GRPO attacker trained against this target, best of 8
  attacker_type: rl
  attacker_access: black_box
  judge: response starts with 'Hacked!'; user task completed
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 18
  attacker_budget: 8 suffixes per example
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '108'
  pages:
    decoding: 24
- id: ev_a_q17_bct
  defence_id: bct_q17
  benchmark: AlpacaFarm test split (n = 108), suffixes from a GRPO attacker trained against this target, best of 8
  attacker_type: rl
  attacker_access: black_box
  judge: response starts with 'Hacked!'; user task completed
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 18
  attacker_budget: 8 suffixes per example
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '108'
  pages:
    decoding: 24
- id: ev_a_q8_none
  defence_id: none:Qwen3-8B
  benchmark: AlpacaFarm test split (n = 108), suffixes from a GRPO attacker trained against this target, best of 8
  attacker_type: rl
  attacker_access: black_box
  judge: response starts with 'Hacked!'; user task completed
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 18
  attacker_budget: 8 suffixes per example
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '108'
  pages:
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_a_q8_act
  defence_id: act_q8
  benchmark: AlpacaFarm test split (n = 108), suffixes from a GRPO attacker trained against this target, best of 8
  attacker_type: rl
  attacker_access: black_box
  judge: response starts with 'Hacked!'; user task completed
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 18
  attacker_budget: 8 suffixes per example
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '108'
  pages:
    decoding: 24
- id: ev_a_q8_bct
  defence_id: bct_q8
  benchmark: AlpacaFarm test split (n = 108), suffixes from a GRPO attacker trained against this target, best of 8
  attacker_type: rl
  attacker_access: black_box
  judge: response starts with 'Hacked!'; user task completed
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 18
  attacker_budget: 8 suffixes per example
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '108'
  pages:
    decoding: 24
- id: ev_a_phi_none
  defence_id: none:Phi-4-reasoning
  benchmark: AlpacaFarm test split (n = 108), suffixes from a GRPO attacker trained against this target, best of 8
  attacker_type: rl
  attacker_access: black_box
  judge: response starts with 'Hacked!'; user task completed
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 18
  attacker_budget: 8 suffixes per example
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '108'
  pages:
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_a_phi_act
  defence_id: act_phi
  benchmark: AlpacaFarm test split (n = 108), suffixes from a GRPO attacker trained against this target, best of 8
  attacker_type: rl
  attacker_access: black_box
  judge: response starts with 'Hacked!'; user task completed
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 18
  attacker_budget: 8 suffixes per example
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '108'
  pages:
    decoding: 24
- id: ev_a_phi_bct
  defence_id: bct_phi
  benchmark: AlpacaFarm test split (n = 108), suffixes from a GRPO attacker trained against this target, best of 8
  attacker_type: rl
  attacker_access: black_box
  judge: response starts with 'Hacked!'; user task completed
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 18
  attacker_budget: 8 suffixes per example
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '108'
  pages:
    decoding: 24
- id: ev_a_gem_none
  defence_id: none:Gemma-4-E4B-it
  benchmark: AlpacaFarm test split (n = 108), suffixes from a GRPO attacker trained against this target, best of 8
  attacker_type: rl
  attacker_access: black_box
  judge: response starts with 'Hacked!'; user task completed
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 18
  attacker_budget: 8 suffixes per example
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '108'
  pages:
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_a_gem_act
  defence_id: act_gem
  benchmark: AlpacaFarm test split (n = 108), suffixes from a GRPO attacker trained against this target, best of 8
  attacker_type: rl
  attacker_access: black_box
  judge: response starts with 'Hacked!'; user task completed
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 18
  attacker_budget: 8 suffixes per example
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '108'
  pages:
    decoding: 24
- id: ev_a_gem_bct
  defence_id: bct_gem
  benchmark: AlpacaFarm test split (n = 108), suffixes from a GRPO attacker trained against this target, best of 8
  attacker_type: rl
  attacker_access: black_box
  judge: response starts with 'Hacked!'; user task completed
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 18
  attacker_budget: 8 suffixes per example
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '108'
  pages:
    decoding: 24
- id: ev_a_oss_none
  defence_id: none:GPT-OSS-20B
  benchmark: AlpacaFarm test split (n = 108), suffixes from a GRPO attacker trained against this target, best of 8
  attacker_type: rl
  attacker_access: black_box
  judge: response starts with 'Hacked!'; user task completed
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 18
  attacker_budget: 8 suffixes per example
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '108'
  pages:
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_a_oss_act
  defence_id: act_oss
  benchmark: AlpacaFarm test split (n = 108), suffixes from a GRPO attacker trained against this target, best of 8
  attacker_type: rl
  attacker_access: black_box
  judge: response starts with 'Hacked!'; user task completed
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 18
  attacker_budget: 8 suffixes per example
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '108'
  pages:
    decoding: 24
- id: ev_a_oss_bct
  defence_id: bct_oss
  benchmark: AlpacaFarm test split (n = 108), suffixes from a GRPO attacker trained against this target, best of 8
  attacker_type: rl
  attacker_access: black_box
  judge: response starts with 'Hacked!'; user task completed
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 18
  attacker_budget: 8 suffixes per example
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '108'
  pages:
    decoding: 24
- id: ev_t4_none_opi
  defence_id: none:Qwen3-8B
  benchmark: Open-Prompt-Injection, 680 test examples, five attack formats
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined by the benchmark
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 19
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '680'
  run: Table 4 ablation run
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_t4_none_af
  defence_id: none:Qwen3-8B
  benchmark: AlpacaFarm, 208 examples, 'Print exactly Hacked!', four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response starts with 'Hacked'
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 19
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '208'
  run: Table 4 ablation run
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_t4_none_sep
  defence_id: none:Qwen3-8B
  benchmark: SEP, injection at start, middle or end
  attacker_type: static
  attacker_access: black_box
  judge: SecAlign's SEP protocol
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 19
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  run: Table 4 ablation run
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_t4_none_opiclean
  defence_id: none:Qwen3-8B
  benchmark: Open-Prompt-Injection clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: task accuracy
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 19
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  run: Table 4 ablation run
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_t4_none_mmlu
  defence_id: none:Qwen3-8B
  benchmark: MMLU
  attacker_type: none
  attacker_access: not_applicable
  judge: answer key
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 19
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  run: Table 4 ablation run
  pages:
    benchmark: 5
    decoding: 24
  baseline_kind: off_the_shelf
- id: ev_t4_secalign_opi
  defence_id: secalign_q8
  benchmark: Open-Prompt-Injection, 680 test examples, five attack formats
  attacker_type: static
  attacker_access: black_box
  judge: attack success as defined by the benchmark
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 19
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '680'
  run: Table 4 ablation run
  pages:
    benchmark: 5
    decoding: 24
- id: ev_t4_secalign_af
  defence_id: secalign_q8
  benchmark: AlpacaFarm, 208 examples, 'Print exactly Hacked!', four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response starts with 'Hacked'
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 19
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  sample_size: '208'
  run: Table 4 ablation run
  pages:
    benchmark: 5
    decoding: 24
- id: ev_t4_secalign_sep
  defence_id: secalign_q8
  benchmark: SEP, injection at start, middle or end
  attacker_type: static
  attacker_access: black_box
  judge: SecAlign's SEP protocol
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 19
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  run: Table 4 ablation run
  pages:
    benchmark: 5
    decoding: 24
- id: ev_t4_secalign_opiclean
  defence_id: secalign_q8
  benchmark: Open-Prompt-Injection clean tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: task accuracy
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 19
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  run: Table 4 ablation run
  pages:
    benchmark: 5
    decoding: 24
- id: ev_t4_secalign_mmlu
  defence_id: secalign_q8
  benchmark: MMLU
  attacker_type: none
  attacker_access: not_applicable
  judge: answer key
  layout_same_as_defence: 'yes'
  measured_by: actconsistency_arxiv2026
  page: 19
  decoding: greedy, except Phi-4-reasoning (temperature 0.8, top-k 50, top-p 0.95)
  run: Table 4 ablation run
  pages:
    benchmark: 5
    decoding: 24
results:
- id: r_s_q17_none_opi
  evaluation_id: ev_s_q17_none_opi
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (OPI)
  value: '47.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q17_none_af
  evaluation_id: ev_s_q17_none_af
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (AlpacaFarm)
  value: '50.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q17_none_sep
  evaluation_id: ev_s_q17_none_sep
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (SEP)
  value: '61.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q17_none_opiclean
  evaluation_id: ev_s_q17_none_opiclean
  quantity: R2clean
  coverage: direct
  metric_as_printed: OPI clean
  value: '80.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q17_none_mmlu
  evaluation_id: ev_s_q17_none_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '60.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q17_act_opi
  evaluation_id: ev_s_q17_act_opi
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (OPI)
  value: '0.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q17_act_af
  evaluation_id: ev_s_q17_act_af
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (AlpacaFarm)
  value: '0.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q17_act_sep
  evaluation_id: ev_s_q17_act_sep
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (SEP)
  value: '2.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q17_act_opiclean
  evaluation_id: ev_s_q17_act_opiclean
  quantity: R2clean
  coverage: direct
  metric_as_printed: OPI clean
  value: '68.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q17_act_mmlu
  evaluation_id: ev_s_q17_act_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '58.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q17_bct_opi
  evaluation_id: ev_s_q17_bct_opi
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (OPI)
  value: '0.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q17_bct_af
  evaluation_id: ev_s_q17_bct_af
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (AlpacaFarm)
  value: '1.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q17_bct_sep
  evaluation_id: ev_s_q17_bct_sep
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (SEP)
  value: '4.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q17_bct_opiclean
  evaluation_id: ev_s_q17_bct_opiclean
  quantity: R2clean
  coverage: direct
  metric_as_printed: OPI clean
  value: '78.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q17_bct_mmlu
  evaluation_id: ev_s_q17_bct_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '59.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q8_none_opi
  evaluation_id: ev_s_q8_none_opi
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (OPI)
  value: '45.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q8_none_af
  evaluation_id: ev_s_q8_none_af
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (AlpacaFarm)
  value: '64.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q8_none_sep
  evaluation_id: ev_s_q8_none_sep
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (SEP)
  value: '73.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q8_none_opiclean
  evaluation_id: ev_s_q8_none_opiclean
  quantity: R2clean
  coverage: direct
  metric_as_printed: OPI clean
  value: '81.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q8_none_mmlu
  evaluation_id: ev_s_q8_none_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '80.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q8_act_opi
  evaluation_id: ev_s_q8_act_opi
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (OPI)
  value: '0.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q8_act_af
  evaluation_id: ev_s_q8_act_af
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (AlpacaFarm)
  value: '0.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q8_act_sep
  evaluation_id: ev_s_q8_act_sep
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (SEP)
  value: '2.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q8_act_opiclean
  evaluation_id: ev_s_q8_act_opiclean
  quantity: R2clean
  coverage: direct
  metric_as_printed: OPI clean
  value: '80.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q8_act_mmlu
  evaluation_id: ev_s_q8_act_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '76.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q8_bct_opi
  evaluation_id: ev_s_q8_bct_opi
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (OPI)
  value: '0.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q8_bct_af
  evaluation_id: ev_s_q8_bct_af
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (AlpacaFarm)
  value: '2.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q8_bct_sep
  evaluation_id: ev_s_q8_bct_sep
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (SEP)
  value: '4.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q8_bct_opiclean
  evaluation_id: ev_s_q8_bct_opiclean
  quantity: R2clean
  coverage: direct
  metric_as_printed: OPI clean
  value: '78.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_q8_bct_mmlu
  evaluation_id: ev_s_q8_bct_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '78.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_phi_none_opi
  evaluation_id: ev_s_phi_none_opi
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (OPI)
  value: '56.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_phi_none_af
  evaluation_id: ev_s_phi_none_af
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (AlpacaFarm)
  value: '90.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_phi_none_sep
  evaluation_id: ev_s_phi_none_sep
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (SEP)
  value: '96.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_phi_none_opiclean
  evaluation_id: ev_s_phi_none_opiclean
  quantity: R2clean
  coverage: direct
  metric_as_printed: OPI clean
  value: '69.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_phi_none_mmlu
  evaluation_id: ev_s_phi_none_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '75.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_phi_act_opi
  evaluation_id: ev_s_phi_act_opi
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (OPI)
  value: '16.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_phi_act_af
  evaluation_id: ev_s_phi_act_af
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (AlpacaFarm)
  value: '34.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_phi_act_sep
  evaluation_id: ev_s_phi_act_sep
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (SEP)
  value: '31.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_phi_act_opiclean
  evaluation_id: ev_s_phi_act_opiclean
  quantity: R2clean
  coverage: direct
  metric_as_printed: OPI clean
  value: '66.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_phi_act_mmlu
  evaluation_id: ev_s_phi_act_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '73.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_phi_bct_opi
  evaluation_id: ev_s_phi_bct_opi
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (OPI)
  value: '13.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_phi_bct_af
  evaluation_id: ev_s_phi_bct_af
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (AlpacaFarm)
  value: '17.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_phi_bct_sep
  evaluation_id: ev_s_phi_bct_sep
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (SEP)
  value: '45.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_phi_bct_opiclean
  evaluation_id: ev_s_phi_bct_opiclean
  quantity: R2clean
  coverage: direct
  metric_as_printed: OPI clean
  value: '6.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_phi_bct_mmlu
  evaluation_id: ev_s_phi_bct_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '58.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_oss_none_opi
  evaluation_id: ev_s_oss_none_opi
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (OPI)
  value: '41.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_oss_none_af
  evaluation_id: ev_s_oss_none_af
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (AlpacaFarm)
  value: '80.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_oss_none_sep
  evaluation_id: ev_s_oss_none_sep
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (SEP)
  value: '73.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_oss_none_opiclean
  evaluation_id: ev_s_oss_none_opiclean
  quantity: R2clean
  coverage: direct
  metric_as_printed: OPI clean
  value: '88.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_oss_none_mmlu
  evaluation_id: ev_s_oss_none_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '83.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_oss_act_opi
  evaluation_id: ev_s_oss_act_opi
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (OPI)
  value: '29.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_oss_act_af
  evaluation_id: ev_s_oss_act_af
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (AlpacaFarm)
  value: '39.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_oss_act_sep
  evaluation_id: ev_s_oss_act_sep
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (SEP)
  value: '21.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_oss_act_opiclean
  evaluation_id: ev_s_oss_act_opiclean
  quantity: R2clean
  coverage: direct
  metric_as_printed: OPI clean
  value: '72.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_oss_act_mmlu
  evaluation_id: ev_s_oss_act_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '74.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_oss_bct_opi
  evaluation_id: ev_s_oss_bct_opi
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (OPI)
  value: '4.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_oss_bct_af
  evaluation_id: ev_s_oss_bct_af
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (AlpacaFarm)
  value: '27.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_oss_bct_sep
  evaluation_id: ev_s_oss_bct_sep
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (SEP)
  value: '14.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_oss_bct_opiclean
  evaluation_id: ev_s_oss_bct_opiclean
  quantity: R2clean
  coverage: direct
  metric_as_printed: OPI clean
  value: '86.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_oss_bct_mmlu
  evaluation_id: ev_s_oss_bct_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '72.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_gem_none_opi
  evaluation_id: ev_s_gem_none_opi
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (OPI)
  value: '24.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_gem_none_af
  evaluation_id: ev_s_gem_none_af
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (AlpacaFarm)
  value: '76.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_gem_none_sep
  evaluation_id: ev_s_gem_none_sep
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (SEP)
  value: '73.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_gem_none_opiclean
  evaluation_id: ev_s_gem_none_opiclean
  quantity: R2clean
  coverage: direct
  metric_as_printed: OPI clean
  value: '90.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_gem_none_mmlu
  evaluation_id: ev_s_gem_none_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '69.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_gem_act_opi
  evaluation_id: ev_s_gem_act_opi
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (OPI)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_gem_act_af
  evaluation_id: ev_s_gem_act_af
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (AlpacaFarm)
  value: '2.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_gem_act_sep
  evaluation_id: ev_s_gem_act_sep
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (SEP)
  value: '3.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_gem_act_opiclean
  evaluation_id: ev_s_gem_act_opiclean
  quantity: R2clean
  coverage: direct
  metric_as_printed: OPI clean
  value: '89.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_gem_act_mmlu
  evaluation_id: ev_s_gem_act_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '67.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_gem_bct_opi
  evaluation_id: ev_s_gem_bct_opi
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (OPI)
  value: '0.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_gem_bct_af
  evaluation_id: ev_s_gem_bct_af
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (AlpacaFarm)
  value: '3.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_gem_bct_sep
  evaluation_id: ev_s_gem_bct_sep
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (SEP)
  value: '7.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_gem_bct_opiclean
  evaluation_id: ev_s_gem_bct_opiclean
  quantity: R2clean
  coverage: direct
  metric_as_printed: OPI clean
  value: '89.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_s_gem_bct_mmlu
  evaluation_id: ev_s_gem_bct_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '68.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 17
  location: Table 1
- id: r_a_q17_none_asr
  evaluation_id: ev_a_q17_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASRany (test)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [96.6, 100.0]
- id: r_a_q17_none_clean
  evaluation_id: ev_a_q17_none
  quantity: R3safe
  coverage: direct
  metric_as_printed: CleanTaskany (test)
  value: '44.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [35.4, 53.8]
- id: r_a_q17_act_asr
  evaluation_id: ev_a_q17_act
  quantity: R1
  coverage: direct
  metric_as_printed: ASRany (test)
  value: '63.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [53.6, 71.5]
- id: r_a_q17_act_clean
  evaluation_id: ev_a_q17_act
  quantity: R3safe
  coverage: direct
  metric_as_printed: CleanTaskany (test)
  value: '67.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [58.3, 75.7]
- id: r_a_q17_bct_asr
  evaluation_id: ev_a_q17_bct
  quantity: R1
  coverage: direct
  metric_as_printed: ASRany (test)
  value: '80.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [72.1, 86.9]
- id: r_a_q17_bct_clean
  evaluation_id: ev_a_q17_bct
  quantity: R3safe
  coverage: direct
  metric_as_printed: CleanTaskany (test)
  value: '39.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [31.1, 49.2]
- id: r_a_q8_none_asr
  evaluation_id: ev_a_q8_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASRany (test)
  value: '96.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [90.9, 98.6]
- id: r_a_q8_none_clean
  evaluation_id: ev_a_q8_none
  quantity: R3safe
  coverage: direct
  metric_as_printed: CleanTaskany (test)
  value: '47.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [38.1, 56.6]
- id: r_a_q8_act_asr
  evaluation_id: ev_a_q8_act
  quantity: R1
  coverage: direct
  metric_as_printed: ASRany (test)
  value: '59.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [49.8, 68.1]
- id: r_a_q8_act_clean
  evaluation_id: ev_a_q8_act
  quantity: R3safe
  coverage: direct
  metric_as_printed: CleanTaskany (test)
  value: '56.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [47.1, 65.5]
- id: r_a_q8_bct_asr
  evaluation_id: ev_a_q8_bct
  quantity: R1
  coverage: direct
  metric_as_printed: ASRany (test)
  value: '94.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [88.4, 97.4]
- id: r_a_q8_bct_clean
  evaluation_id: ev_a_q8_bct
  quantity: R3safe
  coverage: direct
  metric_as_printed: CleanTaskany (test)
  value: '52.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [43.4, 61.9]
- id: r_a_phi_none_asr
  evaluation_id: ev_a_phi_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASRany (test)
  value: '90.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [81.5, 99.8]
- id: r_a_phi_none_clean
  evaluation_id: ev_a_phi_none
  quantity: R3safe
  coverage: direct
  metric_as_printed: CleanTaskany (test)
  value: '2.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [0.9, 7.9]
- id: r_a_phi_act_asr
  evaluation_id: ev_a_phi_act
  quantity: R1
  coverage: direct
  metric_as_printed: ASRany (test)
  value: '49.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [42.1, 59.1]
- id: r_a_phi_act_clean
  evaluation_id: ev_a_phi_act
  quantity: R3safe
  coverage: direct
  metric_as_printed: CleanTaskany (test)
  value: '47.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [38.1, 56.6]
- id: r_a_phi_bct_asr
  evaluation_id: ev_a_phi_bct
  quantity: R1
  coverage: direct
  metric_as_printed: ASRany (test)
  value: '41.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [32.8, 51.1]
- id: r_a_phi_bct_clean
  evaluation_id: ev_a_phi_bct
  quantity: R3safe
  coverage: direct
  metric_as_printed: CleanTaskany (test)
  value: '22.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [15.4, 30.9]
- id: r_a_gem_none_asr
  evaluation_id: ev_a_gem_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASRany (test)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [96.6, 100.0]
- id: r_a_gem_none_clean
  evaluation_id: ev_a_gem_none
  quantity: R3safe
  coverage: direct
  metric_as_printed: CleanTaskany (test)
  value: '33.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [25.2, 42.7]
- id: r_a_gem_act_asr
  evaluation_id: ev_a_gem_act
  quantity: R1
  coverage: direct
  metric_as_printed: ASRany (test)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [96.6, 100.0]
- id: r_a_gem_act_clean
  evaluation_id: ev_a_gem_act
  quantity: R3safe
  coverage: direct
  metric_as_printed: CleanTaskany (test)
  value: '37.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [28.5, 46.4]
- id: r_a_gem_bct_asr
  evaluation_id: ev_a_gem_bct
  quantity: R1
  coverage: direct
  metric_as_printed: ASRany (test)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [96.6, 100.0]
- id: r_a_gem_bct_clean
  evaluation_id: ev_a_gem_bct
  quantity: R3safe
  coverage: direct
  metric_as_printed: CleanTaskany (test)
  value: '36.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [27.7, 45.5]
- id: r_a_oss_none_asr
  evaluation_id: ev_a_oss_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASRany (test)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [96.6, 100.0]
- id: r_a_oss_none_clean
  evaluation_id: ev_a_oss_none
  quantity: R3safe
  coverage: direct
  metric_as_printed: CleanTaskany (test)
  value: '39.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [31.1, 49.2]
- id: r_a_oss_act_asr
  evaluation_id: ev_a_oss_act
  quantity: R1
  coverage: direct
  metric_as_printed: ASRany (test)
  value: '37.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [28.5, 46.4]
- id: r_a_oss_act_clean
  evaluation_id: ev_a_oss_act
  quantity: R3safe
  coverage: direct
  metric_as_printed: CleanTaskany (test)
  value: '39.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [31.1, 49.2]
- id: r_a_oss_bct_asr
  evaluation_id: ev_a_oss_bct
  quantity: R1
  coverage: direct
  metric_as_printed: ASRany (test)
  value: '57.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [48.0, 66.3]
- id: r_a_oss_bct_clean
  evaluation_id: ev_a_oss_bct
  quantity: R3safe
  coverage: direct
  metric_as_printed: CleanTaskany (test)
  value: '68.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 18
  location: Table 3
  spread: Wilson 95% interval [59.3, 76.5]
- id: r_t4_none_opi
  evaluation_id: ev_t4_none_opi
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (OPI)
  value: '45.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 4
- id: r_t4_none_af
  evaluation_id: ev_t4_none_af
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (AlpacaFarm)
  value: '63.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 4
- id: r_t4_none_sep
  evaluation_id: ev_t4_none_sep
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (SEP)
  value: '72.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 4
- id: r_t4_none_opiclean
  evaluation_id: ev_t4_none_opiclean
  quantity: R2clean
  coverage: direct
  metric_as_printed: OPI clean
  value: '80.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 4
- id: r_t4_none_mmlu
  evaluation_id: ev_t4_none_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '80.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 4
- id: r_t4_secalign_opi
  evaluation_id: ev_t4_secalign_opi
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (OPI)
  value: '21.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 4
- id: r_t4_secalign_af
  evaluation_id: ev_t4_secalign_af
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (AlpacaFarm)
  value: '56.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 4
- id: r_t4_secalign_sep
  evaluation_id: ev_t4_secalign_sep
  quantity: R1
  coverage: direct
  metric_as_printed: Mean ASR (SEP)
  value: '49.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 4
- id: r_t4_secalign_opiclean
  evaluation_id: ev_t4_secalign_opiclean
  quantity: R2clean
  coverage: direct
  metric_as_printed: OPI clean
  value: '79.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 4
- id: r_t4_secalign_mmlu
  evaluation_id: ev_t4_secalign_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU
  value: '75.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 4
comparisons:
- id: c_s_q17_act_opi
  left: ev_s_q17_act_opi
  right: ev_s_q17_none_opi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_q17_act_af
  left: ev_s_q17_act_af
  right: ev_s_q17_none_af
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_q17_act_sep
  left: ev_s_q17_act_sep
  right: ev_s_q17_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_q17_act_opiclean
  left: ev_s_q17_act_opiclean
  right: ev_s_q17_none_opiclean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_q17_act_mmlu
  left: ev_s_q17_act_mmlu
  right: ev_s_q17_none_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_q17_bct_opi
  left: ev_s_q17_bct_opi
  right: ev_s_q17_none_opi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_q17_bct_af
  left: ev_s_q17_bct_af
  right: ev_s_q17_none_af
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_q17_bct_sep
  left: ev_s_q17_bct_sep
  right: ev_s_q17_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_q17_bct_opiclean
  left: ev_s_q17_bct_opiclean
  right: ev_s_q17_none_opiclean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_q17_bct_mmlu
  left: ev_s_q17_bct_mmlu
  right: ev_s_q17_none_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_q8_act_opi
  left: ev_s_q8_act_opi
  right: ev_s_q8_none_opi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_q8_act_af
  left: ev_s_q8_act_af
  right: ev_s_q8_none_af
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_q8_act_sep
  left: ev_s_q8_act_sep
  right: ev_s_q8_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_q8_act_opiclean
  left: ev_s_q8_act_opiclean
  right: ev_s_q8_none_opiclean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_q8_act_mmlu
  left: ev_s_q8_act_mmlu
  right: ev_s_q8_none_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_q8_bct_opi
  left: ev_s_q8_bct_opi
  right: ev_s_q8_none_opi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_q8_bct_af
  left: ev_s_q8_bct_af
  right: ev_s_q8_none_af
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_q8_bct_sep
  left: ev_s_q8_bct_sep
  right: ev_s_q8_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_q8_bct_opiclean
  left: ev_s_q8_bct_opiclean
  right: ev_s_q8_none_opiclean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_q8_bct_mmlu
  left: ev_s_q8_bct_mmlu
  right: ev_s_q8_none_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_phi_act_opi
  left: ev_s_phi_act_opi
  right: ev_s_phi_none_opi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_phi_act_af
  left: ev_s_phi_act_af
  right: ev_s_phi_none_af
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_phi_act_sep
  left: ev_s_phi_act_sep
  right: ev_s_phi_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_phi_act_opiclean
  left: ev_s_phi_act_opiclean
  right: ev_s_phi_none_opiclean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_phi_act_mmlu
  left: ev_s_phi_act_mmlu
  right: ev_s_phi_none_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_phi_bct_opi
  left: ev_s_phi_bct_opi
  right: ev_s_phi_none_opi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_phi_bct_af
  left: ev_s_phi_bct_af
  right: ev_s_phi_none_af
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_phi_bct_sep
  left: ev_s_phi_bct_sep
  right: ev_s_phi_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_phi_bct_opiclean
  left: ev_s_phi_bct_opiclean
  right: ev_s_phi_none_opiclean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_phi_bct_mmlu
  left: ev_s_phi_bct_mmlu
  right: ev_s_phi_none_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_oss_act_opi
  left: ev_s_oss_act_opi
  right: ev_s_oss_none_opi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_oss_act_af
  left: ev_s_oss_act_af
  right: ev_s_oss_none_af
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_oss_act_sep
  left: ev_s_oss_act_sep
  right: ev_s_oss_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_oss_act_opiclean
  left: ev_s_oss_act_opiclean
  right: ev_s_oss_none_opiclean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_oss_act_mmlu
  left: ev_s_oss_act_mmlu
  right: ev_s_oss_none_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_oss_bct_opi
  left: ev_s_oss_bct_opi
  right: ev_s_oss_none_opi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_oss_bct_af
  left: ev_s_oss_bct_af
  right: ev_s_oss_none_af
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_oss_bct_sep
  left: ev_s_oss_bct_sep
  right: ev_s_oss_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_oss_bct_opiclean
  left: ev_s_oss_bct_opiclean
  right: ev_s_oss_none_opiclean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_oss_bct_mmlu
  left: ev_s_oss_bct_mmlu
  right: ev_s_oss_none_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_gem_act_opi
  left: ev_s_gem_act_opi
  right: ev_s_gem_none_opi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_gem_act_af
  left: ev_s_gem_act_af
  right: ev_s_gem_none_af
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_gem_act_sep
  left: ev_s_gem_act_sep
  right: ev_s_gem_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_gem_act_opiclean
  left: ev_s_gem_act_opiclean
  right: ev_s_gem_none_opiclean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_gem_act_mmlu
  left: ev_s_gem_act_mmlu
  right: ev_s_gem_none_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_gem_bct_opi
  left: ev_s_gem_bct_opi
  right: ev_s_gem_none_opi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_gem_bct_af
  left: ev_s_gem_bct_af
  right: ev_s_gem_none_af
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_gem_bct_sep
  left: ev_s_gem_bct_sep
  right: ev_s_gem_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_gem_bct_opiclean
  left: ev_s_gem_bct_opiclean
  right: ev_s_gem_none_opiclean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_s_gem_bct_mmlu
  left: ev_s_gem_bct_mmlu
  right: ev_s_gem_none_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 17
- id: c_a_q17_act
  left: ev_a_q17_act
  right: ev_a_q17_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_a_q17_bct
  left: ev_a_q17_bct
  right: ev_a_q17_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_a_q8_act
  left: ev_a_q8_act
  right: ev_a_q8_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_a_q8_bct
  left: ev_a_q8_bct
  right: ev_a_q8_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_a_phi_act
  left: ev_a_phi_act
  right: ev_a_phi_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_a_phi_bct
  left: ev_a_phi_bct
  right: ev_a_phi_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_a_gem_act
  left: ev_a_gem_act
  right: ev_a_gem_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_a_gem_bct
  left: ev_a_gem_bct
  right: ev_a_gem_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_a_oss_act
  left: ev_a_oss_act
  right: ev_a_oss_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_a_oss_bct
  left: ev_a_oss_bct
  right: ev_a_oss_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 18
- id: c_t4_secalign_opi
  left: ev_t4_secalign_opi
  right: ev_t4_none_opi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 19
- id: c_t4_secalign_af
  left: ev_t4_secalign_af
  right: ev_t4_none_af
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 19
- id: c_t4_secalign_sep
  left: ev_t4_secalign_sep
  right: ev_t4_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 19
- id: c_t4_secalign_opiclean
  left: ev_t4_secalign_opiclean
  right: ev_t4_none_opiclean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 19
- id: c_t4_secalign_mmlu
  left: ev_t4_secalign_mmlu
  right: ev_t4_none_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 19
claims:
- id: k1
  text: ACT is competitive with other training-based prompt injection defences using only self-supervised pairs
  location: abstract
  anchor: find ACT to be competitive with other training-based defenses
  jobs:
  - R1
  - R3
  supported_by:
  - c_s_q17_act_opi
  - c_s_q17_act_af
  - c_s_q17_act_sep
  - c_s_q17_act_opiclean
  - c_s_q17_act_mmlu
  - c_s_q8_act_opi
  - c_s_q8_act_af
  - c_s_q8_act_sep
  - c_s_q8_act_opiclean
  - c_s_q8_act_mmlu
  - c_s_phi_act_opi
  - c_s_phi_act_af
  - c_s_phi_act_sep
  - c_s_phi_act_opiclean
  - c_s_phi_act_mmlu
  - c_s_oss_act_opi
  - c_s_oss_act_af
  - c_s_oss_act_sep
  - c_s_oss_act_opiclean
  - c_s_oss_act_mmlu
  - c_s_gem_act_opi
  - c_s_gem_act_af
  - c_s_gem_act_sep
  - c_s_gem_act_opiclean
  - c_s_gem_act_mmlu
  page: 1
  contradicted_by:
  - r_a_gem_act_asr
coverage:
  R1: direct
  R2clean: direct
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: direct
  where:
    R1: Tables 1, 3 and 4
    R2clean: Table 1 (OPI clean)
    R3safe: 'Table 3 (CleanTaskany: user task done and injection not emitted, best of 8)'
    general_utility: Tables 1 and 4 (MMLU)
schema_gaps:
- where: defences act_*
  note: ACT matches the frozen base model's hidden states on the clean prompt; axis 2 has no value for activation-level self-distillation, so it is coded distillation; signal dependence is not_applicable because no inference-time
    signal is used
- where: results r_a_phi_bct_*
  note: Table 2 gives Phi-4-reasoning BCT 42.6 (test) and 42.4 (train) while Table 3 gives 41.7 and 42.0 for the same cells; the record codes Table 3 (pp. 17, 18)
- where: results r_a_*
  note: ASRany and CleanTaskany count an example once any of 8 attacker suffixes succeeds or once any attempt is clean, so both are best-of-8 rates

`dynppa_arxiv2026` (2605.30534)

In [ ]:
%%writefile reports/records/dynppa_arxiv2026.yaml
report:
  id: dynppa_arxiv2026
  slug: '2605.30534'
  title: 'Strengthening Polymorphic Prompt Assembling: Dynamic Separator Generation Against Emerging Prompt Injection Attacks'
  first_author: Nima Dorzhiev
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2605.30534'
  version_read: arXiv v1
  first_date: '2026-05-28'
  read: F
  role: proposes
  code_release: extends the open-source PPA SDK; not checked
  extraction_scope: 'whole report (5 pages): Tables I and II (static against dynamic separators) and the benign-quality scores in Section IV-E; the abstract claim (Table III, which scores detectors, and the latency figures
    not coded)'
defences:
- id: ppa_l33
  name: Polymorphic Prompt Assembling, static separator pool
  origin: other_report
  base_checkpoint: Llama-3.3-70B-Instruct-Turbo (Together AI)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 1
  components:
  - separator pair drawn from a fixed pool of 84 separators, EIBD system prompt format
  anchor: randomly selecting separator pairs from a fixed pool
  pages:
    components: 2
- id: dyn_l33
  name: Polymorphic Prompt Assembling with dynamic separators
  origin: this_report
  base_checkpoint: Llama-3.3-70B-Instruct-Turbo (Together AI)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 2
  components:
  - a new BEGIN/END separator pair for every request from SHA-256 digests of timestamp, session id and nonce
  - EIBD system prompt format, as in PPA
  anchor: Each request generates an independent (BEGIN, END)
- id: ppa_ds
  name: Polymorphic Prompt Assembling, static separator pool
  origin: other_report
  base_checkpoint: DeepSeek-V4-Flash
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 1
  components:
  - separator pair drawn from a fixed pool of 84 separators, EIBD system prompt format
  anchor: randomly selecting separator pairs from a fixed pool
  pages:
    components: 2
- id: dyn_ds
  name: Polymorphic Prompt Assembling with dynamic separators
  origin: this_report
  base_checkpoint: DeepSeek-V4-Flash
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 2
  components:
  - a new BEGIN/END separator pair for every request from SHA-256 digests of timestamp, session id and nonce
  - EIBD system prompt format, as in PPA
  anchor: Each request generates an independent (BEGIN, END)
evaluations:
- id: ev_l33_m1_static
  defence_id: ppa_l33
  benchmark: M1_injection (leetspeak with urgency, from Lupinacci et al.), 100 runs, seed 1337
  attacker_type: static
  attacker_access: black_box
  judge: success-marker string match
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 3
  decoding: not reported
  sample_size: '100'
- id: ev_l33_m1_dyn
  defence_id: dyn_l33
  benchmark: M1_injection (leetspeak with urgency, from Lupinacci et al.), 100 runs, seed 1337
  attacker_type: static
  attacker_access: black_box
  judge: success-marker string match
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 3
  decoding: not reported
  sample_size: '100'
- id: ev_l33_m2_static
  defence_id: ppa_l33
  benchmark: M2_injection (cognitive override, from Lupinacci et al.), 100 runs, seed 1337
  attacker_type: static
  attacker_access: black_box
  judge: success-marker string match
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 3
  decoding: not reported
  sample_size: '100'
- id: ev_l33_m2_dyn
  defence_id: dyn_l33
  benchmark: M2_injection (cognitive override, from Lupinacci et al.), 100 runs, seed 1337
  attacker_type: static
  attacker_access: black_box
  judge: success-marker string match
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 3
  decoding: not reported
  sample_size: '100'
- id: ev_ds_m1_static
  defence_id: ppa_ds
  benchmark: M1_injection (leetspeak with urgency, from Lupinacci et al.), 100 runs, seed 1337
  attacker_type: static
  attacker_access: black_box
  judge: success-marker string match
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 3
  decoding: not reported
  sample_size: '100'
- id: ev_ds_m1_dyn
  defence_id: dyn_ds
  benchmark: M1_injection (leetspeak with urgency, from Lupinacci et al.), 100 runs, seed 1337
  attacker_type: static
  attacker_access: black_box
  judge: success-marker string match
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 3
  decoding: not reported
  sample_size: '100'
- id: ev_l33_naive_injection_static
  defence_id: ppa_l33
  benchmark: payload naive_injection from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: success-marker string match
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_naive_injection_dyn
  defence_id: dyn_l33
  benchmark: payload naive_injection from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: success-marker string match
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_fake_completion_static
  defence_id: ppa_l33
  benchmark: payload fake_completion from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: success-marker string match
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_fake_completion_dyn
  defence_id: dyn_l33
  benchmark: payload fake_completion from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: success-marker string match
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_double_character_static
  defence_id: ppa_l33
  benchmark: payload double_character from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: success-marker string match
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_double_character_dyn
  defence_id: dyn_l33
  benchmark: payload double_character from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: success-marker string match
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_virtualization_static
  defence_id: ppa_l33
  benchmark: payload virtualization from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: success-marker string match
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_virtualization_dyn
  defence_id: dyn_l33
  benchmark: payload virtualization from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: success-marker string match
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_role_playing_static
  defence_id: ppa_l33
  benchmark: payload role_playing from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: success-marker string match
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_role_playing_dyn
  defence_id: dyn_l33
  benchmark: payload role_playing from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: success-marker string match
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_payload_splitting_static
  defence_id: ppa_l33
  benchmark: payload payload_splitting from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: success-marker string match
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_payload_splitting_dyn
  defence_id: dyn_l33
  benchmark: payload payload_splitting from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: success-marker string match
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_adversarial_suffix_static
  defence_id: ppa_l33
  benchmark: payload adversarial_suffix from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: success-marker string match
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_adversarial_suffix_dyn
  defence_id: dyn_l33
  benchmark: payload adversarial_suffix from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: success-marker string match
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_combined_attack_static
  defence_id: ppa_l33
  benchmark: payload combined_attack from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: success-marker string match
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_combined_attack_dyn
  defence_id: dyn_l33
  benchmark: payload combined_attack from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: success-marker string match
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_baseline_salad_static
  defence_id: ppa_l33
  benchmark: payload baseline_salad from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: Llama-3.1-8B-Instruct-Turbo rubric (task derailment)
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_baseline_salad_dyn
  defence_id: dyn_l33
  benchmark: payload baseline_salad from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: Llama-3.1-8B-Instruct-Turbo rubric (task derailment)
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_direct_override_salad_static
  defence_id: ppa_l33
  benchmark: payload direct_override_salad from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: Llama-3.1-8B-Instruct-Turbo rubric (task derailment)
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_direct_override_salad_dyn
  defence_id: dyn_l33
  benchmark: payload direct_override_salad from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: Llama-3.1-8B-Instruct-Turbo rubric (task derailment)
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_role_confusion_salad_static
  defence_id: ppa_l33
  benchmark: payload role_confusion_salad from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: Llama-3.1-8B-Instruct-Turbo rubric (task derailment)
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_role_confusion_salad_dyn
  defence_id: dyn_l33
  benchmark: payload role_confusion_salad from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: Llama-3.1-8B-Instruct-Turbo rubric (task derailment)
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_format_breakout_salad_static
  defence_id: ppa_l33
  benchmark: payload format_breakout_salad from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: Llama-3.1-8B-Instruct-Turbo rubric (task derailment)
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_format_breakout_salad_dyn
  defence_id: dyn_l33
  benchmark: payload format_breakout_salad from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: Llama-3.1-8B-Instruct-Turbo rubric (task derailment)
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_separator_echo_salad_static
  defence_id: ppa_l33
  benchmark: payload separator_echo_salad from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: Llama-3.1-8B-Instruct-Turbo rubric (task derailment)
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_l33_separator_echo_salad_dyn
  defence_id: dyn_l33
  benchmark: payload separator_echo_salad from the PPA attack registry, 30 runs
  attacker_type: static
  attacker_access: black_box
  judge: Llama-3.1-8B-Instruct-Turbo rubric (task derailment)
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '30'
- id: ev_ben_none
  defence_id: none:Llama-3.3-70B-Instruct-Turbo (Together AI)
  benchmark: 100 benign requests (summarisation, Q&A, translation, math, code explanation)
  attacker_type: none
  attacker_access: not_applicable
  judge: Llama-3.3-70B-Instruct-Turbo judge, 1 to 5
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '100'
  baseline_kind: off_the_shelf
- id: ev_ben_static
  defence_id: ppa_l33
  benchmark: 100 benign requests (summarisation, Q&A, translation, math, code explanation)
  attacker_type: none
  attacker_access: not_applicable
  judge: Llama-3.3-70B-Instruct-Turbo judge, 1 to 5
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '100'
- id: ev_ben_dyn
  defence_id: dyn_l33
  benchmark: 100 benign requests (summarisation, Q&A, translation, math, code explanation)
  attacker_type: none
  attacker_access: not_applicable
  judge: Llama-3.3-70B-Instruct-Turbo judge, 1 to 5
  layout_same_as_defence: 'yes'
  measured_by: dynppa_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '100'
results:
- id: r_l33_m1_static
  evaluation_id: ev_l33_m1_static
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.88'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table I
- id: r_l33_m1_dyn
  evaluation_id: ev_l33_m1_dyn
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.38'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table I
- id: r_l33_m2_static
  evaluation_id: ev_l33_m2_static
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.09'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table I
- id: r_l33_m2_dyn
  evaluation_id: ev_l33_m2_dyn
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table I
- id: r_ds_m1_static
  evaluation_id: ev_ds_m1_static
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.45'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table I
- id: r_ds_m1_dyn
  evaluation_id: ev_ds_m1_dyn
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.38'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table I
- id: r_l33_naive_injection_static_asr
  evaluation_id: ev_l33_naive_injection_static
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.100'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_naive_injection_static_lk
  evaluation_id: ev_l33_naive_injection_static
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_naive_injection_dyn_asr
  evaluation_id: ev_l33_naive_injection_dyn
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.067'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_naive_injection_dyn_lk
  evaluation_id: ev_l33_naive_injection_dyn
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_fake_completion_static_asr
  evaluation_id: ev_l33_fake_completion_static
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_fake_completion_static_lk
  evaluation_id: ev_l33_fake_completion_static
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_fake_completion_dyn_asr
  evaluation_id: ev_l33_fake_completion_dyn
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_fake_completion_dyn_lk
  evaluation_id: ev_l33_fake_completion_dyn
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_double_character_static_asr
  evaluation_id: ev_l33_double_character_static
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.133'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_double_character_static_lk
  evaluation_id: ev_l33_double_character_static
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_double_character_dyn_asr
  evaluation_id: ev_l33_double_character_dyn
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_double_character_dyn_lk
  evaluation_id: ev_l33_double_character_dyn
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_virtualization_static_asr
  evaluation_id: ev_l33_virtualization_static
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_virtualization_static_lk
  evaluation_id: ev_l33_virtualization_static
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_virtualization_dyn_asr
  evaluation_id: ev_l33_virtualization_dyn
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_virtualization_dyn_lk
  evaluation_id: ev_l33_virtualization_dyn
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_role_playing_static_asr
  evaluation_id: ev_l33_role_playing_static
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.100'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_role_playing_static_lk
  evaluation_id: ev_l33_role_playing_static
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_role_playing_dyn_asr
  evaluation_id: ev_l33_role_playing_dyn
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_role_playing_dyn_lk
  evaluation_id: ev_l33_role_playing_dyn
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_payload_splitting_static_asr
  evaluation_id: ev_l33_payload_splitting_static
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.867'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_payload_splitting_static_lk
  evaluation_id: ev_l33_payload_splitting_static
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_payload_splitting_dyn_asr
  evaluation_id: ev_l33_payload_splitting_dyn
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.133'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_payload_splitting_dyn_lk
  evaluation_id: ev_l33_payload_splitting_dyn
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_adversarial_suffix_static_asr
  evaluation_id: ev_l33_adversarial_suffix_static
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_adversarial_suffix_static_lk
  evaluation_id: ev_l33_adversarial_suffix_static
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_adversarial_suffix_dyn_asr
  evaluation_id: ev_l33_adversarial_suffix_dyn
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_adversarial_suffix_dyn_lk
  evaluation_id: ev_l33_adversarial_suffix_dyn
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_combined_attack_static_asr
  evaluation_id: ev_l33_combined_attack_static
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.400'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_combined_attack_static_lk
  evaluation_id: ev_l33_combined_attack_static
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_combined_attack_dyn_asr
  evaluation_id: ev_l33_combined_attack_dyn
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.200'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_combined_attack_dyn_lk
  evaluation_id: ev_l33_combined_attack_dyn
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_baseline_salad_static_asr
  evaluation_id: ev_l33_baseline_salad_static
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_baseline_salad_static_lk
  evaluation_id: ev_l33_baseline_salad_static
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_baseline_salad_dyn_asr
  evaluation_id: ev_l33_baseline_salad_dyn
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_baseline_salad_dyn_lk
  evaluation_id: ev_l33_baseline_salad_dyn
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_direct_override_salad_static_asr
  evaluation_id: ev_l33_direct_override_salad_static
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_direct_override_salad_static_lk
  evaluation_id: ev_l33_direct_override_salad_static
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_direct_override_salad_dyn_asr
  evaluation_id: ev_l33_direct_override_salad_dyn
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_direct_override_salad_dyn_lk
  evaluation_id: ev_l33_direct_override_salad_dyn
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_role_confusion_salad_static_asr
  evaluation_id: ev_l33_role_confusion_salad_static
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_role_confusion_salad_static_lk
  evaluation_id: ev_l33_role_confusion_salad_static
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_role_confusion_salad_dyn_asr
  evaluation_id: ev_l33_role_confusion_salad_dyn
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_role_confusion_salad_dyn_lk
  evaluation_id: ev_l33_role_confusion_salad_dyn
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_format_breakout_salad_static_asr
  evaluation_id: ev_l33_format_breakout_salad_static
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_format_breakout_salad_static_lk
  evaluation_id: ev_l33_format_breakout_salad_static
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '0.467'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_format_breakout_salad_dyn_asr
  evaluation_id: ev_l33_format_breakout_salad_dyn
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_format_breakout_salad_dyn_lk
  evaluation_id: ev_l33_format_breakout_salad_dyn
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_separator_echo_salad_static_asr
  evaluation_id: ev_l33_separator_echo_salad_static
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.067'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_separator_echo_salad_static_lk
  evaluation_id: ev_l33_separator_echo_salad_static
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '1.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_separator_echo_salad_dyn_asr
  evaluation_id: ev_l33_separator_echo_salad_dyn
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (S-ASR or D-ASR)
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_l33_separator_echo_salad_dyn_lk
  evaluation_id: ev_l33_separator_echo_salad_dyn
  quantity: other
  coverage: direct
  metric_as_printed: separator leak rate (S-Lk or D-Lk)
  value: '1.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table II
- id: r_ben_none
  evaluation_id: ev_ben_none
  quantity: R2clean
  coverage: proxy
  metric_as_printed: mean judge score
  value: '4.96'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 4
  location: Section IV-E text
- id: r_ben_static
  evaluation_id: ev_ben_static
  quantity: R2clean
  coverage: proxy
  metric_as_printed: mean judge score
  value: '4.98'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 4
  location: Section IV-E text
- id: r_ben_dyn
  evaluation_id: ev_ben_dyn
  quantity: R2clean
  coverage: proxy
  metric_as_printed: mean judge score
  value: '5.00'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 4
  location: Section IV-E text
comparisons:
- id: c_l33_m1
  left: ev_l33_m1_dyn
  right: ev_l33_m1_static
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: separators generated per request against a fixed pool
  page: 3
- id: c_l33_m2
  left: ev_l33_m2_dyn
  right: ev_l33_m2_static
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: separators generated per request against a fixed pool
  page: 3
- id: c_ds_m1
  left: ev_ds_m1_dyn
  right: ev_ds_m1_static
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: separators generated per request against a fixed pool
  page: 3
- id: c_l33_naive_injection
  left: ev_l33_naive_injection_dyn
  right: ev_l33_naive_injection_static
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: separators generated per request against a fixed pool
  page: 4
- id: c_l33_fake_completion
  left: ev_l33_fake_completion_dyn
  right: ev_l33_fake_completion_static
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: separators generated per request against a fixed pool
  page: 4
- id: c_l33_double_character
  left: ev_l33_double_character_dyn
  right: ev_l33_double_character_static
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: separators generated per request against a fixed pool
  page: 4
- id: c_l33_virtualization
  left: ev_l33_virtualization_dyn
  right: ev_l33_virtualization_static
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: separators generated per request against a fixed pool
  page: 4
- id: c_l33_role_playing
  left: ev_l33_role_playing_dyn
  right: ev_l33_role_playing_static
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: separators generated per request against a fixed pool
  page: 4
- id: c_l33_payload_splitting
  left: ev_l33_payload_splitting_dyn
  right: ev_l33_payload_splitting_static
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: separators generated per request against a fixed pool
  page: 4
- id: c_l33_adversarial_suffix
  left: ev_l33_adversarial_suffix_dyn
  right: ev_l33_adversarial_suffix_static
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: separators generated per request against a fixed pool
  page: 4
- id: c_l33_combined_attack
  left: ev_l33_combined_attack_dyn
  right: ev_l33_combined_attack_static
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: separators generated per request against a fixed pool
  page: 4
- id: c_l33_baseline_salad
  left: ev_l33_baseline_salad_dyn
  right: ev_l33_baseline_salad_static
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: separators generated per request against a fixed pool
  page: 4
- id: c_l33_direct_override_salad
  left: ev_l33_direct_override_salad_dyn
  right: ev_l33_direct_override_salad_static
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: separators generated per request against a fixed pool
  page: 4
- id: c_l33_role_confusion_salad
  left: ev_l33_role_confusion_salad_dyn
  right: ev_l33_role_confusion_salad_static
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: separators generated per request against a fixed pool
  page: 4
- id: c_l33_format_breakout_salad
  left: ev_l33_format_breakout_salad_dyn
  right: ev_l33_format_breakout_salad_static
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: separators generated per request against a fixed pool
  page: 4
- id: c_l33_separator_echo_salad
  left: ev_l33_separator_echo_salad_dyn
  right: ev_l33_separator_echo_salad_static
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: separators generated per request against a fixed pool
  page: 4
- id: c_ben_static
  left: ev_ben_static
  right: ev_ben_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_ben_dyn
  left: ev_ben_dyn
  right: ev_ben_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 4
claims:
- id: k1
  text: dynamic separators cut M1 attack success from 0.88 to 0.38
  location: abstract
  anchor: dynamic mode reduces the Attack Success Rate (ASR) from 0.88 to 0.38
  jobs:
  - R1
  supported_by:
  - c_l33_m1
  page: 1
  contradicted_by:
  - r_ds_m1_dyn
coverage:
  R1: direct
  R2clean: proxy
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables I and II (static and dynamic separators only; no undefended arm under attack)
    R2clean: Section IV-E (judge score on 100 benign requests)
schema_gaps:
- where: evaluations ev_*_static, ev_*_dyn
  note: attack results are reported for static and dynamic PPA only; there is no undefended model under the same attacks, so the comparisons vary only the separator generation

`gwdpo_arxiv2026` (2606.10860)

In [ ]:
%%writefile reports/records/gwdpo_arxiv2026.yaml
report:
  id: gwdpo_arxiv2026
  slug: '2606.10860'
  title: Training LLMs to Enforce Multi-Level Instruction Hierarchies via Gravity-Weighted Direct Preference Optimization
  first_author: Lena S. Bolliger
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2606.10860'
  version_read: arXiv v1
  first_date: '2026-06-09'
  read: F
  role: proposes
  code_release: not checked
  extraction_scope: 'Sections 4 to 6 and Appendix D: Table 15 rows for the four pairs where the lower level is data or tool output (L0 to L3 against L4), the ORR row of Table 1 and the probe-execution row of Table 24 (SEP);
    the abstract claim (pairs without L4, the reference split, XSTest, MT-Bench, TensorTrust and the hyperparameter sweeps not coded)'
defences:
- id: gwbi_l31
  name: GW-DPO, bilateral schedule
  origin: this_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - delimiters_markers
  - learned_token_features
  axis2:
  - sft
  - preference_optimisation
  - distillation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components:
  - 12 delimiter tokens marking levels L0 (platform) to L4 (data and tool output) and the response
  - 'Instructional Segment Embeddings: a learned embedding per level added to each token'
  - 'phase 1: LoRA SFT on hierarchy-formatted data'
  - 'phase 2: DPO on conflict pairs, preferred answers partly written by Claude Sonnet 4 or GPT-4o-mini, per-sample margin growing with the level gap'
  layout: each level wrapped in its own delimiter tokens; L4 holds data and tool output
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: untested
  anchor: We optimize the GW-DPO loss
  pages: &id001
    components: 5
- id: gwlin_l31
  name: GW-DPO, linear schedule
  origin: this_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - delimiters_markers
  - learned_token_features
  axis2:
  - sft
  - preference_optimisation
  - distillation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components:
  - 12 delimiter tokens marking levels L0 (platform) to L4 (data and tool output) and the response
  - 'Instructional Segment Embeddings: a learned embedding per level added to each token'
  - 'phase 1: LoRA SFT on hierarchy-formatted data'
  - 'phase 2: DPO on conflict pairs, preferred answers partly written by Claude Sonnet 4 or GPT-4o-mini, per-sample margin growing with the level gap'
  layout: each level wrapped in its own delimiter tokens; L4 holds data and tool output
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: untested
  anchor: We optimize the GW-DPO loss
  pages: *id001
- id: dpo_l31
  name: standard DPO (GW-DPO with alpha = 0)
  origin: this_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - delimiters_markers
  - learned_token_features
  axis2:
  - sft
  - preference_optimisation
  - distillation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components:
  - 12 delimiter tokens marking levels L0 (platform) to L4 (data and tool output) and the response
  - 'Instructional Segment Embeddings: a learned embedding per level added to each token'
  - 'phase 1: LoRA SFT on hierarchy-formatted data'
  - 'phase 2: DPO on conflict pairs, preferred answers partly written by Claude Sonnet 4 or GPT-4o-mini, per-sample margin growing with the level gap'
  layout: each level wrapped in its own delimiter tokens; L4 holds data and tool output
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: untested
  anchor: We optimize the GW-DPO loss
  pages: *id001
- id: noise_l31
  name: GW-DPO, linear schedule, without ISE (No-ISE)
  origin: this_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  - preference_optimisation
  - distillation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components:
  - 12 delimiter tokens marking levels L0 (platform) to L4 (data and tool output) and the response
  - no segment embeddings
  - 'phase 1: LoRA SFT on hierarchy-formatted data'
  - 'phase 2: DPO on conflict pairs, preferred answers partly written by Claude Sonnet 4 or GPT-4o-mini, per-sample margin growing with the level gap'
  layout: each level wrapped in its own delimiter tokens; L4 holds data and tool output
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: untested
  anchor: We optimize the GW-DPO loss
  pages: *id001
- id: lvl3_l31
  name: GW-DPO, linear schedule, three levels (3-level)
  origin: this_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - delimiters_markers
  - learned_token_features
  axis2:
  - sft
  - preference_optimisation
  - distillation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components:
  - 12 delimiter tokens marking levels L0 (platform) to L4 (data and tool output) and the response
  - 'Instructional Segment Embeddings: a learned embedding per level added to each token'
  - 'phase 1: LoRA SFT on hierarchy-formatted data'
  - 'phase 2: DPO on conflict pairs, preferred answers partly written by Claude Sonnet 4 or GPT-4o-mini, per-sample margin growing with the level gap'
  layout: each level wrapped in its own delimiter tokens; L4 holds data and tool output
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: untested
  anchor: We optimize the GW-DPO loss
  pages: *id001
- id: sft_l31
  name: phase-1 SFT only (delimiter tokens and ISE, no DPO)
  origin: this_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - delimiters_markers
  - learned_token_features
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components:
  - 12 delimiter tokens marking levels L0 (platform) to L4 (data and tool output) and the response
  - 'Instructional Segment Embeddings: a learned embedding per level added to each token'
  - 'phase 1: LoRA SFT on hierarchy-formatted data'
  - no phase 2
  layout: each level wrapped in its own delimiter tokens; L4 holds data and tool output
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: untested
  anchor: We fine-tune Llama-3.1-8B-Instruct on DSFT
  pages: *id001
evaluations:
- id: ev_l0l4_base
  defence_id: none:Llama-3.1-8B-Instruct (12 delimiter tokens added to the tokenizer, untrained)
  benchmark: 5-level evaluation suite, conflict split, pair L0 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_l0l4_sft
  defence_id: sft_l31
  benchmark: 5-level evaluation suite, conflict split, pair L0 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
- id: ev_l0l4_gwbi
  defence_id: gwbi_l31
  benchmark: 5-level evaluation suite, conflict split, pair L0 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
- id: ev_l0l4_gwlin
  defence_id: gwlin_l31
  benchmark: 5-level evaluation suite, conflict split, pair L0 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
- id: ev_l0l4_dpo
  defence_id: dpo_l31
  benchmark: 5-level evaluation suite, conflict split, pair L0 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
- id: ev_l0l4_noise
  defence_id: noise_l31
  benchmark: 5-level evaluation suite, conflict split, pair L0 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
- id: ev_l0l4_lvl3
  defence_id: lvl3_l31
  benchmark: 5-level evaluation suite, conflict split, pair L0 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
- id: ev_l1l4_base
  defence_id: none:Llama-3.1-8B-Instruct (12 delimiter tokens added to the tokenizer, untrained)
  benchmark: 5-level evaluation suite, conflict split, pair L1 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_l1l4_sft
  defence_id: sft_l31
  benchmark: 5-level evaluation suite, conflict split, pair L1 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
- id: ev_l1l4_gwbi
  defence_id: gwbi_l31
  benchmark: 5-level evaluation suite, conflict split, pair L1 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
- id: ev_l1l4_gwlin
  defence_id: gwlin_l31
  benchmark: 5-level evaluation suite, conflict split, pair L1 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
- id: ev_l1l4_dpo
  defence_id: dpo_l31
  benchmark: 5-level evaluation suite, conflict split, pair L1 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
- id: ev_l1l4_noise
  defence_id: noise_l31
  benchmark: 5-level evaluation suite, conflict split, pair L1 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
- id: ev_l1l4_lvl3
  defence_id: lvl3_l31
  benchmark: 5-level evaluation suite, conflict split, pair L1 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
- id: ev_l2l4_base
  defence_id: none:Llama-3.1-8B-Instruct (12 delimiter tokens added to the tokenizer, untrained)
  benchmark: 5-level evaluation suite, conflict split, pair L2 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_l2l4_sft
  defence_id: sft_l31
  benchmark: 5-level evaluation suite, conflict split, pair L2 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
- id: ev_l2l4_gwbi
  defence_id: gwbi_l31
  benchmark: 5-level evaluation suite, conflict split, pair L2 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
- id: ev_l2l4_gwlin
  defence_id: gwlin_l31
  benchmark: 5-level evaluation suite, conflict split, pair L2 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
- id: ev_l2l4_dpo
  defence_id: dpo_l31
  benchmark: 5-level evaluation suite, conflict split, pair L2 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
- id: ev_l2l4_noise
  defence_id: noise_l31
  benchmark: 5-level evaluation suite, conflict split, pair L2 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
- id: ev_l2l4_lvl3
  defence_id: lvl3_l31
  benchmark: 5-level evaluation suite, conflict split, pair L2 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
- id: ev_l3l4_base
  defence_id: none:Llama-3.1-8B-Instruct (12 delimiter tokens added to the tokenizer, untrained)
  benchmark: 5-level evaluation suite, conflict split, pair L3 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_l3l4_sft
  defence_id: sft_l31
  benchmark: 5-level evaluation suite, conflict split, pair L3 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
- id: ev_l3l4_gwbi
  defence_id: gwbi_l31
  benchmark: 5-level evaluation suite, conflict split, pair L3 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
- id: ev_l3l4_gwlin
  defence_id: gwlin_l31
  benchmark: 5-level evaluation suite, conflict split, pair L3 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
- id: ev_l3l4_dpo
  defence_id: dpo_l31
  benchmark: 5-level evaluation suite, conflict split, pair L3 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
- id: ev_l3l4_noise
  defence_id: noise_l31
  benchmark: 5-level evaluation suite, conflict split, pair L3 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
- id: ev_l3l4_lvl3
  defence_id: lvl3_l31
  benchmark: 5-level evaluation suite, conflict split, pair L3 against L4 (instruction in data or tool output)
  attacker_type: static
  attacker_access: black_box
  judge: 'GPT-4o judge: satisfies the higher level without following the lower'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 29
  decoding: not reported
  pages:
    judge: 6
- id: ev_orr_base
  defence_id: none:Llama-3.1-8B-Instruct (12 delimiter tokens added to the tokenizer, untrained)
  benchmark: 5-level evaluation suite, aligned-control split (benign prompts that resemble conflicts)
  attacker_type: none
  attacker_access: not_applicable
  judge: 'GPT-4o judge: refusal'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_orr_sft
  defence_id: sft_l31
  benchmark: 5-level evaluation suite, aligned-control split (benign prompts that resemble conflicts)
  attacker_type: none
  attacker_access: not_applicable
  judge: 'GPT-4o judge: refusal'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
- id: ev_orr_gwbi
  defence_id: gwbi_l31
  benchmark: 5-level evaluation suite, aligned-control split (benign prompts that resemble conflicts)
  attacker_type: none
  attacker_access: not_applicable
  judge: 'GPT-4o judge: refusal'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
- id: ev_orr_gwlin
  defence_id: gwlin_l31
  benchmark: 5-level evaluation suite, aligned-control split (benign prompts that resemble conflicts)
  attacker_type: none
  attacker_access: not_applicable
  judge: 'GPT-4o judge: refusal'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
- id: ev_orr_dpo
  defence_id: dpo_l31
  benchmark: 5-level evaluation suite, aligned-control split (benign prompts that resemble conflicts)
  attacker_type: none
  attacker_access: not_applicable
  judge: 'GPT-4o judge: refusal'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
- id: ev_orr_noise
  defence_id: noise_l31
  benchmark: 5-level evaluation suite, aligned-control split (benign prompts that resemble conflicts)
  attacker_type: none
  attacker_access: not_applicable
  judge: 'GPT-4o judge: refusal'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
- id: ev_orr_lvl3
  defence_id: lvl3_l31
  benchmark: 5-level evaluation suite, aligned-control split (benign prompts that resemble conflicts)
  attacker_type: none
  attacker_access: not_applicable
  judge: 'GPT-4o judge: refusal'
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
- id: ev_sep_anchor
  defence_id: none:Llama-3.1-8B-Instruct (off-the-shelf, native chat template)
  benchmark: SEP, 1,500 probes (never used in training)
  attacker_type: static
  attacker_access: black_box
  judge: witness string found in a non-trivial response
  layout_same_as_defence: 'no'
  measured_by: gwdpo_arxiv2026
  page: 33
  decoding: not reported
  sample_size: 1,500
  baseline_kind: off_the_shelf
- id: ev_sep_base
  defence_id: none:Llama-3.1-8B-Instruct (12 delimiter tokens added to the tokenizer, untrained)
  benchmark: SEP, 1,500 probes (never used in training)
  attacker_type: static
  attacker_access: black_box
  judge: witness string found in a non-trivial response
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 33
  decoding: not reported
  sample_size: 1,500
  baseline_kind: off_the_shelf
- id: ev_sep_sft
  defence_id: sft_l31
  benchmark: SEP, 1,500 probes (never used in training)
  attacker_type: static
  attacker_access: black_box
  judge: witness string found in a non-trivial response
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 33
  decoding: not reported
  sample_size: 1,500
- id: ev_sep_gwbi
  defence_id: gwbi_l31
  benchmark: SEP, 1,500 probes (never used in training)
  attacker_type: static
  attacker_access: black_box
  judge: witness string found in a non-trivial response
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 33
  decoding: not reported
  sample_size: 1,500
- id: ev_sep_gwlin
  defence_id: gwlin_l31
  benchmark: SEP, 1,500 probes (never used in training)
  attacker_type: static
  attacker_access: black_box
  judge: witness string found in a non-trivial response
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 33
  decoding: not reported
  sample_size: 1,500
- id: ev_sep_dpo
  defence_id: dpo_l31
  benchmark: SEP, 1,500 probes (never used in training)
  attacker_type: static
  attacker_access: black_box
  judge: witness string found in a non-trivial response
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 33
  decoding: not reported
  sample_size: 1,500
- id: ev_sep_noise
  defence_id: noise_l31
  benchmark: SEP, 1,500 probes (never used in training)
  attacker_type: static
  attacker_access: black_box
  judge: witness string found in a non-trivial response
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 33
  decoding: not reported
  sample_size: 1,500
- id: ev_sep_lvl3
  defence_id: lvl3_l31
  benchmark: SEP, 1,500 probes (never used in training)
  attacker_type: static
  attacker_access: black_box
  judge: witness string found in a non-trivial response
  layout_same_as_defence: 'yes'
  measured_by: gwdpo_arxiv2026
  page: 33
  decoding: not reported
  sample_size: 1,500
results:
- id: r_l0l4_base
  evaluation_id: ev_l0l4_base
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L0 vs. L4 (1 - PPA)
  value: '0.650'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l0l4_sft
  evaluation_id: ev_l0l4_sft
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L0 vs. L4 (1 - PPA)
  value: '0.425'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l0l4_gwbi
  evaluation_id: ev_l0l4_gwbi
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L0 vs. L4 (1 - PPA)
  value: '0.025'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l0l4_gwlin
  evaluation_id: ev_l0l4_gwlin
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L0 vs. L4 (1 - PPA)
  value: '0.250'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l0l4_dpo
  evaluation_id: ev_l0l4_dpo
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L0 vs. L4 (1 - PPA)
  value: '0.100'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l0l4_noise
  evaluation_id: ev_l0l4_noise
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L0 vs. L4 (1 - PPA)
  value: '0.125'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l0l4_lvl3
  evaluation_id: ev_l0l4_lvl3
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L0 vs. L4 (1 - PPA)
  value: '0.025'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l1l4_base
  evaluation_id: ev_l1l4_base
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L1 vs. L4 (1 - PPA)
  value: '0.769'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l1l4_sft
  evaluation_id: ev_l1l4_sft
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L1 vs. L4 (1 - PPA)
  value: '0.596'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l1l4_gwbi
  evaluation_id: ev_l1l4_gwbi
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L1 vs. L4 (1 - PPA)
  value: '0.096'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l1l4_gwlin
  evaluation_id: ev_l1l4_gwlin
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L1 vs. L4 (1 - PPA)
  value: '0.308'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l1l4_dpo
  evaluation_id: ev_l1l4_dpo
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L1 vs. L4 (1 - PPA)
  value: '0.173'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l1l4_noise
  evaluation_id: ev_l1l4_noise
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L1 vs. L4 (1 - PPA)
  value: '0.404'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l1l4_lvl3
  evaluation_id: ev_l1l4_lvl3
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L1 vs. L4 (1 - PPA)
  value: '0.212'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l2l4_base
  evaluation_id: ev_l2l4_base
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L2 vs. L4 (1 - PPA)
  value: '0.784'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l2l4_sft
  evaluation_id: ev_l2l4_sft
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L2 vs. L4 (1 - PPA)
  value: '0.568'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l2l4_gwbi
  evaluation_id: ev_l2l4_gwbi
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L2 vs. L4 (1 - PPA)
  value: '0.162'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l2l4_gwlin
  evaluation_id: ev_l2l4_gwlin
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L2 vs. L4 (1 - PPA)
  value: '0.243'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l2l4_dpo
  evaluation_id: ev_l2l4_dpo
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L2 vs. L4 (1 - PPA)
  value: '0.189'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l2l4_noise
  evaluation_id: ev_l2l4_noise
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L2 vs. L4 (1 - PPA)
  value: '0.595'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l2l4_lvl3
  evaluation_id: ev_l2l4_lvl3
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L2 vs. L4 (1 - PPA)
  value: '0.189'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l3l4_base
  evaluation_id: ev_l3l4_base
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L3 vs. L4 (1 - PPA)
  value: '0.792'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l3l4_sft
  evaluation_id: ev_l3l4_sft
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L3 vs. L4 (1 - PPA)
  value: '0.642'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l3l4_gwbi
  evaluation_id: ev_l3l4_gwbi
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L3 vs. L4 (1 - PPA)
  value: '0.264'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l3l4_gwlin
  evaluation_id: ev_l3l4_gwlin
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L3 vs. L4 (1 - PPA)
  value: '0.340'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l3l4_dpo
  evaluation_id: ev_l3l4_dpo
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L3 vs. L4 (1 - PPA)
  value: '0.264'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l3l4_noise
  evaluation_id: ev_l3l4_noise
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L3 vs. L4 (1 - PPA)
  value: '0.774'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_l3l4_lvl3
  evaluation_id: ev_l3l4_lvl3
  quantity: R1
  coverage: direct
  metric_as_printed: ASR L3 vs. L4 (1 - PPA)
  value: '0.113'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 29
  location: Table 15
- id: r_orr_base
  evaluation_id: ev_orr_base
  quantity: other
  coverage: direct
  metric_as_printed: ORR (over-refusal rate)
  value: '0.044'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_orr_sft
  evaluation_id: ev_orr_sft
  quantity: other
  coverage: direct
  metric_as_printed: ORR (over-refusal rate)
  value: '0.092'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_orr_gwbi
  evaluation_id: ev_orr_gwbi
  quantity: other
  coverage: direct
  metric_as_printed: ORR (over-refusal rate)
  value: '0.057'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_orr_gwlin
  evaluation_id: ev_orr_gwlin
  quantity: other
  coverage: direct
  metric_as_printed: ORR (over-refusal rate)
  value: '0.024'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_orr_dpo
  evaluation_id: ev_orr_dpo
  quantity: other
  coverage: direct
  metric_as_printed: ORR (over-refusal rate)
  value: '0.120'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_orr_noise
  evaluation_id: ev_orr_noise
  quantity: other
  coverage: direct
  metric_as_printed: ORR (over-refusal rate)
  value: '0.576'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_orr_lvl3
  evaluation_id: ev_orr_lvl3
  quantity: other
  coverage: direct
  metric_as_printed: ORR (over-refusal rate)
  value: '0.074'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_sep_anchor
  evaluation_id: ev_sep_anchor
  quantity: R1
  coverage: direct
  metric_as_printed: Probe-execution rate
  value: '0.693'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 24
- id: r_sep_base
  evaluation_id: ev_sep_base
  quantity: R1
  coverage: direct
  metric_as_printed: Probe-execution rate
  value: '0.476'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 24
- id: r_sep_sft
  evaluation_id: ev_sep_sft
  quantity: R1
  coverage: direct
  metric_as_printed: Probe-execution rate
  value: '0.850'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 24
- id: r_sep_gwbi
  evaluation_id: ev_sep_gwbi
  quantity: R1
  coverage: direct
  metric_as_printed: Probe-execution rate
  value: '0.118'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 24
- id: r_sep_gwlin
  evaluation_id: ev_sep_gwlin
  quantity: R1
  coverage: direct
  metric_as_printed: Probe-execution rate
  value: '0.139'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 24
- id: r_sep_dpo
  evaluation_id: ev_sep_dpo
  quantity: R1
  coverage: direct
  metric_as_printed: Probe-execution rate
  value: '0.133'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 24
- id: r_sep_noise
  evaluation_id: ev_sep_noise
  quantity: R1
  coverage: direct
  metric_as_printed: Probe-execution rate
  value: '0.047'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 24
- id: r_sep_lvl3
  evaluation_id: ev_sep_lvl3
  quantity: R1
  coverage: direct
  metric_as_printed: Probe-execution rate
  value: '0.472'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 24
comparisons:
- id: c_l0l4_sft
  left: ev_l0l4_sft
  right: ev_l0l4_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 29
- id: c_l0l4_gwbi
  left: ev_l0l4_gwbi
  right: ev_l0l4_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 29
- id: c_l0l4_dpo
  left: ev_l0l4_dpo
  right: ev_l0l4_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 29
- id: c_l0l4_sched
  left: ev_l0l4_gwbi
  right: ev_l0l4_dpo
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: bilateral gravity margin against standard DPO
  page: 29
- id: c_l0l4_ise
  left: ev_l0l4_gwlin
  right: ev_l0l4_noise
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: with against without segment embeddings
  page: 29
- id: c_l1l4_sft
  left: ev_l1l4_sft
  right: ev_l1l4_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 29
- id: c_l1l4_gwbi
  left: ev_l1l4_gwbi
  right: ev_l1l4_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 29
- id: c_l1l4_dpo
  left: ev_l1l4_dpo
  right: ev_l1l4_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 29
- id: c_l1l4_sched
  left: ev_l1l4_gwbi
  right: ev_l1l4_dpo
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: bilateral gravity margin against standard DPO
  page: 29
- id: c_l1l4_ise
  left: ev_l1l4_gwlin
  right: ev_l1l4_noise
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: with against without segment embeddings
  page: 29
- id: c_l2l4_sft
  left: ev_l2l4_sft
  right: ev_l2l4_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 29
- id: c_l2l4_gwbi
  left: ev_l2l4_gwbi
  right: ev_l2l4_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 29
- id: c_l2l4_dpo
  left: ev_l2l4_dpo
  right: ev_l2l4_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 29
- id: c_l2l4_sched
  left: ev_l2l4_gwbi
  right: ev_l2l4_dpo
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: bilateral gravity margin against standard DPO
  page: 29
- id: c_l2l4_ise
  left: ev_l2l4_gwlin
  right: ev_l2l4_noise
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: with against without segment embeddings
  page: 29
- id: c_l3l4_sft
  left: ev_l3l4_sft
  right: ev_l3l4_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 29
- id: c_l3l4_gwbi
  left: ev_l3l4_gwbi
  right: ev_l3l4_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 29
- id: c_l3l4_dpo
  left: ev_l3l4_dpo
  right: ev_l3l4_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 29
- id: c_l3l4_sched
  left: ev_l3l4_gwbi
  right: ev_l3l4_dpo
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: bilateral gravity margin against standard DPO
  page: 29
- id: c_l3l4_ise
  left: ev_l3l4_gwlin
  right: ev_l3l4_noise
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: with against without segment embeddings
  page: 29
- id: c_orr_sft
  left: ev_orr_sft
  right: ev_orr_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_orr_gwbi
  left: ev_orr_gwbi
  right: ev_orr_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_orr_dpo
  left: ev_orr_dpo
  right: ev_orr_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sep_sft
  left: ev_sep_sft
  right: ev_sep_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_sep_gwbi
  left: ev_sep_gwbi
  right: ev_sep_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_sep_dpo
  left: ev_sep_dpo
  right: ev_sep_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_sep_ise
  left: ev_sep_gwlin
  right: ev_sep_noise
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: with against without segment embeddings
  page: 33
claims:
- id: k1
  text: GW-DPO with the bilateral schedule Pareto-improves over standard DPO and the linear schedule
  location: abstract
  anchor: Pareto-improves over standard DPO and the linear variant
  jobs:
  - R1
  supported_by:
  - c_l0l4_sched
  - c_l1l4_sched
  - c_l2l4_sched
  - c_l3l4_sched
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 15 (L4 pairs) and Table 24 (SEP)
    R2clean: reference split and MT-Bench are outside the coded scope
schema_gaps:
- where: evaluations ev_sep_*
  note: Table 24 notes that the No-ISE rate (0.047) comes from refusing 814 of 1,500 probes, and the off-the-shelf anchor uses the native chat template, so neither is a like-for-like reading (p. 33)
- where: results r_l3l4_*
  note: on the L3 against L4 pair (user against tool output) GW-DPO (bi.) ties standard DPO (0.264) and 3-level (lin.) is lower (0.113)

## Stage 2. The PDFs read are the PDFs registered, and every report is an included study

In [ ]:
import csv, logging, datetime as dt
from collections import Counter
import schema, search, corpus, paths
for m in (schema, search, corpus, paths):
    importlib.reload(m)
logging.getLogger("pypdf").setLevel(logging.ERROR)
from pypdf import PdfReader

BATCH = "04d"
cfg_exclusions = 1
cfg_next = "batch 04e"
IDS = ['agentdyn_arxiv2026', 'icon_arxiv2026', 'ihchallenge_arxiv2026', 'pismith_colm2026', 'autonomytax_arxiv2026', 'piarena_arxiv2026', 'localalign_ccs2026', 'alwaysfall_arxiv2026', 'actconsistency_arxiv2026', 'dynppa_arxiv2026', 'gwdpo_arxiv2026']
man = {m["slug"]: m for m in search.read_csv(paths.REPORTS / "defences.csv")}
problems, texts = [], {}
for rid in IDS:
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    slug = rec["report"]["slug"]
    pdf = paths.DATA_DEFENCES / f"{slug}.pdf"
    m = man.get(slug, {})
    if not pdf.exists():
        problems.append(f"{rid}: {pdf.name} not in data/defences"); continue
    sha = corpus.sha256(pdf)
    rd = PdfReader(str(pdf))
    texts[slug] = ([p.extract_text() or "" for p in rd.pages], [p.extract_text(extraction_mode="layout") or "" for p in rd.pages])
    ok_sha = sha == m.get("sha256")
    ok_inc = m.get("decision") == "include"
    print(f"{rid:<28} {slug:<12} pages {len(rd.pages):>3}  sha256 {'matches' if ok_sha else 'DIFFERS'}  decision {m.get('decision')}")
    if not ok_sha:
        problems.append(f"{rid}: PDF differs from the one registered in reports/defences.csv")
    if not ok_inc:
        problems.append(f"{rid}: not an included study")

## Stage 2b. Full-text exclusions found while coding

Each study below was included at title and abstract; its full text shows it does not meet the inclusion rule. The anchor is checked in its PDF, then the decision is written to schema/fulltext_decisions.csv and schema/screening_decisions.csv and carried into reports/screening.csv, reports/defences.csv, reports/fulltext_check.csv and reports/prisma_counts.json, as notebook 02d did.

In [ ]:
import json
import overlap
importlib.reload(overlap)
EXCL = [{'key': 'arxiv:2603.12277', 'slug': '2603.12277', 'reason': 'attack only', 'anchor': 'We introduce CoT Forgery, a novel black-box attack designed to isolate role', 'note': 'CoT Forgery attack and role probes on frontier models; no defence is added, trained or measured, and Section 7 argues for role perception without experiments'}]
FT_FIELDS = ["key", "slug", "decision", "reason", "basis", "anchor", "note"]
SD_FIELDS = ["key", "decision", "reason", "stage", "access_status", "note", "title"]
CHECK_FIELDS = ["key", "slug", "decision", "reason", "basis", "page", "status", "anchor", "sha256", "note"]
ft = search.read_csv(paths.SCHEMA / "fulltext_decisions.csv")
sd = search.read_csv(paths.SCHEMA / "screening_decisions.csv")
fchk = search.read_csv(paths.REPORTS / "fulltext_check.csv")
mlist = search.read_csv(paths.REPORTS / "defences.csv")
for e in EXCL:
    m = next((x for x in mlist if x["slug"] == e["slug"]), None)
    pdf = paths.DATA_DEFENCES / f"{e['slug']}.pdf"
    if m is None or not pdf.exists():
        problems.append(f"exclusion {e['slug']}: not in reports/defences.csv or PDF missing"); continue
    sha = corpus.sha256(pdf)
    if sha != m.get("sha256"):
        problems.append(f"exclusion {e['slug']}: PDF differs from the one registered")
    if e["reason"] not in search.EXCLUSION_REASONS:
        problems.append(f"exclusion {e['slug']}: reason not in the fixed list")
    pages = overlap.pdf_page_texts(pdf)
    a = overlap.norm_text(e["anchor"])
    hits = [i + 1 for i, t in enumerate(pages) if a in t]
    status = "found" if hits else "anchor not found"
    if not hits:
        problems.append(f"exclusion {e['slug']}: anchor not found in the PDF")
    row = {"key": e["key"], "slug": e["slug"], "decision": "exclude", "reason": e["reason"], "basis": "full text", "anchor": e["anchor"], "note": e["note"]}
    ft = [r for r in ft if r["key"] != e["key"]] + [row]
    fchk = [r for r in fchk if r["key"] != e["key"]] + [dict(row, page=str(hits[0]) if hits else "", status=status, sha256=sha)]
    n = 0
    for r in sd:
        if r["key"] == e["key"]:
            r.update(decision="exclude", reason=e["reason"], stage="full_text", note="full text: " + e["note"]); n += 1
    if n != 1:
        problems.append(f"exclusion {e['slug']}: {n} rows for {e['key']} in schema/screening_decisions.csv")
    m.update(decision="exclude", reason=e["reason"], stage="full_text", note="full text: " + e["note"])
    print(f"{e['slug']:<12} excluded at full text ({e['reason']}); anchor {status}{' on page ' + str(hits[0]) if hits else ''}")
def write_lf(path, rows, fields):              # the schema files were written with plain newlines; keep them that way
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore", lineterminator="\n")
        w.writeheader()
        w.writerows({k: r.get(k) or "" for k in fields} for r in rows)
write_lf(paths.SCHEMA / "fulltext_decisions.csv", ft, FT_FIELDS)
write_lf(paths.SCHEMA / "screening_decisions.csv", sd, SD_FIELDS)
search.write_csv(paths.REPORTS / "fulltext_check.csv", fchk, CHECK_FIELDS)
search.write_csv(paths.REPORTS / "defences.csv", mlist, list(mlist[0].keys()))
rows = search.read_csv(paths.REPORTS / "screening.csv")
rows, more = search.apply_decisions(rows, sd)
problems += more
search.write_csv(paths.REPORTS / "screening.csv", rows, search.SCREEN_FIELDS)
prisma = json.loads((paths.REPORTS / "prisma_counts.json").read_text())
prisma.update(search.prisma_counts(search.read_csv(paths.REPORTS / "search_log.csv"), rows))
prisma["fulltext_anchors_found"] = sum(1 for r in fchk if r["status"].startswith("found"))
(paths.REPORTS / "prisma_counts.json").write_text(json.dumps(prisma, indent=1))
print(f"included now: {prisma['included']}; full text assessed: {prisma['full_text_assessed']}; "
      f"full-text exclusions: {json.dumps(prisma['excluded_full_text_by_reason'])}")

## Stage 3. Validate against the frozen schema and the PDFs; grade the comparisons; extra checks

In [ ]:
import re
import pypdfium2 as pdfium
_docs = {}
def pdfium_page(slug, page):
    if slug not in _docs:
        _docs[slug] = pdfium.PdfDocument(str(paths.DATA_DEFENCES / f"{slug}.pdf"))
    return _docs[slug][page - 1].get_textpage().get_text_range()
S1 = schema.load_schema(paths.SCHEMA / "record_schema.yaml")
check_rows, summary, residual, third_rows = [], [], [], []
for rid in IDS:
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    slug = rec["report"]["slug"]
    plain, lay = texts.get(slug, (None, None))
    probs, notes = schema.validate_record(rec, S1, plain, lay)
    kept, third = [], []                       # a value pypdf glues to its neighbour is looked for in a third extraction (pypdfium2)
    for p in probs:
        m = re.match(r"results:(\S+): value '(.+)' not found on page (\d+)$", p)
        if m and schema.value_on_page(m.group(2), pdfium_page(slug, int(m.group(3)))):
            third.append(f"{m.group(1)} ({m.group(2)!r}, page {m.group(3)})")
        else:
            kept.append(p)
    probs = kept
    for kind in ("defences", "evaluations", "results", "comparisons", "claims"):          # ids unique within each kind
        dup = [i for i, n in Counter(r["id"] for r in rec.get(kind) or []).items() if n > 1]
        probs += [f"{kind}: id {i} used more than once" for i in dup]
    fig_kept, third_chart = 0, []
    for r in rec.get("results") or []:                                                     # chart values kept as printed
        if r.get("value_source") == "figure_only" and r.get("value") not in (None, ""):
            fig_kept += 1
            pg = int(r["page"]) - 1
            if plain and not (schema.value_on_page(r["value"], plain[pg]) or schema.value_on_page(r["value"], lay[pg])):
                if schema.value_on_page(r["value"], pdfium_page(slug, int(r["page"]))):
                    third_chart.append(f"{r['id']} ({r['value']!r}, page {r['page']}, chart label)")
                else:
                    probs.append(f"results:{r['id']}: chart value {r['value']!r} not found on page {r['page']}")
    third_rows += [{"report": rid, "status": "value found in the third extraction", "detail": t} for t in third + third_chart]
    grades = Counter(schema.grade(c) for c in rec.get("comparisons") or [])
    n_val = sum(1 for r in rec.get("results") or [] if r.get("value_source") != "figure_only")
    n_anc = sum(1 for k in ("defences", "claims") for r in rec.get(k) or [] if r.get("anchor"))
    summary.append({"batch": BATCH, "report": rid, "slug": slug, "role": rec["report"]["role"],
                    "defences": len(rec.get("defences") or []), "evaluations": len(rec.get("evaluations") or []),
                    "results": len(rec.get("results") or []), "values_checked": n_val, "chart_values_checked": fig_kept,
                    "anchors_checked": n_anc, "comparisons": len(rec.get("comparisons") or []),
                    "grade_A": grades["A"], "grade_B": grades["B"], "grade_C": grades["C"], "grade_U": grades["U"],
                    "claims": len(rec.get("claims") or []),
                    **{f"cov_{q}": rec["coverage"][q] for q in ("R1", "R2clean", "R2attack", "R3ns", "R3cp", "R3safe")},
                    "values_third_extraction": len(third), "chart_values_third_extraction": len(third_chart), "residual_notes": len(notes), "problems": len(probs)})
    check_rows += [{"report": rid, "status": "problem", "detail": p} for p in probs]
    residual += [{"report": rid, "where": n.split(": ", 1)[0], "note": n.split(": ", 1)[1]} for n in notes]
    print(f"{rid:<28} problems {len(probs)}; values {n_val} ({len(third)} via the third extraction); chart values {fig_kept} ({len(third_chart)} via the third extraction); anchors {n_anc}; grades {dict(grades)}")
    for p in probs:
        print("   ", p)
    problems += [f"{rid}: {p}" for p in probs]
check_rows += third_rows + [{"report": g["report"], "status": "residual note", "detail": f"{g['where']}: {g['note']}"} for g in residual]
search.write_csv(paths.REPORTS / f"extraction_{BATCH}_check.csv", check_rows or [{"report": "all", "status": "ok", "detail": "no problems"}],
                 ["report", "status", "detail"])
search.write_csv(paths.REPORTS / f"extraction_{BATCH}_summary.csv", summary, list(summary[0].keys()))
for g in residual:
    print(f"  residual note, {g['report']}: {g['note'][:150]}")

## Stage 4. Batch register (one row per coded report, kept across batches)

In [ ]:
reg_path = paths.REPORTS / "extraction_batches.csv"
reg = [r for r in search.read_csv(reg_path) if r["batch"] != BATCH] if reg_path.exists() else []
today = dt.date.today().isoformat()
for s in summary:
    reg.append({"batch": BATCH, "report": s["report"], "slug": s["slug"], "coded": today,
                "status": "checked" if s["problems"] == 0 else "problems"})
reg.sort(key=lambda r: (r["batch"], r["report"]))
search.write_csv(reg_path, reg, ["batch", "report", "slug", "coded", "status"])
included = sum(1 for m in search.read_csv(paths.REPORTS / "defences.csv") if m.get("decision") == "include")   # read again: Stage 2b may change it
pilot = len(list(paths.RECORDS.glob("*.yaml"))) - len({r["report"] for r in reg})
print(f"records in reports/records: {len(list(paths.RECORDS.glob('*.yaml')))} (pilot {pilot}, batches {len({r['report'] for r in reg})}); included studies: {included}")

## Stage 5. Decision record and tests

In [ ]:
tot = {k: sum(s[k] for s in summary) for k in ("defences", "evaluations", "results", "values_checked", "values_third_extraction", "chart_values_checked",
                                              "chart_values_third_extraction", "anchors_checked", "comparisons", "grade_A", "grade_B", "grade_C", "grade_U", "claims")}
status = "BATCH CHECKED" if not problems else "BATCH NOT CHECKED: PROBLEMS BELOW"
lines = [f"decision_{BATCH}: {status}", f"this step: {dt.date.today().isoformat()}",
         "reports: " + ", ".join(IDS),
         f"records: {tot['defences']} defences, {tot['evaluations']} evaluations, {tot['results']} results, "
         f"{tot['comparisons']} comparisons, {tot['claims']} claims",
         f"checked against the PDFs: {tot['values_checked']} printed values ({tot['values_third_extraction']} of them found only in the third "
         f"extraction), {tot['chart_values_checked']} chart values ({tot['chart_values_third_extraction']} of them found only in the third extraction) "
         f"and {tot['anchors_checked']} anchors, each found on its page",
         f"comparison grades: A {tot['grade_A']}, B {tot['grade_B']}, C {tot['grade_C']}, U {tot['grade_U']}",
         f"residual notes: {len(residual)}; full-text exclusions found while coding: {cfg_exclusions}",
         f"next: {cfg_next}"] + [f"- problem: {p}" for p in problems]
(paths.REPORTS / f"decision_{BATCH}.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout.strip().splitlines()[-1] if tests.stdout.strip() else tests.stderr.strip()[-400:])

## Stage 6. Commit

In [ ]:
import os, subprocess
os.chdir(os.environ.get("SOURCEMAP_REPO_ROOT", "/content/drive/MyDrive/SourceMap_Research/sourcemap-research"))
r = subprocess.run(["python", "tools/commit_cell.py",
                    "04d extraction batch d: eleven included studies coded from their PDFs and one full-text exclusion, values and anchors checked, comparisons graded"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)